In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

# YF 쏘나타 검색 결과 URL
url = "https://car.encar.com/list/car?page=1&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._.%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._.%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7"
}

resp = requests.get(url, headers=headers, timeout=15)
resp.raise_for_status()

print(resp.status_code)
print(resp.text[:1000])   # 먼저 구조 확인

200
<!DOCTYPE html><html lang="ko"><head><meta charSet="utf-8"/><meta name="naver-site-verification" content="d6816425de244fa5871cff7ed7aa527a3ec6c481"/><meta name="google-site-verification" content="k-XSUZ77xZ_-SZbScyPs2ahKnh-FPiib6Ks4Qoll-Q0"/><meta name="facebook-domain-verification" content="2xnfhiu5wheuiglzcp22q1c2gjsbpf"/><link rel="shortcut icon" href="/favicon.ico"/><meta http-equiv="cache-control" content="no-cache"/><meta http-equiv="expires" content="Tue, 01 Jan 1980 1:00:00 GMT"/><meta http-equiv="pragma" content="no-cache"/><link href="https://ci.encar.com" rel="dns-prefetch"/><link href="https://static.encar.com" rel="dns-prefetch"/><title>엔카믿고 현대 YF 쏘나타 중고차 : 내차팔기·내차사기</title><meta property="og:title" content="엔카믿고 현대 YF 쏘나타 중고차 : 내차팔기·내차사기"/><meta name="description" content="YF 쏘나타중고차는 역시 엔카! 최다 매물과 편리한 구매 서비스로 나만의 중고차를 찾아보세요."/><meta property="og:description" content="YF 쏘나타중고차는 역시 엔카! 최다 매물과 편리한 구매 서비스로 나만의 중고차를 찾아보세요."/><meta property="og:url" content="https://car.en

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
from urllib.parse import urljoin

BASE_URL = "https://car.encar.com"

# YF 쏘나타 검색 URL
SEARCH_URL_TEMPLATE = (
    "https://car.encar.com/list/car?page={page}"
    "&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._."
    "%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._."
    "%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C"
    "%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D"
)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://car.encar.com/"
}

REGIONS = [
    "서울", "경기", "인천", "부산", "대구", "대전", "광주", "울산", "세종",
    "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"
]

FUELS = [
    "가솔린", "디젤", "LPG", "LPG(일반인 구입)", "하이브리드", "가솔린+전기",
    "전기", "CNG", "수소"
]


def extract_price(text: str):
    m = re.search(r'(\d[\d,]*)\s*만원', text)
    return int(m.group(1).replace(",", "")) if m else None


def extract_mileage(text: str):
    m = re.search(r'(\d[\d,]*)\s*km', text, re.IGNORECASE)
    return int(m.group(1).replace(",", "")) if m else None


def extract_year(text: str):
    # 예: 10/10식(11년형), 12/04식
    m = re.search(r'(\d{2}/\d{2}식(?:\(\d{2}년형\))?)', text)
    return m.group(1) if m else None


def extract_fuel(text: str):
    for fuel in FUELS:
        if fuel in text:
            return fuel
    return None


def extract_region(text: str):
    for region in REGIONS:
        if region in text:
            return region
    return None


def extract_name(text: str, year_text: str):
    """
    연식 앞부분까지를 차량명으로 추정
    """
    if year_text and year_text in text:
        name = text.split(year_text)[0].strip()
        return re.sub(r'\s+', ' ', name)
    return None


def normalize_text(text: str):
    return re.sub(r'\s+', ' ', text).strip()


def collect_candidate_links(card, base_url=BASE_URL):
    links = []
    for a_tag in card.select("a[href]"):
        href = a_tag.get("href", "").strip()
        if not href:
            continue
        full_url = urljoin(base_url, href)
        links.append(full_url)

    # 중복 제거
    deduped = []
    seen = set()
    for link in links:
        if link not in seen:
            deduped.append(link)
            seen.add(link)
    return deduped


def parse_card(card):
    raw_text = normalize_text(card.get_text(" ", strip=True))

    if "YF 쏘나타" not in raw_text:
        return None

    if "만원" not in raw_text or "km" not in raw_text:
        return None

    year_text = extract_year(raw_text)
    name = extract_name(raw_text, year_text)
    price = extract_price(raw_text)
    mileage = extract_mileage(raw_text)
    fuel = extract_fuel(raw_text)
    region = extract_region(raw_text)
    links = collect_candidate_links(card)

    return {
        "차량명": name,
        "연식": year_text,
        "주행거리_km": mileage,
        "연료": fuel,
        "지역": region,
        "가격_만원": price,
        "상세링크_후보": " | ".join(links[:5]) if links else None,
        "raw_text": raw_text
    }


def fetch_page(page: int):
    url = SEARCH_URL_TEMPLATE.format(page=page)
    print(f"[INFO] 요청 중: {url}")

    response = requests.get(url, headers=HEADERS, timeout=20)
    response.raise_for_status()

    return response.text


def parse_page(html: str):
    soup = BeautifulSoup(html, "html.parser")

    rows = []

    # li / div / article 전부 후보로 본다
    candidate_cards = soup.select("li, div, article")

    for card in candidate_cards:
        item = parse_card(card)
        if item:
            rows.append(item)

    # 중복 제거용
    if rows:
        df = pd.DataFrame(rows)
        df = df.drop_duplicates(subset=["차량명", "연식", "주행거리_km", "가격_만원", "raw_text"])
        return df

    return pd.DataFrame()


def scrape_yf_sonata(max_pages=5, delay=1.5, save_csv=True):
    all_dfs = []

    for page in range(1, max_pages + 1):
        try:
            html = fetch_page(page)
            df_page = parse_page(html)

            print(f"[INFO] page={page}, 수집건수={len(df_page)}")

            if df_page.empty:
                print("[INFO] 빈 페이지 또는 파싱 실패로 판단, 중단합니다.")
                break

            all_dfs.append(df_page)
            time.sleep(delay)

        except Exception as e:
            print(f"[ERROR] page={page} 수집 실패: {e}")
            break

    if not all_dfs:
        print("[WARN] 수집된 데이터가 없습니다.")
        return pd.DataFrame()

    df_final = pd.concat(all_dfs, ignore_index=True)
    df_final = df_final.drop_duplicates(subset=["차량명", "연식", "주행거리_km", "가격_만원", "raw_text"])
    df_final = df_final.reset_index(drop=True)

    if save_csv:
        filename = "encar_yf_sonata.csv"
        df_final.to_csv(filename, index=False, encoding="utf-8-sig")
        print(f"[INFO] 저장 완료: {filename}")

    return df_final


if __name__ == "__main__":
    df = scrape_yf_sonata(max_pages=10, delay=2.0, save_csv=True)
    print(df.head(20))
    print(f"\n총 수집 건수: {len(df)}")

[INFO] 요청 중: https://car.encar.com/list/car?page=1&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._.%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._.%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D
[INFO] page=1, 수집건수=25
[INFO] 요청 중: https://car.encar.com/list/car?page=2&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._.%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._.%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D
[INFO] page=2, 수집건수=0
[INFO] 빈 페이지 또는 파싱 실패로 판단, 중단합니다.
[INFO] 저장 완료: encar_yf_sonata.csv
                                                  차량명            연식  주행거리_km  \
0   차량검색 확장메뉴열기 최근 업데이트순 검색 조건 변경 0 대 검색조건 적용됨 확장메...  09/10식(10년형)    98

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

SEARCH_URL_TEMPLATE = (
    "https://car.encar.com/list/car?page={page}"
    "&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._."
    "%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._."
    "%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C"
    "%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D"
)

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://car.encar.com/"
}

REGIONS = [
    "서울", "경기", "인천", "부산", "대구", "대전", "광주", "울산", "세종",
    "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"
]

FUELS = [
    "LPG(일반인 구입)", "가솔린+전기", "하이브리드", "가솔린", "디젤", "LPG", "전기"
]


def normalize_text(text: str):
    return re.sub(r"\s+", " ", text).strip()


def extract_detail_link(tag):
    a_tags = tag.select('a[href*="fem.encar.com/cars/detail/"]')
    if not a_tags:
        return None

    href = a_tags[0].get("href", "").strip()
    return href if href else None


def extract_car_id(link: str):
    if not link:
        return None
    m = re.search(r"/cars/detail/(\d+)", link)
    return m.group(1) if m else None


def extract_price(text: str):
    m = re.search(r'(\d[\d,]*)\s*만원', text)
    return int(m.group(1).replace(",", "")) if m else None


def extract_mileage(text: str):
    m = re.search(r'(\d[\d,]*)\s*km', text)
    return int(m.group(1).replace(",", "")) if m else None


def extract_year(text: str):
    m = re.search(r'(\d{2}/\d{2}식(?:\(\d{2}년형\))?)', text)
    return m.group(1) if m else None


def extract_fuel(text: str):
    for fuel in FUELS:
        if fuel in text:
            return fuel
    return None


def extract_region(text: str):
    for region in REGIONS:
        if region in text:
            return region
    return None


def extract_name(text: str):
    """
    차량명은 'YF 쏘나타'부터 연식 직전까지 추출
    """
    m = re.search(r'(YF\s*쏘나타.*?)(?=\s+\d{2}/\d{2}식(?:\(\d{2}년형\))?)', text)
    if m:
        return normalize_text(m.group(1))
    return None


def is_valid_row(name, price, mileage, year_text):
    if not name or "YF 쏘나타" not in name:
        return False
    if price is None or mileage is None or year_text is None:
        return False
    return True


def fetch_page(page: int):
    url = SEARCH_URL_TEMPLATE.format(page=page)
    print(f"[INFO] 요청 중: {url}")
    resp = requests.get(url, headers=HEADERS, timeout=20)
    resp.raise_for_status()
    return resp.text


def parse_page(html: str):
    soup = BeautifulSoup(html, "html.parser")
    rows = []

    # detail 링크가 있는 a 태그를 먼저 찾고,
    # 그 부모 블록을 후보 카드로 본다
    detail_links = soup.select('a[href*="fem.encar.com/cars/detail/"]')
    seen_ids = set()

    for a in detail_links:
        href = a.get("href", "").strip()
        car_id = extract_car_id(href)
        if not car_id or car_id in seen_ids:
            continue

        # 너무 작은 a 태그 자체가 아니라 어느 정도 상위 블록까지 올려서 텍스트 확보
        card = a
        for _ in range(4):
            if card.parent:
                card = card.parent

        raw_text = normalize_text(card.get_text(" ", strip=True))

        name = extract_name(raw_text)
        year_text = extract_year(raw_text)
        mileage = extract_mileage(raw_text)
        fuel = extract_fuel(raw_text)
        region = extract_region(raw_text)
        price = extract_price(raw_text)

        if not is_valid_row(name, price, mileage, year_text):
            continue

        rows.append({
            "매물ID": car_id,
            "차량명": name,
            "연식": year_text,
            "주행거리_km": mileage,
            "연료": fuel,
            "지역": region,
            "가격_만원": price,
            "상세링크": href,
            "raw_text": raw_text
        })
        seen_ids.add(car_id)

    return pd.DataFrame(rows)


def scrape_yf_sonata(max_pages=5, delay=1.5):
    all_dfs = []

    for page in range(1, max_pages + 1):
        try:
            html = fetch_page(page)
            df_page = parse_page(html)

            print(f"[INFO] page={page}, 수집건수={len(df_page)}")

            if df_page.empty:
                print("[INFO] 더 이상 유효한 매물이 없어서 중단합니다.")
                break

            all_dfs.append(df_page)
            time.sleep(delay)

        except Exception as e:
            print(f"[ERROR] page={page} 수집 실패: {e}")
            break

    if not all_dfs:
        return pd.DataFrame()

    df = pd.concat(all_dfs, ignore_index=True)
    df = df.drop_duplicates(subset=["매물ID"]).reset_index(drop=True)
    df.to_csv("encar_yf_sonata_clean.csv", index=False, encoding="utf-8-sig")
    print("[INFO] 저장 완료: encar_yf_sonata_clean.csv")
    return df


if __name__ == "__main__":
    df = scrape_yf_sonata(max_pages=10, delay=2.0)
    print(df.head(20))
    print(f"\n총 수집 건수: {len(df)}")

[INFO] 요청 중: https://car.encar.com/list/car?page=1&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._.%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._.%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D
[INFO] page=1, 수집건수=9
[INFO] 요청 중: https://car.encar.com/list/car?page=2&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._.%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._.%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D
[INFO] page=2, 수집건수=0
[INFO] 더 이상 유효한 매물이 없어서 중단합니다.
[INFO] 저장 완료: encar_yf_sonata_clean.csv
       매물ID                 차량명            연식  주행거리_km           연료  지역  \
0  39486507        YF 쏘나타 LPI 탑  10/11식(11년형)   158915  LPG(일반인 구입)  서울   
1

In [5]:
# pip install selenium pandas webdriver-manager

import re
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager


SEARCH_URL = (
    "https://car.encar.com/list/car?page=1"
    "&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._."
    "%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._."
    "%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C"
    "%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D"
)

REGIONS = [
    "서울", "경기", "인천", "부산", "대구", "대전", "광주", "울산", "세종",
    "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"
]

FUELS = [
    "LPG(일반인 구입)", "가솔린+전기", "하이브리드", "가솔린", "디젤", "LPG", "전기"
]


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def extract_car_id(url: str):
    m = re.search(r"/cars/detail/(\d+)", url)
    return m.group(1) if m else None


def extract_name(text: str):
    m = re.search(r"(YF\s*쏘나타.*?)(?=\s+\d{2}/\d{2}식(?:\(\d{2}년형\))?)", text)
    return normalize_text(m.group(1)) if m else None


def extract_year(text: str):
    m = re.search(r"(\d{2}/\d{2}식(?:\(\d{2}년형\))?)", text)
    return m.group(1) if m else None


def extract_mileage(text: str):
    m = re.search(r"(\d[\d,]*)\s*km", text)
    return int(m.group(1).replace(",", "")) if m else None


def extract_price(text: str):
    m = re.search(r"(\d[\d,]*)\s*만원", text)
    return int(m.group(1).replace(",", "")) if m else None


def extract_fuel(text: str):
    for fuel in FUELS:
        if fuel in text:
            return fuel
    return None


def extract_region(text: str):
    for region in REGIONS:
        if region in text:
            return region
    return None


def setup_driver():
    options = Options()
    # options.add_argument("--headless=new")  # 필요하면 주석 해제
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--lang=ko-KR")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    driver.implicitly_wait(3)
    return driver


def scroll_to_bottom(driver, pause=2.0, max_rounds=15):
    last_height = driver.execute_script("return document.body.scrollHeight")

    for i in range(max_rounds):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)

        new_height = driver.execute_script("return document.body.scrollHeight")
        print(f"[INFO] scroll round {i+1}: {last_height} -> {new_height}")

        if new_height == last_height:
            print("[INFO] 더 이상 스크롤로 로드되는 내용이 없어 보입니다.")
            break

        last_height = new_height


def collect_detail_urls(driver):
    anchors = driver.find_elements(By.CSS_SELECTOR, 'a[href*="fem.encar.com/cars/detail/"]')

    urls = []
    for a in anchors:
        href = a.get_attribute("href")
        if href and "/cars/detail/" in href:
            urls.append(href.split("&advClickPosition=")[0])

    urls = list(dict.fromkeys(urls))
    print(f"[INFO] 상세 링크 수집: {len(urls)}건")
    return urls


def scrape_from_loaded_page(driver, detail_urls):
    rows = []

    for idx, url in enumerate(detail_urls, start=1):
        try:
            car_id = extract_car_id(url)
            print(f"[INFO] ({idx}/{len(detail_urls)}) 수집 중: {car_id}")

            driver.get(url)
            time.sleep(2)

            text = normalize_text(driver.find_element(By.TAG_NAME, "body").text)

            name = extract_name(text)
            year_text = extract_year(text)
            mileage = extract_mileage(text)
            fuel = extract_fuel(text)
            region = extract_region(text)
            price = extract_price(text)

            if not name or not year_text or price is None or mileage is None:
                print(f"[WARN] 핵심 필드 누락, 스킵: {car_id}")
                continue

            rows.append({
                "매물ID": car_id,
                "차량명": name,
                "연식": year_text,
                "주행거리_km": mileage,
                "연료": fuel,
                "지역": region,
                "가격_만원": price,
                "상세링크": url,
                "raw_text": text[:3000]
            })

        except Exception as e:
            print(f"[ERROR] {url} 수집 실패: {e}")

    df = pd.DataFrame(rows).drop_duplicates(subset=["매물ID"]).reset_index(drop=True)
    return df


def main():
    driver = setup_driver()

    try:
        print("[INFO] 검색 페이지 접속")
        driver.get(SEARCH_URL)
        time.sleep(3)

        scroll_to_bottom(driver, pause=2.0, max_rounds=20)
        detail_urls = collect_detail_urls(driver)

        df = scrape_from_loaded_page(driver, detail_urls)
        df.to_csv("encar_yf_sonata_selenium.csv", index=False, encoding="utf-8-sig")

        print(df.head(20))
        print(f"\n총 수집 건수: {len(df)}")
        print("[INFO] 저장 완료: encar_yf_sonata_selenium.csv")

    finally:
        driver.quit()


if __name__ == "__main__":
    main()

[INFO] 검색 페이지 접속
[INFO] scroll round 1: 71394 -> 71394
[INFO] 더 이상 스크롤로 로드되는 내용이 없어 보입니다.
[INFO] 상세 링크 수집: 210건
[INFO] (1/210) 수집 중: 40903390
[INFO] (2/210) 수집 중: 41509017
[INFO] (3/210) 수집 중: 40477859
[INFO] (4/210) 수집 중: 39486507
[INFO] (5/210) 수집 중: 41054615
[INFO] (6/210) 수집 중: 41225357
[INFO] (7/210) 수집 중: 40955073
[INFO] (8/210) 수집 중: 41265617
[INFO] (9/210) 수집 중: 41310694
[INFO] (10/210) 수집 중: 40955073
[INFO] (11/210) 수집 중: 41643943
[INFO] (12/210) 수집 중: 39444771
[INFO] (13/210) 수집 중: 41639147
[INFO] (14/210) 수집 중: 41019165
[INFO] (15/210) 수집 중: 41054615
[INFO] (16/210) 수집 중: 41038293
[INFO] (17/210) 수집 중: 41018137
[INFO] (18/210) 수집 중: 41225133
[INFO] (19/210) 수집 중: 41265648
[INFO] (20/210) 수집 중: 41300549
[INFO] (21/210) 수집 중: 41576521
[INFO] (22/210) 수집 중: 39501982
[INFO] (23/210) 수집 중: 40512700
[INFO] (24/210) 수집 중: 40041895
[INFO] (25/210) 수집 중: 41508073
[INFO] (26/210) 수집 중: 40412711
[INFO] (27/210) 수집 중: 39979642
[INFO] (28/210) 수집 중: 41265617
[INFO] (29/210) 수집 중: 4133899

In [6]:
import re
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options


SEARCH_URL = (
    "https://car.encar.com/list/car?page=1"
    "&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._."
    "%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._."
    "%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C"
    "%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D"
)

REGIONS = [
    "서울", "경기", "인천", "부산", "대구", "대전", "광주", "울산", "세종",
    "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"
]

FUELS = [
    "LPG(일반인 구입)", "가솔린+전기", "하이브리드", "가솔린", "디젤", "LPG", "전기", "수소", "CNG"
]

TRANSMISSIONS = [
    "오토", "자동", "수동", "세미오토", "CVT"
]

POPULAR_COLORS = [
    "흰색", "화이트", "검정", "블랙", "쥐색", "회색", "그레이", "은색", "실버"
]


def setup_driver(headless=False):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--lang=ko-KR")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=options)
    driver.implicitly_wait(3)
    return driver


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def extract_car_id(url: str):
    m = re.search(r"/cars/detail/(\d+)", url)
    return m.group(1) if m else None


def scroll_until_stable(driver, pause=2.0, max_rounds=20):
    last_height = driver.execute_script("return document.body.scrollHeight")
    stable_count = 0

    for i in range(max_rounds):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)

        new_height = driver.execute_script("return document.body.scrollHeight")
        print(f"[SCROLL] {i+1}회차: {last_height} -> {new_height}")

        if new_height == last_height:
            stable_count += 1
        else:
            stable_count = 0

        if stable_count >= 2:
            print("[INFO] 스크롤 로딩 종료로 판단")
            break

        last_height = new_height


def collect_detail_links(driver):
    anchors = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/cars/detail/"]')
    urls = []

    for a in anchors:
        href = a.get_attribute("href")
        if href and "/cars/detail/" in href:
            href = href.split("&advClickPosition=")[0]
            urls.append(href)

    urls = list(dict.fromkeys(urls))
    return urls


# ---------- 텍스트 파싱 함수들 ----------

def extract_name(text: str):
    text = normalize_text(text)

    # YF 쏘나타부터 연식 전까지
    m = re.search(r"(YF\s*쏘나타.*?)(?=\s+\d{2}/\d{2}식(?:\(\d{2}년형\))?)", text)
    if m:
        return m.group(1).strip()

    return None


def split_model_trim(name: str):
    if not name:
        return None, None, None

    manufacturer = "현대"  # 현재는 YF 쏘나타 전용
    model = "YF 쏘나타"

    trim = name.replace("YF 쏘나타", "").strip()
    trim = trim if trim else None

    return manufacturer, model, trim


def extract_year(text: str):
    m = re.search(r"(\d{2}/\d{2}식(?:\(\d{2}년형\))?)", text)
    return m.group(1) if m else None


def convert_year_text(year_text: str):
    """
    10/05식 -> 2010
    12/04식 -> 2012
    """
    if not year_text:
        return None
    m = re.search(r"(\d{2})/\d{2}식", year_text)
    if not m:
        return None
    yy = int(m.group(1))
    return 2000 + yy


def extract_mileage(text: str):
    m = re.search(r"(\d[\d,]*)\s*km", text, re.IGNORECASE)
    return int(m.group(1).replace(",", "")) if m else None


def extract_price(text: str):
    # 가장 처음 등장하는 만원 단위 가격
    m = re.search(r"(\d[\d,]*)\s*만원", text)
    return int(m.group(1).replace(",", "")) if m else None


def extract_fuel(text: str):
    for fuel in FUELS:
        if fuel in text:
            return fuel
    return None


def extract_region(text: str):
    for region in REGIONS:
        if region in text:
            return region
    return None


def extract_transmission(text: str):
    for tm in TRANSMISSIONS:
        if tm in text:
            return tm
    return None


def extract_displacement_cc(text: str):
    patterns = [
        r"(\d{3,4})\s*cc",
        r"배기량\s*(\d{3,4})",
    ]
    for p in patterns:
        m = re.search(p, text, re.IGNORECASE)
        if m:
            return int(m.group(1))
    return None


def extract_color(text: str):
    """
    아주 정교하진 않지만, 우선순위 높은 방식:
    1) '색상' 키워드 주변 탐색
    2) 주요 색상 키워드 탐색
    """
    # 예: 색상 흰색 / 외장색 검정
    m = re.search(r"(?:색상|외장색)\s*[: ]?\s*([가-힣A-Za-z]+)", text)
    if m:
        return m.group(1).strip()

    for c in POPULAR_COLORS:
        if c in text:
            return c

    return None


def extract_accident_flag(text: str):
    """
    사고 여부를 단순 규칙으로 추출
    """
    text = normalize_text(text)

    positive_no_accident = [
        "무사고",
        "완전무사고"
    ]
    negative_accident = [
        "사고",
        "교환",
        "수리"
    ]

    # 무사고가 있으면 우선 긍정
    for kw in positive_no_accident:
        if kw in text:
            return "무사고"

    # 사고 관련 문구가 있으면 사고이력 추정
    for kw in negative_accident:
        if kw in text:
            return "사고/수리이력 의심"

    return None


def extract_flood_flag(text: str):
    text = normalize_text(text)

    if "침수이력 없음" in text or "침수 없음" in text:
        return "침수없음"
    if "침수" in text:
        return "침수이력 의심"

    return None


def extract_usage_history(text: str):
    text = normalize_text(text)

    found = []
    if "렌트" in text or "렌터카" in text:
        found.append("렌트이력")
    if "영업용" in text:
        found.append("영업용이력")
    if "법인" in text:
        found.append("법인이력")

    if found:
        return ",".join(found)
    return None


def extract_owner_change_count(text: str):
    patterns = [
        r"소유자 변경\s*(\d+)\s*회",
        r"소유자변경\s*(\d+)\s*회",
        r"명의 변경\s*(\d+)\s*회"
    ]
    for p in patterns:
        m = re.search(p, text)
        if m:
            return int(m.group(1))
    return None


def extract_options(text: str):
    option_keywords = [
        "선루프", "내비", "네비", "스마트키", "후방카메라",
        "열선시트", "통풍시트", "가죽시트", "메모리시트",
        "크루즈컨트롤", "스마트크루즈", "차선이탈", "어라운드뷰", "블랙박스"
    ]
    found = [kw for kw in option_keywords if kw in text]
    return ",".join(sorted(set(found))) if found else None


def parse_detail_text(text: str, url: str):
    text = normalize_text(text)

    name = extract_name(text)
    manufacturer, model, trim = split_model_trim(name)

    year_text = extract_year(text)

    row = {
        "매물ID": extract_car_id(url),
        "제조사": manufacturer,
        "모델": model,
        "세부트림": trim,
        "차량명": name,
        "연식_원문": year_text,
        "연식": convert_year_text(year_text),
        "주행거리_km": extract_mileage(text),
        "연료": extract_fuel(text),
        "변속기": extract_transmission(text),
        "배기량_cc": extract_displacement_cc(text),
        "색상": extract_color(text),
        "지역": extract_region(text),
        "가격_만원": extract_price(text),
        "사고유무": extract_accident_flag(text),
        "침수유무": extract_flood_flag(text),
        "용도이력": extract_usage_history(text),
        "소유자변경횟수": extract_owner_change_count(text),
        "옵션원문": extract_options(text),
        "상세링크": url,
        "raw_text": text
    }
    return row


# ---------- 메인 파이프라인 ----------

def collect_yf_links(driver):
    print("[INFO] 검색 페이지 접속")
    driver.get(SEARCH_URL)
    time.sleep(3)

    scroll_until_stable(driver, pause=2.0, max_rounds=20)
    urls = collect_detail_links(driver)

    # YF 쏘나타 관련 링크만 1차 필터
    print(f"[INFO] 상세 링크 총 {len(urls)}건 수집")
    return urls


def scrape_detail_pages(driver, urls, max_items=None):
    rows = []

    target_urls = urls[:max_items] if max_items else urls

    for idx, url in enumerate(target_urls, start=1):
        try:
            car_id = extract_car_id(url)
            print(f"[INFO] ({idx}/{len(target_urls)}) 수집 중: {car_id}")

            driver.get(url)
            time.sleep(2.5)

            body_text = driver.find_element(By.TAG_NAME, "body").text
            text = normalize_text(body_text)

            row = parse_detail_text(text, url)
            rows.append(row)

        except Exception as e:
            print(f"[ERROR] {url} 수집 실패: {e}")

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df.drop_duplicates(subset=["매물ID"]).reset_index(drop=True)

    return df


def clean_dataframe(df: pd.DataFrame):
    if df.empty:
        return df

    # YF 쏘나타가 아닌 것 제거
    df = df[df["차량명"].fillna("").str.contains("YF 쏘나타", na=False)].copy()

    # 가격 / 연식 / 주행거리 없는 행은 우선 제거
    df = df.dropna(subset=["가격_만원", "연식", "주행거리_km"])

    # 이상치 아주 거칠게 정리
    df = df[(df["가격_만원"] > 0) & (df["가격_만원"] < 5000)]
    df = df[(df["주행거리_km"] >= 0) & (df["주행거리_km"] < 500000)]
    df = df[(df["연식"] >= 2009) & (df["연식"] <= 2014)]

    df = df.reset_index(drop=True)
    return df


def main():
    driver = setup_driver(headless=False)

    try:
        detail_urls = collect_yf_links(driver)

        # 처음엔 10~15개만 시험해보는 게 좋아
        df_raw = scrape_detail_pages(driver, detail_urls, max_items=15)

        print("\n[INFO] 원본 추출 결과")
        print(df_raw.head())

        df_clean = clean_dataframe(df_raw)

        print("\n[INFO] 정리 후 결과")
        print(df_clean.head())
        print(f"\n총 수집 건수: {len(df_clean)}")

        df_raw.to_csv("encar_yf_sonata_detail_raw.csv", index=False, encoding="utf-8-sig")
        df_clean.to_csv("encar_yf_sonata_detail_clean.csv", index=False, encoding="utf-8-sig")

        print("[INFO] 저장 완료:")
        print("- encar_yf_sonata_detail_raw.csv")
        print("- encar_yf_sonata_detail_clean.csv")

    finally:
        driver.quit()


if __name__ == "__main__":
    main()

[INFO] 검색 페이지 접속
[SCROLL] 1회차: 71436 -> 71436
[SCROLL] 2회차: 71436 -> 71436
[INFO] 스크롤 로딩 종료로 판단
[INFO] 상세 링크 총 210건 수집
[INFO] (1/15) 수집 중: 41054615
[INFO] (2/15) 수집 중: 39486507
[INFO] (3/15) 수집 중: 40903390
[INFO] (4/15) 수집 중: 41225357
[INFO] (5/15) 수집 중: 40477859
[INFO] (6/15) 수집 중: 41509017
[INFO] (7/15) 수집 중: 40955073
[INFO] (8/15) 수집 중: 41310694
[INFO] (9/15) 수집 중: 40955073
[INFO] (10/15) 수집 중: 41265617
[INFO] (11/15) 수집 중: 41019165
[INFO] (12/15) 수집 중: 41499870
[INFO] (13/15) 수집 중: 41054615
[INFO] (14/15) 수집 중: 41038293
[INFO] (15/15) 수집 중: 41018137

[INFO] 원본 추출 결과
       매물ID 제조사      모델               세부트림                      차량명   연식_원문  \
0  41054615  현대  YF 쏘나타        CVVL 럭셔리 연식        YF 쏘나타CVVL 럭셔리 연식  12/04식   
1  39486507  현대  YF 쏘나타           LPI 탑 연식           YF 쏘나타LPI 탑 연식  10/11식   
2  40903390  현대  YF 쏘나타  LPI 프리미어(장애인용) 연식  YF 쏘나타LPI 프리미어(장애인용) 연식  10/01식   
3  41225357  현대  YF 쏘나타         Y20 프라임 연식         YF 쏘나타Y20 프라임 연식  10/09식   
4  40477859  현대  YF 쏘나타  LPI

In [7]:
import pandas as pd
import re

df = pd.read_csv("encar_yf_sonata_detail_clean.csv")

def clean_name(name):
    if pd.isna(name):
        return None
    name = str(name)
    name = re.sub(r'\s*연식$', '', name).strip()
    name = re.sub(r'YF\s*쏘나타', 'YF 쏘나타 ', name).strip()
    name = re.sub(r'\s+', ' ', name)
    return name

def split_trim(name):
    if pd.isna(name):
        return None
    trim = str(name).replace("YF 쏘나타", "").strip()
    return trim if trim else None

def normalize_color(color):
    if pd.isna(color):
        return None
    color = str(color).strip()
    mapping = {
        "화이트": "흰색",
        "블랙": "검정",
        "실버": "은색",
        "그레이": "회색"
    }
    return mapping.get(color, color)

def normalize_accident(val):
    if pd.isna(val):
        return None
    val = str(val).strip()
    if val == "무사고":
        return 0
    if "사고" in val or "수리" in val:
        return 1
    return None

df["차량명"] = df["차량명"].apply(clean_name)
df["세부트림"] = df["차량명"].apply(split_trim)
df["색상"] = df["색상"].apply(normalize_color)
df["사고여부_flag"] = df["사고유무"].apply(normalize_accident)

# 옵션 중복 정리
df["옵션원문"] = df["옵션원문"].astype(str).str.replace("내비,네비", "내비", regex=False)
df["옵션원문"] = df["옵션원문"].str.replace("네비,내비", "내비", regex=False)

# 기본 확인
print(df[["차량명", "세부트림", "색상", "사고유무", "사고여부_flag"]].head(10))

df.to_csv("encar_yf_sonata_detail_refined.csv", index=False, encoding="utf-8-sig")
print("저장 완료: encar_yf_sonata_detail_refined.csv")

                     차량명            세부트림    색상        사고유무  사고여부_flag
0        YF 쏘나타 CVVL 럭셔리        CVVL 럭셔리  None         무사고          0
1           YF 쏘나타 LPI 탑           LPI 탑  None  사고/수리이력 의심          1
2  YF 쏘나타 LPI 프리미어(장애인용)  LPI 프리미어(장애인용)  None         무사고          0
3         YF 쏘나타 Y20 프라임         Y20 프라임  None  사고/수리이력 의심          1
4  YF 쏘나타 LPI 프리미어(장애인용)  LPI 프리미어(장애인용)    은색         무사고          0
5       YF 쏘나타 Y20 프라임블랙       Y20 프라임블랙    흰색         무사고          0
6      YF 쏘나타 Y20 프라임고급형      Y20 프라임고급형  None  사고/수리이력 의심          1
7         YF 쏘나타 LPI 럭셔리         LPI 럭셔리  None  사고/수리이력 의심          1
8         YF 쏘나타 Y20 럭셔리         Y20 럭셔리  None  사고/수리이력 의심          1
9       YF 쏘나타 Y20 프라임블랙       Y20 프라임블랙    검정         무사고          0
저장 완료: encar_yf_sonata_detail_refined.csv


In [9]:
import re
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options


SEARCH_URL = (
    "https://car.encar.com/list/car?page=1"
    "&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._."
    "%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._."
    "%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C"
    "%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D"
)

REGIONS = [
    "서울", "경기", "인천", "부산", "대구", "대전", "광주", "울산", "세종",
    "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"
]

FUELS = [
    "LPG(일반인 구입)", "가솔린+전기", "하이브리드", "가솔린", "디젤", "LPG", "전기", "수소", "CNG"
]

TRANSMISSIONS = [
    "오토", "자동", "수동", "세미오토", "CVT"
]

COLOR_MAP = {
    "화이트": "흰색",
    "흰색": "흰색",
    "블랙": "검정",
    "검정": "검정",
    "검은색": "검정",
    "실버": "은색",
    "은색": "은색",
    "그레이": "회색",
    "회색": "회색",
    "쥐색": "회색",
    "청색": "파랑",
    "파랑": "파랑",
    "블루": "파랑",
    "빨강": "빨강",
    "레드": "빨강",
    "진주색": "진주",
    "진주": "진주",
    "베이지": "베이지",
    "브라운": "갈색",
    "갈색": "갈색"
}


# ---------------------------
# 드라이버
# ---------------------------

def setup_driver(headless=False):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--lang=ko-KR")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=options)
    driver.implicitly_wait(3)
    return driver


# ---------------------------
# 유틸
# ---------------------------

def normalize_text(text):
    if text is None:
        return None
    return re.sub(r"\s+", " ", str(text)).strip()


def extract_car_id(url):
    if not url:
        return None
    m = re.search(r"/cars/detail/(\d+)", url)
    return m.group(1) if m else None


def safe_text(element):
    try:
        return normalize_text(element.text)
    except:
        return None


def unique_join(text_list):
    cleaned = []
    seen = set()

    for t in text_list:
        t = normalize_text(t)
        if not t:
            continue
        if t not in seen:
            cleaned.append(t)
            seen.add(t)

    return " || ".join(cleaned) if cleaned else None


# ---------------------------
# 목록 페이지
# ---------------------------

def scroll_until_stable(driver, pause=2.0, max_rounds=20):
    last_height = driver.execute_script("return document.body.scrollHeight")
    stable_count = 0

    for i in range(max_rounds):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)

        new_height = driver.execute_script("return document.body.scrollHeight")
        print(f"[SCROLL] {i+1}회차: {last_height} -> {new_height}")

        if new_height == last_height:
            stable_count += 1
        else:
            stable_count = 0

        if stable_count >= 2:
            print("[INFO] 스크롤 종료로 판단")
            break

        last_height = new_height


def collect_detail_links(driver):
    anchors = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/cars/detail/"]')
    urls = []

    for a in anchors:
        try:
            href = a.get_attribute("href")
            if href and "/cars/detail/" in href:
                href = href.split("&advClickPosition=")[0]
                urls.append(href)
        except:
            pass

    urls = list(dict.fromkeys(urls))
    return urls


def collect_yf_links(driver):
    print("[INFO] 검색 페이지 접속")
    driver.get(SEARCH_URL)
    time.sleep(3)

    scroll_until_stable(driver, pause=2.0, max_rounds=20)
    urls = collect_detail_links(driver)

    print(f"[INFO] 상세 링크 {len(urls)}건 수집")
    return urls


# ---------------------------
# 상세페이지 섹션 수집
# ---------------------------

def collect_candidate_sections(driver):
    """
    페이지 구조가 바뀌어도 최대한 버티도록
    헤더/제목/테이블/리스트 기반 후보를 넓게 수집한다.
    """
    sections = {
        "기본정보_text": [],
        "차량상태_text": [],
        "옵션_text": [],
        "설명_text": []
    }

    all_blocks = driver.find_elements(By.CSS_SELECTOR, "section, article, div, ul, table")

    for block in all_blocks:
        txt = safe_text(block)
        if not txt or len(txt) < 20:
            continue

        low = txt.lower()

        # 기본정보 후보
        if any(k in txt for k in ["연식", "주행거리", "연료", "변속기", "배기량", "색상", "지역", "차종"]):
            sections["기본정보_text"].append(txt)

        # 차량상태 후보
        if any(k in txt for k in ["무사고", "사고", "교환", "판금", "수리", "침수", "용도", "렌트", "법인", "영업용", "소유자"]):
            sections["차량상태_text"].append(txt)

        # 옵션 후보
        if any(k in txt for k in ["옵션", "선루프", "내비", "네비", "스마트키", "열선", "통풍", "후방카메라", "가죽시트"]):
            sections["옵션_text"].append(txt)

        # 설명 후보
        if any(k in txt for k in ["차량설명", "판매자", "딜러", "특이사항", "강조", "장점"]):
            sections["설명_text"].append(txt)

    # body 전체도 보존
    sections["기본정보_text"] = unique_join(sections["기본정보_text"])
    sections["차량상태_text"] = unique_join(sections["차량상태_text"])
    sections["옵션_text"] = unique_join(sections["옵션_text"])
    sections["설명_text"] = unique_join(sections["설명_text"])

    return sections


# ---------------------------
# 라벨 기반 추출
# ---------------------------

def find_label_value(text, labels, value_pattern=r"([^\|\n\r]+)"):
    if not text:
        return None

    for label in labels:
        patterns = [
            rf"{label}\s*[:：]?\s*{value_pattern}",
            rf"{label}\s+{value_pattern}",
        ]
        for p in patterns:
            m = re.search(p, text, re.IGNORECASE)
            if m:
                return normalize_text(m.group(1))
    return None


def extract_name(text):
    if not text:
        return None

    m = re.search(r"(YF\s*쏘나타.*?)(?=\s+\d{2}/\d{2}식(?:\(\d{2}년형\))?)", text)
    if m:
        name = normalize_text(m.group(1))
        name = re.sub(r"\s*연식$", "", name).strip()
        return name

    return None


def split_model_trim(name):
    if not name:
        return None, None, None

    manufacturer = "현대"
    model = "YF 쏘나타"
    trim = normalize_text(name.replace("YF 쏘나타", "")) or None

    return manufacturer, model, trim


def extract_year_text(text):
    if not text:
        return None
    m = re.search(r"(\d{2}/\d{2}식(?:\(\d{2}년형\))?)", text)
    return m.group(1) if m else None


def convert_year_text(year_text):
    if not year_text:
        return None
    m = re.search(r"(\d{2})/\d{2}식", year_text)
    if not m:
        return None
    yy = int(m.group(1))
    return 2000 + yy


def extract_mileage(text):
    if not text:
        return None
    m = re.search(r"(\d[\d,]*)\s*km", text, re.IGNORECASE)
    return int(m.group(1).replace(",", "")) if m else None


def extract_price(text):
    if not text:
        return None
    m = re.search(r"(\d[\d,]*)\s*만원", text)
    return int(m.group(1).replace(",", "")) if m else None


def extract_fuel(text):
    if not text:
        return None
    for fuel in FUELS:
        if fuel in text:
            return fuel
    return None


def extract_region(text):
    if not text:
        return None
    for region in REGIONS:
        if region in text:
            return region
    return None


def extract_transmission(text):
    if not text:
        return None

    label_value = find_label_value(text, ["변속기", "미션"], value_pattern=r"([가-힣A-Za-z]+)")
    if label_value:
        for tm in TRANSMISSIONS:
            if tm in label_value:
                return tm

    for tm in TRANSMISSIONS:
        if tm in text:
            return tm

    return None


def extract_displacement_cc(text):
    if not text:
        return None

    label_value = find_label_value(text, ["배기량"], value_pattern=r"(\d{3,4})\s*cc?")
    if label_value:
        m = re.search(r"(\d{3,4})", label_value)
        if m:
            return int(m.group(1))

    patterns = [
        r"(\d{3,4})\s*cc",
        r"배기량\s*(\d{3,4})"
    ]
    for p in patterns:
        m = re.search(p, text, re.IGNORECASE)
        if m:
            return int(m.group(1))

    return None


def normalize_color(color):
    if not color:
        return None
    color = normalize_text(color)
    return COLOR_MAP.get(color, color)


def extract_color(text):
    if not text:
        return None

    label_value = find_label_value(
        text,
        ["색상", "외장색", "외장 컬러"],
        value_pattern=r"([가-힣A-Za-z]+)"
    )
    if label_value:
        return normalize_color(label_value)

    for k, v in COLOR_MAP.items():
        if k in text:
            return v

    return None


def extract_accident_flag(condition_text):
    if not condition_text:
        return None

    t = normalize_text(condition_text)

    # 무사고 우선
    if any(k in t for k in ["완전무사고", "무사고"]):
        return "무사고"

    # 보다 보수적
    strong_signals = [
        "사고이력 있음",
        "교환",
        "판금",
        "수리이력",
        "수리 흔적"
    ]
    if any(k in t for k in strong_signals):
        return "사고이력"

    return None


def extract_flood_flag(condition_text):
    if not condition_text:
        return None

    t = normalize_text(condition_text)

    if "침수이력 없음" in t or "침수 없음" in t:
        return "침수없음"

    if "침수이력 있음" in t or "침수 차량" in t:
        return "침수이력"

    return None


def extract_usage_history(condition_text):
    if not condition_text:
        return None

    t = normalize_text(condition_text)
    found = []

    if "렌트" in t or "렌터카" in t:
        found.append("렌트이력")
    if "영업용" in t:
        found.append("영업용이력")
    if "법인" in t:
        found.append("법인이력")

    return ",".join(found) if found else None


def extract_owner_change_count(condition_text):
    if not condition_text:
        return None

    patterns = [
        r"소유자 변경\s*(\d+)\s*회",
        r"소유자변경\s*(\d+)\s*회",
        r"명의 변경\s*(\d+)\s*회"
    ]
    for p in patterns:
        m = re.search(p, condition_text)
        if m:
            return int(m.group(1))
    return None


def extract_options(option_text):
    if not option_text:
        return None

    option_keywords = [
        "선루프", "내비", "네비", "스마트키", "후방카메라",
        "열선시트", "통풍시트", "가죽시트", "메모리시트",
        "크루즈컨트롤", "스마트크루즈", "차선이탈", "어라운드뷰", "블랙박스"
    ]
    found = []

    for kw in option_keywords:
        if kw in option_text:
            found.append(kw)

    # 내비/네비 통합
    normalized = []
    for x in found:
        if x == "네비":
            x = "내비"
        normalized.append(x)

    normalized = sorted(set(normalized))
    return ",".join(normalized) if normalized else None


# ---------------------------
# 상세페이지 파싱
# ---------------------------

def parse_detail_page(driver, url):
    driver.get(url)
    time.sleep(2.5)

    raw_text = normalize_text(driver.find_element(By.TAG_NAME, "body").text)
    sections = collect_candidate_sections(driver)

    merged_basic = " ".join(filter(None, [sections["기본정보_text"], raw_text]))
    merged_condition = " ".join(filter(None, [sections["차량상태_text"], raw_text]))
    merged_option = sections["옵션_text"]
    merged_desc = sections["설명_text"]

    name = extract_name(raw_text)
    manufacturer, model, trim = split_model_trim(name)
    year_text = extract_year_text(raw_text)

    row = {
        "매물ID": extract_car_id(url),
        "차량명": name,
        "제조사": manufacturer,
        "모델": model,
        "세부트림": trim,
        "연식_원문": year_text,
        "연식": convert_year_text(year_text),
        "주행거리_km": extract_mileage(merged_basic),
        "연료": extract_fuel(merged_basic),
        "변속기": extract_transmission(merged_basic),
        "배기량_cc": extract_displacement_cc(merged_basic),
        "색상": extract_color(merged_basic),
        "지역": extract_region(merged_basic),
        "가격_만원": extract_price(raw_text),
        "사고유무": extract_accident_flag(merged_condition),
        "침수유무": extract_flood_flag(merged_condition),
        "용도이력": extract_usage_history(merged_condition),
        "소유자변경횟수": extract_owner_change_count(merged_condition),
        "옵션원문": extract_options(merged_option),
        "기본정보_text": sections["기본정보_text"],
        "차량상태_text": sections["차량상태_text"],
        "옵션_text": sections["옵션_text"],
        "설명_text": sections["설명_text"],
        "raw_text": raw_text,
        "상세링크": url
    }

    return row


# ---------------------------
# 후처리
# ---------------------------

def clean_dataframe(df):
    if df.empty:
        return df

    df = df.copy()

    df = df[df["차량명"].fillna("").str.contains("YF 쏘나타", na=False)]
    df = df.drop_duplicates(subset=["매물ID"]).reset_index(drop=True)

    # 기본 필드 필터
    df = df.dropna(subset=["가격_만원", "연식", "주행거리_km"])

    df = df[(df["가격_만원"] > 0) & (df["가격_만원"] < 5000)]
    df = df[(df["주행거리_km"] >= 0) & (df["주행거리_km"] < 500000)]
    df = df[(df["연식"] >= 2009) & (df["연식"] <= 2014)]

    # 색상 정규화
    df["색상"] = df["색상"].apply(normalize_color)

    return df.reset_index(drop=True)


# ---------------------------
# 메인
# ---------------------------

def main():
    driver = setup_driver(headless=False)

    try:
        urls = collect_yf_links(driver)

        # 처음에는 5~10개만 테스트 추천
        target_urls = urls[:8]
        print(f"[INFO] 테스트 대상: {len(target_urls)}건")

        rows = []
        for i, url in enumerate(target_urls, start=1):
            try:
                print(f"[INFO] ({i}/{len(target_urls)}) 수집 중: {extract_car_id(url)}")
                row = parse_detail_page(driver, url)
                rows.append(row)
            except Exception as e:
                print(f"[ERROR] {url} 실패: {e}")

        df_raw = pd.DataFrame(rows)
        df_clean = clean_dataframe(df_raw)

        print("\n[RAW HEAD]")
        print(df_raw.head())

        print("\n[CLEAN HEAD]")
        print(df_clean.head())

        print(f"\n원본 건수: {len(df_raw)}")
        print(f"정제 건수: {len(df_clean)}")

        df_raw.to_csv("encar_yf_sonata_section_raw.csv", index=False, encoding="utf-8-sig")
        df_clean.to_csv("encar_yf_sonata_section_clean.csv", index=False, encoding="utf-8-sig")

        print("\n[INFO] 저장 완료")
        print("- encar_yf_sonata_section_raw.csv")
        print("- encar_yf_sonata_section_clean.csv")

    finally:
        driver.quit()


if __name__ == "__main__":
    main()

[INFO] 검색 페이지 접속
[SCROLL] 1회차: 71502 -> 71502
[SCROLL] 2회차: 71502 -> 71502
[INFO] 스크롤 종료로 판단
[INFO] 상세 링크 210건 수집
[INFO] 테스트 대상: 8건
[INFO] (1/8) 수집 중: 40903390
[INFO] (2/8) 수집 중: 41509017
[INFO] (3/8) 수집 중: 39486507
[INFO] (4/8) 수집 중: 41225357
[INFO] (5/8) 수집 중: 41054615
[INFO] (6/8) 수집 중: 40955073
[INFO] (7/8) 수집 중: 40477859
[INFO] (8/8) 수집 중: 41265617

[RAW HEAD]
       매물ID                   차량명 제조사      모델            세부트림   연식_원문    연식  \
0  40903390  YF 쏘나타LPI 프리미어(장애인용)  현대  YF 쏘나타  LPI 프리미어(장애인용)  10/01식  2010   
1  41509017       YF 쏘나타Y20 프라임블랙  현대  YF 쏘나타       Y20 프라임블랙  10/10식  2010   
2  39486507           YF 쏘나타LPI 탑  현대  YF 쏘나타           LPI 탑  10/11식  2010   
3  41225357         YF 쏘나타Y20 프라임  현대  YF 쏘나타         Y20 프라임  10/09식  2010   
4  41054615        YF 쏘나타CVVL 럭셔리  현대  YF 쏘나타        CVVL 럭셔리  12/04식  2012   

   주행거리_km           연료 변속기  ...  침수유무  용도이력 소유자변경횟수  \
0   180575  LPG(일반인 구입)  자동  ...  None  None    None   
1   113663          가솔린  오토  ...  None  None 

In [10]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

url = "https://fem.encar.com/cars/detail/40124634?advClickPosition=mweb_mhightlight_g88_t400&listAdvType=mhighlight&type=detail&view_type=hs_ad"

options = Options()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=options)

driver.get(url)
time.sleep(3)

html = driver.page_source

with open("encar_detail_sample.html", "w", encoding="utf-8") as f:
    f.write(html)

print("저장 완료: encar_detail_sample.html")

driver.quit()

저장 완료: encar_detail_sample.html


In [3]:
import re
import time
import json
import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


SEARCH_URL = (
    "https://car.encar.com/list/car?page=1"
    "&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._."
    "%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._."
    "%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C"
    "%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D"
)


# -----------------------------
# 드라이버
# -----------------------------
def setup_driver(headless=False):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--lang=ko-KR")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=options)
    driver.implicitly_wait(3)
    return driver


def normalize_text(text):
    if text is None:
        return None
    return re.sub(r"\s+", " ", str(text)).strip()


def extract_car_id(url):
    m = re.search(r"/cars/detail/(\d+)", url)
    return m.group(1) if m else None


# -----------------------------
# 목록 링크 수집
# -----------------------------
def scroll_until_stable(driver, pause=2.0, max_rounds=20):
    last_height = driver.execute_script("return document.body.scrollHeight")
    stable_count = 0

    for i in range(max_rounds):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)

        new_height = driver.execute_script("return document.body.scrollHeight")
        print(f"[SCROLL] {i+1}: {last_height} -> {new_height}")

        if new_height == last_height:
            stable_count += 1
        else:
            stable_count = 0

        if stable_count >= 2:
            print("[INFO] 스크롤 종료로 판단")
            break

        last_height = new_height


def collect_detail_links(driver):
    anchors = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/cars/detail/"]')
    urls = []

    for a in anchors:
        try:
            href = a.get_attribute("href")
            if href and "/cars/detail/" in href:
                href = href.split("&advClickPosition=")[0]
                urls.append(href)
        except Exception:
            pass

    urls = list(dict.fromkeys(urls))
    return urls


def collect_yf_links(driver):
    print("[INFO] YF 쏘나타 검색 페이지 접속")
    driver.get(SEARCH_URL)
    time.sleep(3)

    scroll_until_stable(driver, pause=2.0, max_rounds=20)
    links = collect_detail_links(driver)

    print(f"[INFO] 상세 링크 수집 완료: {len(links)}건")
    return links


# -----------------------------
# 상세페이지 파싱 유틸
# -----------------------------
def parse_meta_description(soup):
    result = {}
    meta_desc = soup.find("meta", attrs={"name": "description"})
    if not meta_desc:
        return result

    content = meta_desc.get("content", "")
    patterns = {
        "연식_메타": r"연식:([^,]+)",
        "주행거리_메타": r"주행거리:([^,]+)",
        "연료_메타": r"연료:([^,]+)",
        "색상_메타": r"색상:([^,]+)",
        "지역_메타": r"지역:([^,]+?) 중고차",
    }

    for key, pat in patterns.items():
        m = re.search(pat, content)
        if m:
            result[key] = m.group(1).strip()

    return result


def parse_title(soup):
    h3 = soup.find("h3")
    if not h3:
        return None, None, None

    spans = [normalize_text(x.get_text(" ", strip=True)) for x in h3.find_all("span")]
    spans = [x for x in spans if x]

    if len(spans) >= 2:
        model = spans[0]
        trim = spans[1]
        full_name = f"{model} {trim}".strip()
        return full_name, model, trim

    full_name = normalize_text(h3.get_text(" ", strip=True))
    return full_name, None, None


def parse_basic_info(soup):
    result = {
        "연식_원문": None,
        "주행거리_원문": None,
        "연료": None,
        "차량번호": None,
        "등록번호": None,
        "조회수": None,
        "찜수": None,
        "해시태그": None,
        "실촬영여부": None,
    }

    # 기본정보 dl
    dl = soup.select_one("dl.ar1Ivd7EgX")
    if dl:
        dt_list = dl.find_all("dt")
        dd_list = dl.find_all("dd")

        pairs = []
        for dt, dd in zip(dt_list, dd_list):
            k = normalize_text(dt.get_text(" ", strip=True))
            v = normalize_text(dd.get_text(" ", strip=True))
            pairs.append((k, v))

        for k, v in pairs:
            if "연식" in k:
                result["연식_원문"] = v
            elif "주행거리" in k:
                result["주행거리_원문"] = v
            elif "연료" in k:
                result["연료"] = v
            elif "차량번호" in k:
                result["차량번호"] = v

    # 등록번호 / 조회수 / 찜
    info_lis = soup.select("ul.rVigc5A_1H li")
    for li in info_lis:
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt.startswith("등록번호"):
            result["등록번호"] = txt.replace("등록번호", "").strip()
        elif txt.startswith("조회수"):
            m = re.search(r"조회수\s*([\d,]+)", txt)
            if m:
                result["조회수"] = int(m.group(1).replace(",", ""))
        elif txt.startswith("찜"):
            m = re.search(r"찜\s*([\d,]+)", txt)
            if m:
                result["찜수"] = int(m.group(1).replace(",", ""))

    # 해시태그
    hashtags = [normalize_text(li.get_text(" ", strip=True)) for li in soup.select("ul.kYC8KS_YZ1 li")]
    hashtags = [x for x in hashtags if x]
    result["해시태그"] = ",".join(hashtags) if hashtags else None

    # 실촬영 여부
    shot = soup.select_one("p.s8FzQTBVpE")
    if shot:
        result["실촬영여부"] = normalize_text(shot.get_text(" ", strip=True))

    return result


def convert_year(year_text):
    if not year_text:
        return None
    m = re.search(r"(\d{2})/(\d{2})식", year_text)
    if not m:
        return None
    yy = int(m.group(1))
    return 2000 + yy


def convert_mileage(mileage_text):
    if not mileage_text:
        return None
    m = re.search(r"([\d,]+)", mileage_text)
    return int(m.group(1).replace(",", "")) if m else None


def parse_option_items(soup):
    """
    주요 옵션을 yes/no로 추출
    """
    option_dict = {}
    option_items = soup.select("ul.dz3qFYruNO li")

    for li in option_items:
        txt = normalize_text(li.get_text(" ", strip=True))
        if not txt:
            continue

        # blind span에 '있음/없음'이 들어가는 구조
        blind = li.select_one("span.blind")
        status = normalize_text(blind.get_text(" ", strip=True)) if blind else None

        # blind 텍스트 제거한 이름 추정
        name = txt
        if status:
            name = normalize_text(txt.replace(status, ""))

        # 줄바꿈/괄호 정리
        name = name.replace("\n", " ")
        name = normalize_text(name)

        if name:
            option_dict[name] = 1 if status == "있음" else 0 if status == "없음" else None

    return option_dict


def parse_vehicle_status(soup):
    result = {
        "차량키개수": None,
        "틴팅_앞유리": None,
        "앞타이어트레드_mm": None,
        "뒤타이어트레드_mm": None,
        "차량상태_raw": None,
    }

    items = []
    for li in soup.select("ul.IdK1KVYuyr li"):
        ps = li.find_all("p")
        vals = [normalize_text(p.get_text(" ", strip=True)) for p in ps]
        vals = [v for v in vals if v]
        if vals:
            items.append(" | ".join(vals))

        joined = " ".join(vals)

        if "차량 키 개수" in joined:
            m = re.search(r"(\d+)개", joined)
            if m:
                result["차량키개수"] = int(m.group(1))

        elif "틴팅(앞 유리)" in joined:
            result["틴팅_앞유리"] = 1 if "있음" in joined else 0 if "없음" in joined else None

        elif "타이어트레드 잔량 (앞)" in joined:
            m = re.search(r"(\d+)mm", joined)
            if m:
                result["앞타이어트레드_mm"] = int(m.group(1))

        elif "타이어트레드 잔량 (뒤)" in joined:
            m = re.search(r"(\d+)mm", joined)
            if m:
                result["뒤타이어트레드_mm"] = int(m.group(1))

    result["차량상태_raw"] = " || ".join(items) if items else None
    return result


def parse_performance_record(soup):
    result = {
        "교환": None,
        "판금": None,
        "부식": None,
        "제시번호": None,
        "성능기록부_raw": None,
    }

    # 제시번호
    p = soup.select_one("div.Hs3feBPsTj p.WjUuNLum7b")
    if p:
        txt = normalize_text(p.get_text(" ", strip=True))
        m = re.search(r"제시번호\s*:\s*(.+)", txt)
        if m:
            result["제시번호"] = m.group(1).strip()

    items = []
    for li in soup.select("ul.wB0X7nC0cq li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt:
            items.append(txt)

        if "교환" in txt:
            m = re.search(r"교환\s*(\d+)", txt)
            if m:
                result["교환"] = int(m.group(1))
        elif "판금" in txt:
            m = re.search(r"판금\s*(\d+)", txt)
            if m:
                result["판금"] = int(m.group(1))
        elif "부식" in txt:
            if "없음" in txt:
                result["부식"] = 0
            else:
                result["부식"] = 1

    result["성능기록부_raw"] = " || ".join(items) if items else None
    return result


def parse_history(soup):
    result = {
        "내차피해금액": None,
        "내차피해횟수": None,
        "타차가해금액": None,
        "타차가해횟수": None,
        "특이사항": None,
        "차량이력_raw": None,
    }

    items = []
    for li in soup.select("ul.OO25I4KzD2 li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt:
            items.append(txt)

        if "내차 피해" in txt:
            m = re.search(r"총\s*([\d,]+)원\s*\((\d+)회\)", txt)
            if m:
                result["내차피해금액"] = int(m.group(1).replace(",", ""))
                result["내차피해횟수"] = int(m.group(2))

        elif "타차 가해" in txt:
            m = re.search(r"총\s*([\d,]+)원\s*\((\d+)회\)", txt)
            if m:
                result["타차가해금액"] = int(m.group(1).replace(",", ""))
                result["타차가해횟수"] = int(m.group(2))

        elif "특이 사항" in txt:
            result["특이사항"] = txt.replace("특이 사항", "").strip()

    result["차량이력_raw"] = " || ".join(items) if items else None
    return result


def parse_seller_info(soup):
    result = {
        "판매자상호": None,
        "판매자명": None,
        "판매중대수": None,
        "판매완료대수": None,
        "판매자지역": None,
        "종사원증번호": None,
    }

    seller_wrap = soup.select_one("div.fSPJGX0WLk")
    if not seller_wrap:
        return result

    txt = normalize_text(seller_wrap.get_text(" ", strip=True))

    name_span = seller_wrap.select_one("span.St9bdeN3m2")
    seller_name = seller_wrap.select_one("strong.k8v1ohDI6G")

    if name_span:
        result["판매자상호"] = normalize_text(name_span.get_text(" ", strip=True))
    if seller_name:
        result["판매자명"] = normalize_text(seller_name.get_text(" ", strip=True))

    m1 = re.search(r"판매중\s*([\d,]+)", txt)
    if m1:
        result["판매중대수"] = int(m1.group(1).replace(",", ""))

    m2 = re.search(r"판매완료\s*([\d,]+)", txt)
    if m2:
        result["판매완료대수"] = int(m2.group(1).replace(",", ""))

    # 지역은 종사원증번호 앞쪽 li로 보임
    m3 = re.search(r"(서울|경기|인천|부산|대구|대전|광주|울산|세종|강원|충북|충남|전북|전남|경북|경남|제주)\s+[가-힣]+시?", txt)
    if m3:
        result["판매자지역"] = m3.group(0)

    m4 = re.search(r"종사원증번호\s*([0-9\-]+)", txt)
    if m4:
        result["종사원증번호"] = m4.group(1)

    return result


def try_click_expand_buttons(driver):
    """
    버튼을 눌렀을 때 더 많은 텍스트가 DOM에 붙는 경우를 노림.
    실패해도 그냥 넘어감.
    """
    candidate_texts = [
        "옵션모두보기",
        "성능기록부 자세히보기",
        "차량이력 자세히 보기",
        "차량관리상태 모두보기",
        "판매자 정보",
    ]

    for label in candidate_texts:
        try:
            btn = driver.find_element(By.XPATH, f"//button[contains(., '{label}')]")
            driver.execute_script("arguments[0].click();", btn)
            time.sleep(1.0)
        except Exception:
            pass


def parse_detail_page(driver, url):
    driver.get(url)
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )
    time.sleep(2)

    # 가능하면 더보기 버튼도 눌러보기
    try_click_expand_buttons(driver)

    page_source = driver.page_source
    soup = BeautifulSoup(page_source, "lxml")

    meta = parse_meta_description(soup)
    full_name, model, trim = parse_title(soup)
    basic = parse_basic_info(soup)
    options = parse_option_items(soup)
    status = parse_vehicle_status(soup)
    perf = parse_performance_record(soup)
    history = parse_history(soup)
    seller = parse_seller_info(soup)

    row = {
        "매물ID": extract_car_id(url),
        "차량명": full_name,
        "모델": model,
        "세부트림": trim,
        "연식_원문": basic["연식_원문"] or meta.get("연식_메타"),
        "연식": convert_year(basic["연식_원문"] or meta.get("연식_메타")),
        "주행거리_원문": basic["주행거리_원문"] or meta.get("주행거리_메타"),
        "주행거리_km": convert_mileage(basic["주행거리_원문"] or meta.get("주행거리_메타")),
        "연료": basic["연료"] or meta.get("연료_메타"),
        "색상": meta.get("색상_메타"),
        "지역": meta.get("지역_메타"),
        "차량번호": basic["차량번호"],
        "등록번호": basic["등록번호"],
        "조회수": basic["조회수"],
        "찜수": basic["찜수"],
        "해시태그": basic["해시태그"],
        "실촬영여부": basic["실촬영여부"],
        "상세링크": url,
        "페이지타이틀": normalize_text(soup.title.get_text(" ", strip=True)) if soup.title else None,
    }

    row.update(status)
    row.update(perf)
    row.update(history)
    row.update(seller)

    # 옵션을 열 단위로 펼치기
    for k, v in options.items():
        row[f"옵션_{k}"] = v

    # raw 보존
    row["옵션_raw"] = json.dumps(options, ensure_ascii=False)
    row["메타_raw"] = json.dumps(meta, ensure_ascii=False)

    return row


# -----------------------------
# 실행
# -----------------------------
def main():
    driver = setup_driver(headless=False)

    try:
        links = collect_yf_links(driver)
        target_links = links[:8]

        print(f"[INFO] 실제 수집 대상: {len(target_links)}건")

        rows = []
        for i, url in enumerate(target_links, start=1):
            try:
                print(f"[INFO] ({i}/{len(target_links)}) 수집 중: {extract_car_id(url)}")
                row = parse_detail_page(driver, url)
                rows.append(row)
            except Exception as e:
                print(f"[ERROR] {url} -> {e}")

        df = pd.DataFrame(rows)

        # JSON도 같이 저장
        with open("encar_yf_top8_full.json", "w", encoding="utf-8") as f:
            json.dump(rows, f, ensure_ascii=False, indent=2)

        df.to_csv("encar_yf_top8_full.csv", index=False, encoding="utf-8-sig")

        print("\n[INFO] 저장 완료")
        print("- encar_yf_top8_full.csv")
        print("- encar_yf_top8_full.json")
        print("\n[HEAD]")
        print(df.head())

    finally:
        driver.quit()


if __name__ == "__main__":
    main()

[INFO] YF 쏘나타 검색 페이지 접속
[SCROLL] 1: 76045 -> 76045
[SCROLL] 2: 76045 -> 76045
[INFO] 스크롤 종료로 판단
[INFO] 상세 링크 수집 완료: 210건
[INFO] 실제 수집 대상: 8건
[INFO] (1/8) 수집 중: 39486507
[INFO] (2/8) 수집 중: 40903390
[INFO] (3/8) 수집 중: 40955073
[INFO] (4/8) 수집 중: 41509017
[INFO] (5/8) 수집 중: 41054615
[INFO] (6/8) 수집 중: 40477859
[INFO] (7/8) 수집 중: 41225357
[INFO] (8/8) 수집 중: 41265617

[INFO] 저장 완료
- encar_yf_top8_full.csv
- encar_yf_top8_full.json

[HEAD]
       매물ID                    차량명      모델            세부트림  \
0  39486507           YF 쏘나타 LPI 탑  YF 쏘나타           LPI 탑   
1  40903390  YF 쏘나타 LPI 프리미어(장애인용)  YF 쏘나타  LPI 프리미어(장애인용)   
2  40955073         YF 쏘나타 Y20 프라임  YF 쏘나타         Y20 프라임   
3  41509017         YF 쏘나타 Y20 프라임  YF 쏘나타         Y20 프라임   
4  41054615        YF 쏘나타 CVVL 럭셔리  YF 쏘나타        CVVL 럭셔리   

                연식_원문    연식    주행거리_원문  주행거리_km           연료   색상  ...  \
0  10/11식 (11년형) 연형정보  2010  158,915km   158915  LPG(일반인 구입)   흰색  ...   
1         10/01식 연형정보  2010  180,575km   

In [4]:
def close_popups(driver):
    popup_xpaths = [
        "//button[contains(., '닫기')]",
        "//button[contains(., '나중에')]",
        "//button[contains(., '오늘 보지 않기')]",
        "//button[contains(., '취소')]",
        "//button[contains(., '확인')]",
    ]
    for xp in popup_xpaths:
        try:
            buttons = driver.find_elements(By.XPATH, xp)
            for btn in buttons:
                if btn.is_displayed():
                    driver.execute_script("arguments[0].click();", btn)
                    time.sleep(0.5)
        except Exception:
            pass

    # ESC도 한 번
    try:
        from selenium.webdriver.common.keys import Keys
        body = driver.find_element(By.TAG_NAME, "body")
        body.send_keys(Keys.ESCAPE)
        time.sleep(0.3)
    except Exception:
        pass


def expand_all_possible(driver):
    labels = [
        "옵션모두보기",
        "성능기록부 자세히보기",
        "차량이력 자세히 보기",
        "차량관리상태 모두보기",
        "판매자 정보",
        "더보기",
        "자세히",
    ]
    for label in labels:
        try:
            buttons = driver.find_elements(By.XPATH, f"//button[contains(., '{label}')]")
            for btn in buttons:
                if btn.is_displayed():
                    driver.execute_script("arguments[0].click();", btn)
                    time.sleep(0.8)
                    close_popups(driver)
        except Exception:
            pass


def parse_new_car_price_from_text(text):
    patterns = [
        r"신차가\s*[:：]?\s*([\d,]+)\s*만원",
        r"출고가\s*[:：]?\s*([\d,]+)\s*만원",
        r"신차 가격\s*[:：]?\s*([\d,]+)\s*만원",
        r"현재 신차가\s*[:：]?\s*([\d,]+)\s*만원",
    ]
    for pat in patterns:
        m = re.search(pat, text)
        if m:
            return int(m.group(1).replace(",", ""))
    return None


def get_new_car_price(driver):
    # 팝업 닫고 더보기 먼저
    close_popups(driver)
    expand_all_possible(driver)

    body_text = driver.find_element(By.TAG_NAME, "body").text
    price = parse_new_car_price_from_text(body_text)
    if price is not None:
        return price

    # page_source에서도 한 번 더
    html = driver.page_source
    m = re.search(r"(신차가|출고가|신차 가격|현재 신차가)[^0-9]{0,20}([\d,]+)\s*만원", html)
    if m:
        return int(m.group(2).replace(",", ""))

    return None

In [5]:
import re
import time
import json
import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


SEARCH_URL = (
    "https://car.encar.com/list/car?page=1"
    "&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._."
    "%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._."
    "%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C"
    "%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D"
)


# -----------------------------
# 공통 유틸
# -----------------------------
def normalize_text(text):
    if text is None:
        return None
    return re.sub(r"\s+", " ", str(text)).strip()


def extract_car_id(url):
    m = re.search(r"/cars/detail/(\d+)", url)
    return m.group(1) if m else None


def to_int_from_text(text):
    if not text:
        return None
    m = re.search(r"([\d,]+)", str(text))
    return int(m.group(1).replace(",", "")) if m else None


def setup_driver(headless=False):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--lang=ko-KR")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=options)
    driver.implicitly_wait(3)
    return driver


def wait_body(driver, sec=10):
    WebDriverWait(driver, sec).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )


def close_popups(driver):
    popup_xpaths = [
        "//button[contains(., '닫기')]",
        "//button[contains(., '나중에')]",
        "//button[contains(., '오늘 보지 않기')]",
        "//button[contains(., '취소')]",
        "//button[contains(., '확인')]",
    ]
    for xp in popup_xpaths:
        try:
            buttons = driver.find_elements(By.XPATH, xp)
            for btn in buttons:
                if btn.is_displayed():
                    driver.execute_script("arguments[0].click();", btn)
                    time.sleep(0.3)
        except Exception:
            pass


def click_if_exists(driver, text):
    try:
        buttons = driver.find_elements(By.XPATH, f"//button[contains(., '{text}')]")
        for btn in buttons:
            if btn.is_displayed():
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(0.8)
                return True
    except Exception:
        pass
    return False


# -----------------------------
# 목록 링크 수집
# -----------------------------
def scroll_until_stable(driver, pause=2.0, max_rounds=20):
    last_height = driver.execute_script("return document.body.scrollHeight")
    stable_count = 0

    for i in range(max_rounds):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)

        new_height = driver.execute_script("return document.body.scrollHeight")
        print(f"[SCROLL] {i+1}: {last_height} -> {new_height}")

        if new_height == last_height:
            stable_count += 1
        else:
            stable_count = 0

        if stable_count >= 2:
            print("[INFO] 스크롤 종료")
            break

        last_height = new_height


def collect_detail_links(driver):
    anchors = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/cars/detail/"]')
    urls = []

    for a in anchors:
        try:
            href = a.get_attribute("href")
            if href and "/cars/detail/" in href:
                href = href.split("&advClickPosition=")[0]
                urls.append(href)
        except Exception:
            pass

    return list(dict.fromkeys(urls))


def collect_yf_links(driver):
    print("[INFO] YF 쏘나타 검색 페이지 접속")
    driver.get(SEARCH_URL)
    wait_body(driver)
    time.sleep(2)
    close_popups(driver)

    scroll_until_stable(driver, pause=2.0, max_rounds=20)
    links = collect_detail_links(driver)

    print(f"[INFO] 상세 링크 수집 완료: {len(links)}건")
    return links


# -----------------------------
# 상세페이지 파싱
# -----------------------------
def parse_meta_description(soup):
    result = {}
    meta_desc = soup.find("meta", attrs={"name": "description"})
    if not meta_desc:
        return result

    content = meta_desc.get("content", "")
    patterns = {
        "연식_메타": r"연식:([^,]+)",
        "주행거리_메타": r"주행거리:([^,]+)",
        "연료_메타": r"연료:([^,]+)",
        "색상_메타": r"색상:([^,]+)",
        "지역_메타": r"지역:([^,]+?) 중고차",
    }

    for key, pat in patterns.items():
        m = re.search(pat, content)
        if m:
            result[key] = m.group(1).strip()

    return result


def convert_year(year_text):
    if not year_text:
        return None
    m = re.search(r"(\d{2})/(\d{2})식", year_text)
    if not m:
        return None
    return 2000 + int(m.group(1))


def convert_mileage(mileage_text):
    if not mileage_text:
        return None
    return to_int_from_text(mileage_text)


def parse_title(soup):
    h3 = soup.find("h3")
    if not h3:
        return None, None, None

    spans = [normalize_text(x.get_text(" ", strip=True)) for x in h3.find_all("span")]
    spans = [x for x in spans if x]

    if len(spans) >= 2:
        model = spans[0]
        trim = spans[1]
        full_name = f"{model} {trim}".strip()
        return full_name, model, trim

    full_name = normalize_text(h3.get_text(" ", strip=True))
    return full_name, None, None


def parse_basic_info(soup):
    result = {
        "연식_원문": None,
        "주행거리_원문": None,
        "연료": None,
        "차량번호": None,
        "등록번호": None,
        "조회수": None,
        "찜수": None,
        "해시태그": None,
        "실촬영여부": None,
    }

    dl = soup.select_one("dl.ar1Ivd7EgX")
    if dl:
        dt_list = dl.find_all("dt")
        dd_list = dl.find_all("dd")

        for dt, dd in zip(dt_list, dd_list):
            k = normalize_text(dt.get_text(" ", strip=True))
            v = normalize_text(dd.get_text(" ", strip=True))

            if "연식" in k:
                result["연식_원문"] = v
            elif "주행거리" in k:
                result["주행거리_원문"] = v
            elif "연료" in k:
                result["연료"] = v
            elif "차량번호" in k:
                result["차량번호"] = v

    info_lis = soup.select("ul.rVigc5A_1H li")
    for li in info_lis:
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt.startswith("등록번호"):
            result["등록번호"] = txt.replace("등록번호", "").strip()
        elif txt.startswith("조회수"):
            result["조회수"] = to_int_from_text(txt)
        elif txt.startswith("찜"):
            result["찜수"] = to_int_from_text(txt)

    hashtags = [normalize_text(li.get_text(" ", strip=True)) for li in soup.select("ul.kYC8KS_YZ1 li")]
    hashtags = [x for x in hashtags if x]
    result["해시태그"] = ",".join(hashtags) if hashtags else None

    shot = soup.select_one("p.s8FzQTBVpE")
    if shot:
        result["실촬영여부"] = normalize_text(shot.get_text(" ", strip=True))

    return result


def parse_option_items_from_main(soup):
    option_dict = {}
    option_items = soup.select("ul.dz3qFYruNO li")

    for li in option_items:
        txt = normalize_text(li.get_text(" ", strip=True))
        if not txt:
            continue

        blind = li.select_one("span.blind")
        status = normalize_text(blind.get_text(" ", strip=True)) if blind else None

        name = txt
        if status:
            name = normalize_text(txt.replace(status, ""))

        if name:
            option_dict[name] = 1 if status == "있음" else 0 if status == "없음" else None

    return option_dict


def parse_vehicle_status(soup):
    result = {
        "차량키개수": None,
        "틴팅_앞유리": None,
        "앞타이어트레드_mm": None,
        "뒤타이어트레드_mm": None,
        "차량상태_raw": None,
    }

    items = []
    for li in soup.select("ul.IdK1KVYuyr li"):
        ps = li.find_all("p")
        vals = [normalize_text(p.get_text(" ", strip=True)) for p in ps]
        vals = [v for v in vals if v]
        joined = " | ".join(vals)

        if joined:
            items.append(joined)

        if "차량 키 개수" in joined:
            m = re.search(r"(\d+)개", joined)
            if m:
                result["차량키개수"] = int(m.group(1))

        elif "틴팅(앞 유리)" in joined:
            result["틴팅_앞유리"] = 1 if "있음" in joined else 0 if "없음" in joined else None

        elif "타이어트레드 잔량 (앞)" in joined:
            m = re.search(r"(\d+)mm", joined)
            if m:
                result["앞타이어트레드_mm"] = int(m.group(1))

        elif "타이어트레드 잔량 (뒤)" in joined:
            m = re.search(r"(\d+)mm", joined)
            if m:
                result["뒤타이어트레드_mm"] = int(m.group(1))

    result["차량상태_raw"] = " || ".join(items) if items else None
    return result


def parse_performance_record(soup):
    result = {
        "교환": None,
        "판금": None,
        "부식": None,
        "제시번호": None,
        "성능기록부_raw": None,
    }

    p = soup.select_one("div.Hs3feBPsTj p.WjUuNLum7b")
    if p:
        txt = normalize_text(p.get_text(" ", strip=True))
        m = re.search(r"제시번호\s*:\s*(.+)", txt)
        if m:
            result["제시번호"] = m.group(1).strip()

    items = []
    for li in soup.select("ul.wB0X7nC0cq li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt:
            items.append(txt)

        if "교환" in txt:
            if "없음" in txt:
                result["교환"] = 0
            else:
                m = re.search(r"교환\s*(\d+)", txt)
                if m:
                    result["교환"] = int(m.group(1))

        elif "판금" in txt:
            if "없음" in txt:
                result["판금"] = 0
            else:
                m = re.search(r"판금\s*(\d+)", txt)
                if m:
                    result["판금"] = int(m.group(1))

        elif "부식" in txt:
            result["부식"] = 0 if "없음" in txt else 1

    result["성능기록부_raw"] = " || ".join(items) if items else None
    return result


def parse_history(soup):
    result = {
        "내차피해금액": None,
        "내차피해횟수": None,
        "타차가해금액": None,
        "타차가해횟수": None,
        "특이사항": None,
        "차량이력_raw": None,
    }

    items = []
    for li in soup.select("ul.OO25I4KzD2 li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt:
            items.append(txt)

        if "내차 피해" in txt:
            if "없음" in txt:
                result["내차피해금액"] = 0
                result["내차피해횟수"] = 0
            else:
                m = re.search(r"총\s*([\d,]+)원\s*\((\d+)회\)", txt)
                if m:
                    result["내차피해금액"] = int(m.group(1).replace(",", ""))
                    result["내차피해횟수"] = int(m.group(2))

        elif "타차 가해" in txt:
            if "없음" in txt:
                result["타차가해금액"] = 0
                result["타차가해횟수"] = 0
            else:
                m = re.search(r"총\s*([\d,]+)원\s*\((\d+)회\)", txt)
                if m:
                    result["타차가해금액"] = int(m.group(1).replace(",", ""))
                    result["타차가해횟수"] = int(m.group(2))

        elif "특이 사항" in txt:
            result["특이사항"] = txt.replace("특이 사항", "").strip()

    result["차량이력_raw"] = " || ".join(items) if items else None
    return result


def parse_seller_info(soup):
    result = {
        "판매자상호": None,
        "판매자명": None,
        "판매중대수": None,
        "판매완료대수": None,
        "판매자지역": None,
        "종사원증번호": None,
    }

    seller_wrap = soup.select_one("div.fSPJGX0WLk")
    if not seller_wrap:
        return result

    txt = normalize_text(seller_wrap.get_text(" ", strip=True))

    name_span = seller_wrap.select_one("span.St9bdeN3m2")
    seller_name = seller_wrap.select_one("strong.k8v1ohDI6G")

    if name_span:
        result["판매자상호"] = normalize_text(name_span.get_text(" ", strip=True))
    if seller_name:
        result["판매자명"] = normalize_text(seller_name.get_text(" ", strip=True))

    m1 = re.search(r"판매중\s*([\d,]+)", txt)
    if m1:
        result["판매중대수"] = int(m1.group(1).replace(",", ""))

    m2 = re.search(r"판매완료\s*([\d,]+)", txt)
    if m2:
        result["판매완료대수"] = int(m2.group(1).replace(",", ""))

    m3 = re.search(r"(서울|경기|인천|부산|대구|대전|광주|울산|세종|강원|충북|충남|전북|전남|경북|경남|제주)\s+[가-힣]+시?", txt)
    if m3:
        result["판매자지역"] = m3.group(0)

    m4 = re.search(r"종사원증번호\s*([0-9\-]+)", txt)
    if m4:
        result["종사원증번호"] = m4.group(1)

    return result


# -----------------------------
# 신차대비 페이지
# -----------------------------
def parse_newcar_page(driver, car_id):
    url = f"https://fem.encar.com/cars/newcar/{car_id}"
    driver.get(url)
    wait_body(driver)
    time.sleep(1.5)
    close_popups(driver)

    body_text = normalize_text(driver.find_element(By.TAG_NAME, "body").text)
    html = driver.page_source

    patterns = [
        r"신차가\s*[:：]?\s*([\d,]+)\s*만원",
        r"출고가\s*[:：]?\s*([\d,]+)\s*만원",
        r"신차 가격\s*[:：]?\s*([\d,]+)\s*만원",
        r"현재 신차가\s*[:：]?\s*([\d,]+)\s*만원",
    ]

    newcar_price = None
    for pat in patterns:
        m = re.search(pat, body_text)
        if m:
            newcar_price = int(m.group(1).replace(",", ""))
            break

    if newcar_price is None:
        for pat in patterns:
            m = re.search(pat, html)
            if m:
                newcar_price = int(m.group(1).replace(",", ""))
                break

    return {
        "신차기준URL": url,
        "신차기준가격_만원": newcar_price,
        "신차기준_raw_text": body_text[:5000],
    }


# -----------------------------
# 옵션 전용 페이지
# -----------------------------
def parse_option_page(driver, car_id):
    url = f"https://fem.encar.com/cars/option/{car_id}"
    driver.get(url)
    wait_body(driver)
    time.sleep(1.5)
    close_popups(driver)

    body_text = normalize_text(driver.find_element(By.TAG_NAME, "body").text)
    soup = BeautifulSoup(driver.page_source, "lxml")

    # 페이지 구조가 main detail과 다를 수 있으니 유연하게
    option_dict = {}

    # 1차: li 기반
    for li in soup.select("li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if not txt:
            continue

        # "옵션명 있음/없음" 형태
        m = re.match(r"(.+?)\s+(있음|없음)$", txt)
        if m:
            name = normalize_text(m.group(1))
            status = m.group(2)
            option_dict[name] = 1 if status == "있음" else 0

    # 2차: body text fallback
    if not option_dict:
        lines = [normalize_text(x) for x in body_text.split("  ")]
        lines = [x for x in lines if x]
        for txt in lines:
            m = re.match(r"(.+?)\s+(있음|없음)$", txt)
            if m:
                name = normalize_text(m.group(1))
                status = m.group(2)
                option_dict[name] = 1 if status == "있음" else 0

    return {
        "옵션전용URL": url,
        "옵션전용_raw_text": body_text[:5000],
        "옵션전용_json": json.dumps(option_dict, ensure_ascii=False),
    }


# -----------------------------
# 통합 파서
# -----------------------------
def parse_detail_page(driver, url):
    driver.get(url)
    wait_body(driver)
    time.sleep(2)
    close_popups(driver)

    click_if_exists(driver, "판매자 정보")
    click_if_exists(driver, "차량관리상태 모두보기")
    click_if_exists(driver, "성능기록부 자세히보기")
    click_if_exists(driver, "차량이력 자세히 보기")

    page_source = driver.page_source
    soup = BeautifulSoup(page_source, "lxml")

    meta = parse_meta_description(soup)
    full_name, model, trim = parse_title(soup)
    basic = parse_basic_info(soup)
    options_main = parse_option_items_from_main(soup)
    status = parse_vehicle_status(soup)
    perf = parse_performance_record(soup)
    history = parse_history(soup)
    seller = parse_seller_info(soup)

    row = {
        "매물ID": extract_car_id(url),
        "차량명": full_name,
        "모델": model,
        "세부트림": trim,
        "연식_원문": basic["연식_원문"] or meta.get("연식_메타"),
        "연식": convert_year(basic["연식_원문"] or meta.get("연식_메타")),
        "주행거리_원문": basic["주행거리_원문"] or meta.get("주행거리_메타"),
        "주행거리_km": convert_mileage(basic["주행거리_원문"] or meta.get("주행거리_메타")),
        "연료": basic["연료"] or meta.get("연료_메타"),
        "색상": meta.get("색상_메타"),
        "지역": meta.get("지역_메타"),
        "차량번호": basic["차량번호"],
        "등록번호": basic["등록번호"],
        "조회수": basic["조회수"],
        "찜수": basic["찜수"],
        "해시태그": basic["해시태그"],
        "실촬영여부": basic["실촬영여부"],
        "상세링크": url,
    }

    row.update(status)
    row.update(perf)
    row.update(history)
    row.update(seller)

    for k, v in options_main.items():
        row[f"옵션메인_{k}"] = v

    row["옵션메인_json"] = json.dumps(options_main, ensure_ascii=False)

    # 신차대비 페이지
    try:
        newcar_info = parse_newcar_page(driver, row["매물ID"])
        row.update(newcar_info)
    except Exception as e:
        row["신차기준URL"] = f"https://fem.encar.com/cars/newcar/{row['매물ID']}"
        row["신차기준가격_만원"] = None
        row["신차기준_raw_text"] = f"ERROR: {e}"

    # 옵션 전용 페이지
    try:
        option_info = parse_option_page(driver, row["매물ID"])
        row.update(option_info)
    except Exception as e:
        row["옵션전용URL"] = f"https://fem.encar.com/cars/option/{row['매물ID']}"
        row["옵션전용_raw_text"] = f"ERROR: {e}"
        row["옵션전용_json"] = "{}"

    return row


# -----------------------------
# 실행
# -----------------------------
def main():
    driver = setup_driver(headless=False)

    try:
        links = collect_yf_links(driver)
        target_links = links[:5]

        print(f"[INFO] 실제 수집 대상: {len(target_links)}건")

        rows = []
        for i, url in enumerate(target_links, start=1):
            try:
                print(f"[INFO] ({i}/{len(target_links)}) 수집 중: {extract_car_id(url)}")
                row = parse_detail_page(driver, url)
                rows.append(row)
            except Exception as e:
                print(f"[ERROR] {url} -> {e}")

        df = pd.DataFrame(rows)

        with open("encar_yf_top5_integrated.json", "w", encoding="utf-8") as f:
            json.dump(rows, f, ensure_ascii=False, indent=2)

        df.to_csv("encar_yf_top5_integrated.csv", index=False, encoding="utf-8-sig")

        print("\n[INFO] 저장 완료")
        print("- encar_yf_top5_integrated.csv")
        print("- encar_yf_top5_integrated.json")
        print("\n[HEAD]")
        print(df.head())

    finally:
        driver.quit()


if __name__ == "__main__":
    main()

[INFO] YF 쏘나타 검색 페이지 접속
[SCROLL] 1: 44515 -> 44515
[SCROLL] 2: 44515 -> 44515
[INFO] 스크롤 종료
[INFO] 상세 링크 수집 완료: 210건
[INFO] 실제 수집 대상: 5건
[INFO] (1/5) 수집 중: 40955073
[INFO] (2/5) 수집 중: 41054615
[INFO] (3/5) 수집 중: 41225357
[INFO] (4/5) 수집 중: 40477859
[INFO] (5/5) 수집 중: 39486507

[INFO] 저장 완료
- encar_yf_top5_integrated.csv
- encar_yf_top5_integrated.json

[HEAD]
       매물ID                    차량명      모델            세부트림  \
0  40955073         YF 쏘나타 Y20 프라임  YF 쏘나타         Y20 프라임   
1  41054615        YF 쏘나타 CVVL 럭셔리  YF 쏘나타        CVVL 럭셔리   
2  41225357         YF 쏘나타 Y20 프라임  YF 쏘나타         Y20 프라임   
3  40477859  YF 쏘나타 LPI 프리미어(장애인용)  YF 쏘나타  LPI 프리미어(장애인용)   
4  39486507           YF 쏘나타 LPI 탑  YF 쏘나타           LPI 탑   

                연식_원문    연식    주행거리_원문  주행거리_km           연료   색상  ...  \
0         10/05식 연형정보  2010  136,187km   136187          가솔린  검정색  ...   
1         12/04식 연형정보  2012  105,294km   105294          가솔린   은색  ...   
2  10/09식 (11년형) 연형정보  2010  117,226km   11

In [6]:
import os
import re
import time
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


SEARCH_URL = (
    "https://car.encar.com/list/car?page=1"
    "&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._."
    "%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._."
    "%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C"
    "%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D"
)


def setup_driver(headless=False):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--lang=ko-KR")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=options)
    driver.implicitly_wait(3)
    return driver


def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip() if text else ""


def wait_body(driver, sec=10):
    WebDriverWait(driver, sec).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )


def extract_car_id(url):
    m = re.search(r"/cars/detail/(\d+)", url)
    return m.group(1) if m else None


def close_popups(driver):
    """
    로그인/광고/안내 팝업을 최대한 닫는다.
    실패해도 그냥 넘어간다.
    """
    popup_xpaths = [
        "//button[contains(., '닫기')]",
        "//button[contains(., '나중에')]",
        "//button[contains(., '오늘 보지 않기')]",
        "//button[contains(., '취소')]",
        "//button[contains(., '확인')]",
        "//button[contains(., '다음에')]",
        "//a[contains(., '닫기')]",
        "//span[contains(., '닫기')]/ancestor::button",
    ]

    for xp in popup_xpaths:
        try:
            buttons = driver.find_elements(By.XPATH, xp)
            for btn in buttons:
                try:
                    if btn.is_displayed():
                        driver.execute_script("arguments[0].click();", btn)
                        time.sleep(0.3)
                except Exception:
                    pass
        except Exception:
            pass

    # 오버레이 제거 시도
    try:
        driver.execute_script("""
            const selectors = [
                '[role="dialog"]',
                '.modal',
                '.popup',
                '.layer',
                '.dimmed',
                '.overlay'
            ];
            selectors.forEach(sel => {
                document.querySelectorAll(sel).forEach(el => {
                    el.style.display = 'none';
                    el.remove?.();
                });
            });
        """)
    except Exception:
        pass


def scroll_until_stable(driver, pause=2.0, max_rounds=20):
    last_height = driver.execute_script("return document.body.scrollHeight")
    stable_count = 0

    for i in range(max_rounds):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)

        new_height = driver.execute_script("return document.body.scrollHeight")

        if new_height == last_height:
            stable_count += 1
        else:
            stable_count = 0

        if stable_count >= 2:
            break

        last_height = new_height


def collect_detail_links(driver):
    anchors = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/cars/detail/"]')
    urls = []

    for a in anchors:
        try:
            href = a.get_attribute("href")
            if href and "/cars/detail/" in href:
                href = href.split("&advClickPosition=")[0]
                urls.append(href)
        except Exception:
            pass

    return list(dict.fromkeys(urls))


def collect_yf_links(driver, top_n=5):
    driver.get(SEARCH_URL)
    wait_body(driver)
    time.sleep(2)
    close_popups(driver)
    scroll_until_stable(driver, pause=2.0, max_rounds=20)

    links = collect_detail_links(driver)
    return links[:top_n]


def save_page_source(driver, url, html_path, txt_path=None, sleep_sec=2.0):
    driver.get(url)
    wait_body(driver)
    time.sleep(sleep_sec)
    close_popups(driver)
    time.sleep(0.5)

    html = driver.page_source
    html_path.write_text(html, encoding="utf-8")

    if txt_path is not None:
        body_text = normalize_text(driver.find_element(By.TAG_NAME, "body").text)
        txt_path.write_text(body_text, encoding="utf-8")


def collect_all_html_for_car(driver, detail_url, base_dir):
    car_id = extract_car_id(detail_url)
    if not car_id:
        print(f"[WARN] 매물ID 추출 실패: {detail_url}")
        return

    car_dir = Path(base_dir) / car_id
    car_dir.mkdir(parents=True, exist_ok=True)

    page_map = {
        "detail": detail_url,
        "newcar": f"https://fem.encar.com/cars/newcar/{car_id}",
        "option": f"https://fem.encar.com/cars/option/{car_id}",
    }

    for page_name, url in page_map.items():
        print(f"[INFO] {car_id} - {page_name} 수집 중")
        try:
            save_page_source(
                driver,
                url,
                html_path=car_dir / f"{page_name}.html",
                txt_path=car_dir / f"{page_name}.txt",
                sleep_sec=2.0
            )
        except Exception as e:
            print(f"[ERROR] {car_id} - {page_name}: {e}")


def main():
    base_dir = "encar_html"
    Path(base_dir).mkdir(parents=True, exist_ok=True)

    driver = setup_driver(headless=False)

    try:
        links = collect_yf_links(driver, top_n=5)
        print(f"[INFO] 대상 링크 수: {len(links)}")

        for idx, detail_url in enumerate(links, start=1):
            car_id = extract_car_id(detail_url)
            print(f"\n[INFO] ({idx}/{len(links)}) 매물ID={car_id}")
            collect_all_html_for_car(driver, detail_url, base_dir)

        print("\n[INFO] 완료")
        print(f"[INFO] 저장 폴더: {Path(base_dir).resolve()}")

    finally:
        driver.quit()


if __name__ == "__main__":
    main()

[INFO] 대상 링크 수: 5

[INFO] (1/5) 매물ID=41054615
[INFO] 41054615 - detail 수집 중
[INFO] 41054615 - newcar 수집 중
[INFO] 41054615 - option 수집 중

[INFO] (2/5) 매물ID=40903390
[INFO] 40903390 - detail 수집 중
[INFO] 40903390 - newcar 수집 중
[INFO] 40903390 - option 수집 중

[INFO] (3/5) 매물ID=39486507
[INFO] 39486507 - detail 수집 중
[INFO] 39486507 - newcar 수집 중
[INFO] 39486507 - option 수집 중

[INFO] (4/5) 매물ID=40955073
[INFO] 40955073 - detail 수집 중
[INFO] 40955073 - newcar 수집 중
[INFO] 40955073 - option 수집 중

[INFO] (5/5) 매물ID=41509017
[INFO] 41509017 - detail 수집 중
[INFO] 41509017 - newcar 수집 중
[INFO] 41509017 - option 수집 중

[INFO] 완료
[INFO] 저장 폴더: C:\Users\Admin\hipython\버뮤다_프로젝트_2\encar_html


In [8]:
import re
import json
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup


BASE_DIR = Path("encar_html")   # 예: encar_html/41509017/detail.html
OUT_CSV = "encar_parsed_from_saved_html.csv"
OUT_JSON = "encar_parsed_from_saved_html.json"


# -----------------------------
# 유틸
# -----------------------------
def read_text_safe(path: Path) -> str:
    if path.exists():
        return path.read_text(encoding="utf-8", errors="ignore")
    return ""


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip() if text else ""


def find_first(patterns, text, cast=None, flags=re.S):
    """
    patterns: 정규식 패턴 리스트
    - 캡처 그룹이 있으면 group(1)
    - 없으면 group(0)
    """
    for pat in patterns:
        m = re.search(pat, text, flags)
        if m:
            val = m.group(1) if m.lastindex else m.group(0)
            if cast:
                try:
                    return cast(val)
                except Exception:
                    return val
            return val
    return None


def to_int(val):
    if val is None:
        return None
    return int(str(val).replace(",", "").strip())


def parse_int_from_text(text):
    if not text:
        return None
    m = re.search(r"([\d,]+)", text)
    return int(m.group(1).replace(",", "")) if m else None


def parse_json_array_str(text):
    """
    예: ["001","003"] 문자열을 실제 list로 변환
    """
    if not text:
        return []
    try:
        return json.loads(text)
    except Exception:
        return []


# -----------------------------
# detail.html 파싱
# -----------------------------
def parse_meta_description(soup):
    result = {}
    meta = soup.find("meta", attrs={"name": "description"})
    if not meta:
        return result

    content = meta.get("content", "")

    result["연식_메타"] = find_first([r"연식:([^,]+)"], content)
    result["주행거리_메타"] = find_first([r"주행거리:([^,]+)"], content)
    result["연료_메타"] = find_first([r"연료:([^,]+)"], content)
    result["색상_메타"] = find_first([r"색상:([^,]+)"], content)
    result["지역_메타"] = find_first([r"지역:([^,]+?) 중고차"], content)

    return result


def parse_dom_basic(soup):
    result = {
        "차량명_dom": None,
        "모델_dom": None,
        "세부트림_dom": None,
        "연식_원문_dom": None,
        "주행거리_원문_dom": None,
        "연료_dom": None,
        "차량번호_dom": None,
        "등록번호_dom": None,
        "조회수_dom": None,
        "찜수_dom": None,
        "해시태그_dom": None,
    }

    h3 = soup.find("h3")
    if h3:
        spans = [normalize_text(x.get_text(" ", strip=True)) for x in h3.find_all("span")]
        spans = [x for x in spans if x]
        if len(spans) >= 2:
            result["모델_dom"] = spans[0]
            result["세부트림_dom"] = spans[1]
            result["차량명_dom"] = f"{spans[0]} {spans[1]}".strip()
        else:
            result["차량명_dom"] = normalize_text(h3.get_text(" ", strip=True))

    dl = soup.select_one("dl.ar1Ivd7EgX")
    if dl:
        dts = dl.find_all("dt")
        dds = dl.find_all("dd")
        for dt, dd in zip(dts, dds):
            k = normalize_text(dt.get_text(" ", strip=True))
            v = normalize_text(dd.get_text(" ", strip=True))

            if "연식" in k:
                result["연식_원문_dom"] = v
            elif "주행거리" in k:
                result["주행거리_원문_dom"] = v
            elif "연료" in k:
                result["연료_dom"] = v
            elif "차량번호" in k:
                result["차량번호_dom"] = v

    for li in soup.select("ul.rVigc5A_1H li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt.startswith("등록번호"):
            result["등록번호_dom"] = txt.replace("등록번호", "").strip()
        elif txt.startswith("조회수"):
            result["조회수_dom"] = parse_int_from_text(txt)
        elif txt.startswith("찜"):
            result["찜수_dom"] = parse_int_from_text(txt)

    tags = [normalize_text(li.get_text(" ", strip=True)) for li in soup.select("ul.kYC8KS_YZ1 li")]
    tags = [x for x in tags if x]
    result["해시태그_dom"] = ",".join(tags) if tags else None

    return result


def parse_detail_embedded(html):
    """
    detail.html 내부 JS/JSON 비슷한 덩어리에서 값 추출
    """
    result = {}

    key_patterns = {
        "manufacturerName": [r'"manufacturerName":"([^"]+)"'],
        "modelName": [r'"modelName":"([^"]+)"'],
        "gradeName": [r'"gradeName":"([^"]+)"'],
        "gradeDetailName": [r'"gradeDetailName":"([^"]+)"'],
        "vehicleNo": [r'"vehicleNo":"([^"]+)"'],
        "vin": [r'"vin":"([^"]+)"'],
        "requestUrl": [r'"requestUrl":"([^"]+)"'],
        "dealerName": [r'"dealer":\{"userId":"[^"]+","name":"([^"]+)"'],
        "firmName": [r'"firm":\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterName": [r'"diagnosisCenters":\[\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterPhone": [r'"telephoneNumber":"([^"]+)"'],
        "diagnosisCenterAddress": [r'"address":"([^"]+)"'],
        "pageAccessToken": [r'"pageAccessToken":"([^"]+)"'],
    }

    int_patterns = {
        "originPrice": [r'"originPrice":(\d+)'],
        "price": [r'"price":(\d+)'],
        "viewCount": [r'"viewCount":(\d+)'],
        "subscribeCount": [r'"subscribeCount":(\d+)'],
        "vehicleId": [r'"vehicleId":(\d+)'],
        "queryCarId": [r'"queryCarId":(\d+)'],
        "seizingCount": [r'"seizingCount":(\d+)'],
        "pledgeCount": [r'"pledgeCount":(\d+)'],
        "encarDiagnosis": [r'"encarDiagnosis":(-?\d+)'],
        "encarMeetGo": [r'"encarMeetGo":(-?\d+)'],
    }

    for key, pats in key_patterns.items():
        result[key] = find_first(pats, html)

    for key, pats in int_patterns.items():
        result[key] = find_first(pats, html, cast=to_int)

    # 옵션 코드 배열
    standard_raw = find_first([r'"standard":(\[[^\]]*\])'], html)
    choice_raw = find_first([r'"choice":(\[[^\]]*\])'], html)
    tuning_raw = find_first([r'"tuning":(\[[^\]]*\])'], html)
    etc_raw = find_first([r'"etc":(\[[^\]]*\])'], html)

    result["option_standard_codes"] = parse_json_array_str(standard_raw)
    result["option_choice_codes"] = parse_json_array_str(choice_raw)
    result["option_tuning_codes"] = parse_json_array_str(tuning_raw)
    result["option_etc_codes"] = parse_json_array_str(etc_raw)

    return result


def convert_year(year_text):
    if not year_text:
        return None
    m = re.search(r"(\d{2})/(\d{2})", year_text)
    return 2000 + int(m.group(1)) if m else None


def convert_mileage(mileage_text):
    return parse_int_from_text(mileage_text)


def parse_detail_file(detail_path: Path):
    html = read_text_safe(detail_path)
    soup = BeautifulSoup(html, "lxml")

    meta = parse_meta_description(soup)
    dom = parse_dom_basic(soup)
    emb = parse_detail_embedded(html)

    model = emb.get("modelName") or dom.get("모델_dom")
    grade = emb.get("gradeName")
    grade_detail = emb.get("gradeDetailName") or dom.get("세부트림_dom")

    full_name = None
    if model and grade_detail:
        full_name = f"{model} {grade_detail}".strip()
    elif dom.get("차량명_dom"):
        full_name = dom["차량명_dom"]

    row = {
        "매물ID": emb.get("vehicleId") or detail_path.parent.name,
        "제조사": emb.get("manufacturerName"),
        "모델": model,
        "등급명": grade,
        "세부트림": grade_detail,
        "차량명": full_name,
        "현재가격_만원": emb.get("price"),
        "신차기준가_만원": emb.get("originPrice"),
        "조회수": emb.get("viewCount") or dom.get("조회수_dom"),
        "찜수": emb.get("subscribeCount") or dom.get("찜수_dom"),
        "차량번호": emb.get("vehicleNo") or dom.get("차량번호_dom"),
        "VIN": emb.get("vin"),
        "연식_원문": dom.get("연식_원문_dom") or meta.get("연식_메타"),
        "연식": convert_year(dom.get("연식_원문_dom") or meta.get("연식_메타")),
        "주행거리_원문": dom.get("주행거리_원문_dom") or meta.get("주행거리_메타"),
        "주행거리_km": convert_mileage(dom.get("주행거리_원문_dom") or meta.get("주행거리_메타")),
        "연료": dom.get("연료_dom") or meta.get("연료_메타"),
        "색상": meta.get("색상_메타"),
        "지역": meta.get("지역_메타"),
        "등록번호": dom.get("등록번호_dom"),
        "해시태그": dom.get("해시태그_dom"),
        "딜러명": emb.get("dealerName"),
        "상사명": emb.get("firmName"),
        "진단센터명": emb.get("diagnosisCenterName"),
        "진단센터전화": emb.get("diagnosisCenterPhone"),
        "진단센터주소": emb.get("diagnosisCenterAddress"),
        "압류건수": emb.get("seizingCount"),
        "저당건수": emb.get("pledgeCount"),
        "엔카진단여부값": emb.get("encarDiagnosis"),
        "엔카믿고값": emb.get("encarMeetGo"),
        "requestUrl": emb.get("requestUrl"),
        "pageAccessToken": emb.get("pageAccessToken"),
        "옵션_기본코드개수": len(emb.get("option_standard_codes", [])),
        "옵션_선택코드개수": len(emb.get("option_choice_codes", [])),
        "옵션_튜닝코드개수": len(emb.get("option_tuning_codes", [])),
        "옵션_기타코드개수": len(emb.get("option_etc_codes", [])),
        "옵션_기본코드": ",".join(emb.get("option_standard_codes", [])) if emb.get("option_standard_codes") else None,
        "옵션_선택코드": ",".join(emb.get("option_choice_codes", [])) if emb.get("option_choice_codes") else None,
        "옵션_튜닝코드": ",".join(emb.get("option_tuning_codes", [])) if emb.get("option_tuning_codes") else None,
        "옵션_기타코드": ",".join(emb.get("option_etc_codes", [])) if emb.get("option_etc_codes") else None,
    }

    return row


# -----------------------------
# newcar / option 보조 파싱
# -----------------------------
def parse_newcar_file(newcar_path: Path):
    html = read_text_safe(newcar_path)
    soup = BeautifulSoup(html, "lxml")
    text = normalize_text(soup.get_text(" ", strip=True))

    note = find_first(
        [r"(신차가는 판매자가 입력한 등급\+선택옵션 기준으로 산출되었습니다\.)"],
        text
    )

    # 혹시 텍스트에 직접 가격이 있으면 시도
    direct_price = find_first([
        r"신차가\s*[:：]?\s*([\d,]+)\s*만원",
        r"출고가\s*[:：]?\s*([\d,]+)\s*만원",
        r"신차 가격\s*[:：]?\s*([\d,]+)\s*만원",
        r"현재 신차가\s*[:：]?\s*([\d,]+)\s*만원",
    ], text, cast=to_int)

    return {
        "newcar_note": note,
        "newcar_direct_price_만원": direct_price,
        "newcar_raw_preview": text[:500] if text else None,
    }


def parse_option_file(option_path: Path):
    html = read_text_safe(option_path)
    soup = BeautifulSoup(html, "lxml")
    text = normalize_text(soup.get_text(" ", strip=True))

    option_count = len(re.findall(r"(있음|없음)", text))

    return {
        "option_raw_preview": text[:500] if text else None,
        "option_visible_status_token_count": option_count,
    }


# -----------------------------
# 전체 실행
# -----------------------------
def main():
    rows = []

    if not BASE_DIR.exists():
        print(f"[ERROR] 폴더가 없습니다: {BASE_DIR.resolve()}")
        return

    for car_dir in sorted([p for p in BASE_DIR.iterdir() if p.is_dir()]):
        detail_path = car_dir / "detail.html"
        newcar_path = car_dir / "newcar.html"
        option_path = car_dir / "option.html"

        if not detail_path.exists():
            continue

        row = parse_detail_file(detail_path)

        if newcar_path.exists():
            row.update(parse_newcar_file(newcar_path))
        if option_path.exists():
            row.update(parse_option_file(option_path))

        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

    with open(OUT_JSON, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

    print(f"저장 완료: {OUT_CSV}")
    print(f"저장 완료: {OUT_JSON}")
    print("\n[HEAD]")
    print(df.head())
    print("\n[COLUMNS]")
    print(df.columns.tolist())


if __name__ == "__main__":
    main()

저장 완료: encar_parsed_from_saved_html.csv
저장 완료: encar_parsed_from_saved_html.json

[HEAD]
       매물ID 제조사      모델             등급명            세부트림  \
0  39486507  현대  YF 쏘나타           LPI 탑           LPI 탑   
1  40903390  현대  YF 쏘나타  LPI 프리미어(장애인용)  LPI 프리미어(장애인용)   
2  40955073  현대  YF 쏘나타         Y20 프라임             고급형   
3  41054615  현대  YF 쏘나타        CVVL 럭셔리        CVVL 럭셔리   
4  41509017  현대  YF 쏘나타         Y20 프라임              블랙   

                     차량명  현재가격_만원  신차기준가_만원   조회수  찜수  ... 옵션_기타코드개수  \
0           YF 쏘나타 LPI 탑      250       NaN  2450  58  ...         1   
1  YF 쏘나타 LPI 프리미어(장애인용)      330       NaN   129   1  ...         0   
2             YF 쏘나타 고급형      430    2345.0  1458  20  ...         0   
3        YF 쏘나타 CVVL 럭셔리      430    2450.0  1057  26  ...         0   
4              YF 쏘나타 블랙      399    2410.0   910  16  ...         0   

                                             옵션_기본코드 옵션_선택코드  옵션_튜닝코드  \
0  001,003,005,006,007,008,010,014,015,017,021,02.

In [9]:
import re
import json
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup


BASE_DIR = Path("encar_html")
OUT_CSV = "encar_parsed_from_saved_html_v2.csv"
OUT_JSON = "encar_parsed_from_saved_html_v2.json"


def read_text_safe(path: Path) -> str:
    if path.exists():
        return path.read_text(encoding="utf-8", errors="ignore")
    return ""


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip() if text else ""


def find_first(patterns, text, cast=None, flags=re.S):
    for pat in patterns:
        m = re.search(pat, text, flags)
        if m:
            val = m.group(1) if m.lastindex else m.group(0)
            if cast:
                try:
                    return cast(val)
                except Exception:
                    return val
            return val
    return None


def to_int(val):
    if val is None:
        return None
    return int(str(val).replace(",", "").strip())


def parse_int_from_text(text):
    if not text:
        return None
    m = re.search(r"([\d,]+)", text)
    return int(m.group(1).replace(",", "")) if m else None


def parse_json_array_str(text):
    if not text:
        return []
    try:
        return json.loads(text)
    except Exception:
        return []


def parse_meta_description(soup):
    result = {}
    meta = soup.find("meta", attrs={"name": "description"})
    if not meta:
        return result

    content = meta.get("content", "")

    result["연식_메타"] = find_first([r"연식:([^,]+)"], content)
    result["주행거리_메타"] = find_first([r"주행거리:([^,]+)"], content)
    result["연료_메타"] = find_first([r"연료:([^,]+)"], content)
    result["색상_메타"] = find_first([r"색상:([^,]+)"], content)
    result["지역_메타"] = find_first([r"지역:([^,]+?) 중고차"], content)

    return result


def parse_dom_basic(soup):
    result = {
        "차량명_dom": None,
        "모델_dom": None,
        "세부트림_dom": None,
        "연식_원문_dom": None,
        "주행거리_원문_dom": None,
        "연료_dom": None,
        "차량번호_dom": None,
        "등록번호_dom": None,
        "조회수_dom": None,
        "찜수_dom": None,
        "해시태그_dom": None,
    }

    h3 = soup.find("h3")
    if h3:
        spans = [normalize_text(x.get_text(" ", strip=True)) for x in h3.find_all("span")]
        spans = [x for x in spans if x]
        if len(spans) >= 2:
            result["모델_dom"] = spans[0]
            result["세부트림_dom"] = spans[1]
            result["차량명_dom"] = " ".join(spans).strip()
        else:
            result["차량명_dom"] = normalize_text(h3.get_text(" ", strip=True))

    dl = soup.select_one("dl.ar1Ivd7EgX")
    if dl:
        dts = dl.find_all("dt")
        dds = dl.find_all("dd")
        for dt, dd in zip(dts, dds):
            k = normalize_text(dt.get_text(" ", strip=True))
            v = normalize_text(dd.get_text(" ", strip=True))
            if "연식" in k:
                result["연식_원문_dom"] = v
            elif "주행거리" in k:
                result["주행거리_원문_dom"] = v
            elif "연료" in k:
                result["연료_dom"] = v
            elif "차량번호" in k:
                result["차량번호_dom"] = v

    for li in soup.select("ul.rVigc5A_1H li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt.startswith("등록번호"):
            result["등록번호_dom"] = txt.replace("등록번호", "").strip()
        elif txt.startswith("조회수"):
            result["조회수_dom"] = parse_int_from_text(txt)
        elif txt.startswith("찜"):
            result["찜수_dom"] = parse_int_from_text(txt)

    tags = [normalize_text(li.get_text(" ", strip=True)) for li in soup.select("ul.kYC8KS_YZ1 li")]
    tags = [x for x in tags if x]
    result["해시태그_dom"] = ",".join(tags) if tags else None

    return result


def parse_detail_embedded(html):
    result = {}

    str_patterns = {
        "manufacturerName": [r'"manufacturerName":"([^"]+)"'],
        "modelName": [r'"modelName":"([^"]+)"'],
        "gradeName": [r'"gradeName":"([^"]+)"'],
        "gradeDetailName": [r'"gradeDetailName":"([^"]+)"'],
        "vehicleNo": [r'"vehicleNo":"([^"]+)"'],
        "vin": [r'"vin":"([^"]+)"'],
        "requestUrl": [r'"requestUrl":"([^"]+)"'],
        "dealerName": [r'"dealer":\{"userId":"[^"]+","name":"([^"]+)"'],
        "firmName": [r'"firm":\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterName": [r'"diagnosisCenters":\[\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterPhone": [r'"telephoneNumber":"([^"]+)"'],
        "diagnosisCenterAddress": [r'"address":"([^"]+)"'],
        "pageAccessToken": [r'"pageAccessToken":"([^"]+)"'],
        "yearMonth": [r'"yearMonth":"([^"]+)"'],
        "formYear": [r'"formYear":"([^"]+)"'],
        "fuelName": [r'"fuelName":"([^"]+)"'],
        "colorName": [r'"colorName":"([^"]+)"'],
        "transmissionName": [r'"transmissionName":"([^"]+)"'],
        "bodyName": [r'"bodyName":"([^"]+)"'],
        "oneLineText": [r'"oneLineText":"([^"]+)"'],
    }

    int_patterns = {
        "originPrice": [r'"originPrice":(\d+)'],
        "price": [r'"price":(\d+)'],
        "viewCount": [r'"viewCount":(\d+)'],
        "subscribeCount": [r'"subscribeCount":(\d+)'],
        "vehicleId": [r'"vehicleId":(\d+)'],
        "queryCarId": [r'"queryCarId":(\d+)'],
        "seizingCount": [r'"seizingCount":(\d+)'],
        "pledgeCount": [r'"pledgeCount":(\d+)'],
        "encarDiagnosis": [r'"encarDiagnosis":(-?\d+)'],
        "encarMeetGo": [r'"encarMeetGo":(-?\d+)'],
        "mileage": [r'"mileage":(\d+)'],
        "displacement": [r'"displacement":(\d+)'],
        "seatCount": [r'"seatCount":(\d+)'],
    }

    for key, pats in str_patterns.items():
        result[key] = find_first(pats, html)

    for key, pats in int_patterns.items():
        result[key] = find_first(pats, html, cast=to_int)

    standard_raw = find_first([r'"standard":(\[[^\]]*\])'], html)
    choice_raw = find_first([r'"choice":(\[[^\]]*\])'], html)
    tuning_raw = find_first([r'"tuning":(\[[^\]]*\])'], html)
    etc_raw = find_first([r'"etc":(\[[^\]]*\]|"[^"]*"|\[[^\]]*\])'], html)

    result["option_standard_codes"] = parse_json_array_str(standard_raw)
    result["option_choice_codes"] = parse_json_array_str(choice_raw)
    result["option_tuning_codes"] = parse_json_array_str(tuning_raw)

    if etc_raw and etc_raw.startswith("["):
        result["option_etc_values"] = parse_json_array_str(etc_raw)
    elif etc_raw:
        result["option_etc_values"] = [etc_raw]
    else:
        result["option_etc_values"] = []

    return result


def build_full_trim(model, grade, detail):
    parts = []
    for p in [model, grade, detail]:
        p = normalize_text(p)
        if p and p not in parts:
            parts.append(p)
    return " ".join(parts) if parts else None


def convert_year_from_embedded(year_month, form_year, fallback_text):
    if form_year:
        try:
            return int(form_year)
        except Exception:
            pass
    if year_month and len(year_month) >= 4:
        try:
            return int(year_month[:4])
        except Exception:
            pass
    if fallback_text:
        m = re.search(r"(\d{2})/", fallback_text)
        if m:
            return 2000 + int(m.group(1))
    return None


def parse_detail_file(detail_path: Path):
    html = read_text_safe(detail_path)
    soup = BeautifulSoup(html, "lxml")

    meta = parse_meta_description(soup)
    dom = parse_dom_basic(soup)
    emb = parse_detail_embedded(html)

    model = emb.get("modelName") or dom.get("모델_dom")
    grade = emb.get("gradeName")
    detail = emb.get("gradeDetailName") or dom.get("세부트림_dom")

    row = {
        "매물ID": emb.get("vehicleId") or detail_path.parent.name,
        "제조사": emb.get("manufacturerName"),
        "모델": model,
        "등급명": grade,
        "세부트림": detail,
        "차량명": build_full_trim(model, grade, detail),
        "현재가격_만원": emb.get("price"),
        "신차기준가_만원": emb.get("originPrice"),
        "조회수": emb.get("viewCount") or dom.get("조회수_dom"),
        "찜수": emb.get("subscribeCount") or dom.get("찜수_dom"),
        "차량번호": emb.get("vehicleNo") or dom.get("차량번호_dom"),
        "VIN": emb.get("vin"),
        "연식_원문": dom.get("연식_원문_dom") or meta.get("연식_메타"),
        "연식": convert_year_from_embedded(emb.get("yearMonth"), emb.get("formYear"), dom.get("연식_원문_dom") or meta.get("연식_메타")),
        "주행거리_원문": dom.get("주행거리_원문_dom") or meta.get("주행거리_메타"),
        "주행거리_km": emb.get("mileage") or parse_int_from_text(dom.get("주행거리_원문_dom") or meta.get("주행거리_메타")),
        "연료": emb.get("fuelName") or dom.get("연료_dom") or meta.get("연료_메타"),
        "색상": emb.get("colorName") or meta.get("색상_메타"),
        "지역": meta.get("지역_메타"),
        "변속기": emb.get("transmissionName"),
        "배기량_cc": emb.get("displacement"),
        "차급": emb.get("bodyName"),
        "좌석수": emb.get("seatCount"),
        "등록번호": dom.get("등록번호_dom"),
        "해시태그": dom.get("해시태그_dom"),
        "딜러명": emb.get("dealerName"),
        "상사명": emb.get("firmName"),
        "진단센터명": emb.get("diagnosisCenterName"),
        "진단센터전화": emb.get("diagnosisCenterPhone"),
        "진단센터주소": emb.get("diagnosisCenterAddress"),
        "압류건수": emb.get("seizingCount"),
        "저당건수": emb.get("pledgeCount"),
        "엔카진단여부값": emb.get("encarDiagnosis"),
        "엔카믿고값": emb.get("encarMeetGo"),
        "광고한줄문구": emb.get("oneLineText"),
        "requestUrl": emb.get("requestUrl"),
        "pageAccessToken": emb.get("pageAccessToken"),
        "옵션_기본코드개수": len(emb.get("option_standard_codes", [])),
        "옵션_선택코드개수": len(emb.get("option_choice_codes", [])),
        "옵션_튜닝코드개수": len(emb.get("option_tuning_codes", [])),
        "옵션_기타개수": len(emb.get("option_etc_values", [])),
        "옵션_기본코드": ",".join(emb.get("option_standard_codes", [])) if emb.get("option_standard_codes") else None,
        "옵션_선택코드": ",".join(emb.get("option_choice_codes", [])) if emb.get("option_choice_codes") else None,
        "옵션_튜닝코드": ",".join(emb.get("option_tuning_codes", [])) if emb.get("option_tuning_codes") else None,
        "옵션_기타값": " | ".join(emb.get("option_etc_values", [])) if emb.get("option_etc_values") else None,
    }

    return row


def parse_newcar_file(newcar_path: Path):
    html = read_text_safe(newcar_path)
    soup = BeautifulSoup(html, "lxml")
    text = normalize_text(soup.get_text(" ", strip=True))

    note = find_first([r"(신차가는 판매자가 입력한 등급\+선택옵션 기준으로 산출되었습니다\.)"], text)

    direct_price = find_first([
        r"([\d,]+)만원\s*\(신차가\)",
        r"신차가\s*[:：]?\s*([\d,]+)\s*만원",
        r"출고가\s*[:：]?\s*([\d,]+)\s*만원",
        r"신차 가격\s*[:：]?\s*([\d,]+)\s*만원",
        r"현재 신차가\s*[:：]?\s*([\d,]+)\s*만원",
    ], text, cast=to_int)

    grade_base_price = find_first([
        r"등급 기준\s*([\d,]+)\s*만원",
        r"선택옵션 미포함 가격입니다\.\s*.*?([\d,]+)\s*만원"
    ], text, cast=to_int)

    option_price = find_first([
        r"선택옵션\s*\d+종\s*([\d,]+)\s*만원"
    ], text, cast=to_int)

    return {
        "newcar_note": note,
        "newcar_direct_price_만원": direct_price,
        "newcar_grade_base_price_만원": grade_base_price,
        "newcar_option_price_만원": option_price,
        "newcar_raw_preview": text[:700] if text else None,
    }


def parse_option_file(option_path: Path):
    html = read_text_safe(option_path)
    soup = BeautifulSoup(html, "lxml")
    text = normalize_text(soup.get_text(" ", strip=True))

    option_count = len(re.findall(r"(있음|없음)", text))

    return {
        "option_raw_preview": text[:500] if text else None,
        "option_visible_status_token_count": option_count,
    }


def main():
    rows = []

    for car_dir in sorted([p for p in BASE_DIR.iterdir() if p.is_dir()]):
        detail_path = car_dir / "detail.html"
        newcar_path = car_dir / "newcar.html"
        option_path = car_dir / "option.html"

        if not detail_path.exists():
            continue

        row = parse_detail_file(detail_path)

        if newcar_path.exists():
            row.update(parse_newcar_file(newcar_path))
        if option_path.exists():
            row.update(parse_option_file(option_path))

        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv("encar_parsed_from_saved_html_v3.csv", index=False, encoding="utf-8-sig")

    with open("encar_parsed_from_saved_html_v3.json", "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

    print("저장 완료: encar_parsed_from_saved_html_v3.csv")
    print("저장 완료: encar_parsed_from_saved_html_v3.json")
    print(df.head())
    print(df.columns.tolist())


if __name__ == "__main__":
    main()

저장 완료: encar_parsed_from_saved_html_v3.csv
저장 완료: encar_parsed_from_saved_html_v3.json
       매물ID 제조사      모델             등급명            세부트림  \
0  39486507  현대  YF 쏘나타           LPI 탑           LPI 탑   
1  40903390  현대  YF 쏘나타  LPI 프리미어(장애인용)  LPI 프리미어(장애인용)   
2  40955073  현대  YF 쏘나타         Y20 프라임             고급형   
3  41054615  현대  YF 쏘나타        CVVL 럭셔리        CVVL 럭셔리   
4  41509017  현대  YF 쏘나타         Y20 프라임              블랙   

                     차량명  현재가격_만원  신차기준가_만원   조회수  찜수  ... 옵션_선택코드 옵션_튜닝코드  \
0           YF 쏘나타 LPI 탑      250       NaN  2450  58  ...    None    None   
1  YF 쏘나타 LPI 프리미어(장애인용)      330       NaN   129   1  ...    None    None   
2     YF 쏘나타 Y20 프라임 고급형      430    2345.0  1458  20  ...    1077    None   
3        YF 쏘나타 CVVL 럭셔리      430    2450.0  1057  26  ...    None    None   
4      YF 쏘나타 Y20 프라임 블랙      399    2410.0   910  16  ...    1045    None   

            옵션_기타값                          newcar_note  \
0  · 버튼시동장치\n· 썬루프            

In [10]:
import re
import json
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup


BASE_DIR = Path("encar_html")
OUT_CSV = "encar_parsed_from_saved_html_v4.csv"
OUT_JSON = "encar_parsed_from_saved_html_v4.json"


def read_text_safe(path: Path) -> str:
    if path.exists():
        return path.read_text(encoding="utf-8", errors="ignore")
    return ""


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip() if text else ""


def find_first(patterns, text, cast=None, flags=re.S):
    for pat in patterns:
        m = re.search(pat, text, flags)
        if m:
            val = m.group(1) if m.lastindex else m.group(0)
            if cast:
                try:
                    return cast(val)
                except Exception:
                    return val
            return val
    return None


def to_int(val):
    if val is None:
        return None
    return int(str(val).replace(",", "").strip())


def parse_int_from_text(text):
    if not text:
        return None
    m = re.search(r"([\d,]+)", text)
    return int(m.group(1).replace(",", "")) if m else None


def parse_json_array_str(text):
    if not text:
        return []
    try:
        return json.loads(text)
    except Exception:
        return []


def parse_meta_description(soup):
    result = {}
    meta = soup.find("meta", attrs={"name": "description"})
    if not meta:
        return result

    content = meta.get("content", "")
    result["연식_메타"] = find_first([r"연식:([^,]+)"], content)
    result["주행거리_메타"] = find_first([r"주행거리:([^,]+)"], content)
    result["연료_메타"] = find_first([r"연료:([^,]+)"], content)
    result["색상_메타"] = find_first([r"색상:([^,]+)"], content)
    result["지역_메타"] = find_first([r"지역:([^,]+?) 중고차"], content)
    return result


def parse_dom_basic(soup):
    result = {
        "차량명_dom": None,
        "모델_dom": None,
        "세부트림_dom": None,
        "연식_원문_dom": None,
        "주행거리_원문_dom": None,
        "연료_dom": None,
        "차량번호_dom": None,
        "등록번호_dom": None,
        "조회수_dom": None,
        "찜수_dom": None,
        "해시태그_dom": None,
        "실촬영문구_dom": None,
    }

    h3 = soup.find("h3")
    if h3:
        spans = [normalize_text(x.get_text(" ", strip=True)) for x in h3.find_all("span")]
        spans = [x for x in spans if x]
        if len(spans) >= 2:
            result["모델_dom"] = spans[0]
            result["세부트림_dom"] = spans[1]
            result["차량명_dom"] = " ".join(spans).strip()
        else:
            result["차량명_dom"] = normalize_text(h3.get_text(" ", strip=True))

    dl = soup.select_one("dl.ar1Ivd7EgX")
    if dl:
        dts = dl.find_all("dt")
        dds = dl.find_all("dd")
        for dt, dd in zip(dts, dds):
            k = normalize_text(dt.get_text(" ", strip=True))
            v = normalize_text(dd.get_text(" ", strip=True))
            if "연식" in k:
                result["연식_원문_dom"] = v
            elif "주행거리" in k:
                result["주행거리_원문_dom"] = v
            elif "연료" in k:
                result["연료_dom"] = v
            elif "차량번호" in k:
                result["차량번호_dom"] = v

    for li in soup.select("ul.rVigc5A_1H li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt.startswith("등록번호"):
            result["등록번호_dom"] = txt.replace("등록번호", "").strip()
        elif txt.startswith("조회수"):
            result["조회수_dom"] = parse_int_from_text(txt)
        elif txt.startswith("찜"):
            result["찜수_dom"] = parse_int_from_text(txt)

    tags = [normalize_text(li.get_text(" ", strip=True)) for li in soup.select("ul.kYC8KS_YZ1 li")]
    tags = [x for x in tags if x]
    result["해시태그_dom"] = ",".join(tags) if tags else None

    shot = soup.select_one("p.s8FzQTBVpE")
    if shot:
        result["실촬영문구_dom"] = normalize_text(shot.get_text(" ", strip=True))

    return result


def parse_detail_embedded(html):
    result = {}

    str_patterns = {
        "manufacturerName": [r'"manufacturerName":"([^"]+)"'],
        "modelName": [r'"modelName":"([^"]+)"'],
        "gradeName": [r'"gradeName":"([^"]+)"'],
        "gradeDetailName": [r'"gradeDetailName":"([^"]+)"'],
        "vehicleNo": [r'"vehicleNo":"([^"]+)"'],
        "vin": [r'"vin":"([^"]+)"'],
        "requestUrl": [r'"requestUrl":"([^"]+)"'],
        "dealerName": [r'"dealer":\{"userId":"[^"]+","name":"([^"]+)"'],
        "firmName": [r'"firm":\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterName": [r'"diagnosisCenters":\[\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterPhone": [r'"telephoneNumber":"([^"]+)"'],
        "diagnosisCenterAddress": [r'"address":"([^"]+)"'],
        "pageAccessToken": [r'"pageAccessToken":"([^"]+)"'],
        "yearMonth": [r'"yearMonth":"([^"]+)"'],
        "formYear": [r'"formYear":"([^"]+)"'],
        "fuelName": [r'"fuelName":"([^"]+)"'],
        "colorName": [r'"colorName":"([^"]+)"'],
        "transmissionName": [r'"transmissionName":"([^"]+)"'],
        "bodyName": [r'"bodyName":"([^"]+)"'],
        "oneLineText": [r'"oneLineText":"([^"]+)"'],
    }

    int_patterns = {
        "originPrice": [r'"originPrice":(\d+)'],
        "price": [r'"price":(\d+)'],
        "viewCount": [r'"viewCount":(\d+)'],
        "subscribeCount": [r'"subscribeCount":(\d+)'],
        "vehicleId": [r'"vehicleId":(\d+)'],
        "seizingCount": [r'"seizingCount":(\d+)'],
        "pledgeCount": [r'"pledgeCount":(\d+)'],
        "encarDiagnosis": [r'"encarDiagnosis":(-?\d+)'],
        "encarMeetGo": [r'"encarMeetGo":(-?\d+)'],
        "mileage": [r'"mileage":(\d+)'],
        "displacement": [r'"displacement":(\d+)'],
        "seatCount": [r'"seatCount":(\d+)'],
    }

    for key, pats in str_patterns.items():
        result[key] = find_first(pats, html)

    for key, pats in int_patterns.items():
        result[key] = find_first(pats, html, cast=to_int)

    standard_raw = find_first([r'"standard":(\[[^\]]*\])'], html)
    choice_raw = find_first([r'"choice":(\[[^\]]*\])'], html)
    tuning_raw = find_first([r'"tuning":(\[[^\]]*\])'], html)
    etc_raw = find_first([r'"etc":(\[[^\]]*\]|"[^"]*")'], html)

    result["option_standard_codes"] = parse_json_array_str(standard_raw)
    result["option_choice_codes"] = parse_json_array_str(choice_raw)
    result["option_tuning_codes"] = parse_json_array_str(tuning_raw)

    if etc_raw and etc_raw.startswith("["):
        result["option_etc_values"] = parse_json_array_str(etc_raw)
    elif etc_raw:
        result["option_etc_values"] = [etc_raw.strip('"')]
    else:
        result["option_etc_values"] = []

    return result


def parse_dom_major_options(soup):
    result = {}
    for li in soup.select("ul.dz3qFYruNO li"):
        text = normalize_text(li.get_text(" ", strip=True))
        blind = li.select_one("span.blind")
        status = normalize_text(blind.get_text(" ", strip=True)) if blind else None

        if status:
            name = normalize_text(text.replace(status, ""))
            result[f"주요옵션_{name}"] = 1 if status == "있음" else 0 if status == "없음" else None

    return result


def parse_dom_seller_info(soup):
    result = {
        "판매자상호_dom": None,
        "판매자명_dom": None,
        "판매자유형_dom": None,
        "판매중대수_dom": None,
        "판매완료대수_dom": None,
        "판매자지역_dom": None,
        "종사원증번호_dom": None,
    }

    btn = soup.select_one("button.uPEfVsKnZx")
    if btn:
        brand = btn.select_one("span.ESlvHTih77")
        name = btn.select_one("strong.k3tS4rXdrQ")
        seller_type = btn.select_one("span.xf9ufEfgKR")

        if brand:
            result["판매자상호_dom"] = normalize_text(brand.get_text(" ", strip=True))
        if name:
            result["판매자명_dom"] = normalize_text(name.get_text(" ", strip=True))
        if seller_type:
            result["판매자유형_dom"] = normalize_text(seller_type.get_text(" ", strip=True))

    lis = soup.select("ul.VtNR8dNHOS li")
    for li in lis:
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt.startswith("판매중"):
            result["판매중대수_dom"] = parse_int_from_text(txt)
        elif txt.startswith("판매완료"):
            result["판매완료대수_dom"] = parse_int_from_text(txt)
        elif "종사원증번호" in txt:
            m = re.search(r"종사원증번호\s*([A-Z0-9\-]+)", txt)
            if m:
                result["종사원증번호_dom"] = m.group(1)
        elif any(region in txt for region in ["서울", "경기", "인천", "부산", "대구", "대전", "광주", "울산", "세종", "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"]):
            result["판매자지역_dom"] = txt

    return result


def parse_dom_extra_flags(soup):
    result = {
        "보험이력공개여부_dom": None,
        "성능점검유형_dom": None,
        "신차가격표링크_dom": None,
        "성능점검설명_dom": None,
    }

    for btn in soup.select("button.fSZoGuAX4M"):
        txt = normalize_text(btn.get_text(" ", strip=True))
        if "보험이력" in txt:
            result["보험이력공개여부_dom"] = "비공개" if "비공개" in txt else txt
        elif "성능점검내역" in txt:
            result["성능점검유형_dom"] = txt.replace("성능점검내역", "").strip()

    newprice_link = soup.select_one('div[data-impression="신차비교"] a[href*="newprice"]')
    if newprice_link:
        result["신차가격표링크_dom"] = newprice_link.get("href")

    perf_desc = soup.select_one("div.Hs3feBPsTj p.lGhsYmQaGE")
    if perf_desc:
        result["성능점검설명_dom"] = normalize_text(perf_desc.get_text(" ", strip=True))

    return result


def build_full_trim(model, grade, detail):
    parts = []
    for p in [model, grade, detail]:
        p = normalize_text(p)
        if p and p not in parts:
            parts.append(p)
    return " ".join(parts) if parts else None


def convert_year_from_embedded(year_month, form_year, fallback_text):
    if form_year:
        try:
            return int(form_year)
        except Exception:
            pass
    if year_month and len(year_month) >= 4:
        try:
            return int(year_month[:4])
        except Exception:
            pass
    if fallback_text:
        m = re.search(r"(\d{2})/", fallback_text)
        if m:
            return 2000 + int(m.group(1))
    return None


def parse_detail_file(detail_path: Path):
    html = read_text_safe(detail_path)
    soup = BeautifulSoup(html, "lxml")

    meta = parse_meta_description(soup)
    dom = parse_dom_basic(soup)
    emb = parse_detail_embedded(html)
    major_options = parse_dom_major_options(soup)
    seller_dom = parse_dom_seller_info(soup)
    flags_dom = parse_dom_extra_flags(soup)

    model = emb.get("modelName") or dom.get("모델_dom")
    grade = emb.get("gradeName")
    detail = emb.get("gradeDetailName") or dom.get("세부트림_dom")

    row = {
        "매물ID": emb.get("vehicleId") or detail_path.parent.name,
        "제조사": emb.get("manufacturerName"),
        "모델": model,
        "등급명": grade,
        "세부트림": None if normalize_text(grade) == normalize_text(detail) else detail,
        "차량명": build_full_trim(model, grade, None if normalize_text(grade) == normalize_text(detail) else detail),
        "현재가격_만원": emb.get("price"),
        "신차기준가_만원": emb.get("originPrice"),
        "조회수": emb.get("viewCount") or dom.get("조회수_dom"),
        "찜수": emb.get("subscribeCount") or dom.get("찜수_dom"),
        "차량번호": emb.get("vehicleNo") or dom.get("차량번호_dom"),
        "VIN": emb.get("vin"),
        "연식_원문": dom.get("연식_원문_dom") or meta.get("연식_메타"),
        "연식": convert_year_from_embedded(emb.get("yearMonth"), emb.get("formYear"), dom.get("연식_원문_dom") or meta.get("연식_메타")),
        "주행거리_원문": dom.get("주행거리_원문_dom") or meta.get("주행거리_메타"),
        "주행거리_km": emb.get("mileage") or parse_int_from_text(dom.get("주행거리_원문_dom") or meta.get("주행거리_메타")),
        "연료": emb.get("fuelName") or dom.get("연료_dom") or meta.get("연료_메타"),
        "색상": emb.get("colorName") or meta.get("색상_메타"),
        "지역": meta.get("지역_메타"),
        "변속기": emb.get("transmissionName"),
        "배기량_cc": emb.get("displacement"),
        "차급": emb.get("bodyName"),
        "좌석수": emb.get("seatCount"),
        "등록번호": dom.get("등록번호_dom"),
        "해시태그": dom.get("해시태그_dom"),
        "실촬영문구": dom.get("실촬영문구_dom"),
        "딜러명": emb.get("dealerName") or seller_dom.get("판매자명_dom"),
        "상사명": emb.get("firmName") or seller_dom.get("판매자상호_dom"),
        "판매자유형": seller_dom.get("판매자유형_dom"),
        "판매중대수": seller_dom.get("판매중대수_dom"),
        "판매완료대수": seller_dom.get("판매완료대수_dom"),
        "판매자지역": seller_dom.get("판매자지역_dom"),
        "종사원증번호": seller_dom.get("종사원증번호_dom"),
        "진단센터명": emb.get("diagnosisCenterName"),
        "진단센터전화": emb.get("diagnosisCenterPhone"),
        "진단센터주소": emb.get("diagnosisCenterAddress"),
        "압류건수": emb.get("seizingCount"),
        "저당건수": emb.get("pledgeCount"),
        "엔카진단여부값": emb.get("encarDiagnosis"),
        "엔카믿고값": emb.get("encarMeetGo"),
        "광고한줄문구": emb.get("oneLineText"),
        "보험이력공개여부": flags_dom.get("보험이력공개여부_dom"),
        "성능점검유형": flags_dom.get("성능점검유형_dom"),
        "성능점검설명": flags_dom.get("성능점검설명_dom"),
        "신차가격표링크": flags_dom.get("신차가격표링크_dom"),
        "requestUrl": emb.get("requestUrl"),
        "pageAccessToken": emb.get("pageAccessToken"),
        "옵션_기본코드개수": len(emb.get("option_standard_codes", [])),
        "옵션_선택코드개수": len(emb.get("option_choice_codes", [])),
        "옵션_튜닝코드개수": len(emb.get("option_tuning_codes", [])),
        "옵션_기타개수": len(emb.get("option_etc_values", [])),
        "옵션_기본코드": ",".join(emb.get("option_standard_codes", [])) if emb.get("option_standard_codes") else None,
        "옵션_선택코드": ",".join(emb.get("option_choice_codes", [])) if emb.get("option_choice_codes") else None,
        "옵션_튜닝코드": ",".join(emb.get("option_tuning_codes", [])) if emb.get("option_tuning_codes") else None,
        "옵션_기타값": " | ".join(emb.get("option_etc_values", [])) if emb.get("option_etc_values") else None,
    }

    row.update(major_options)
    return row


def parse_newcar_file(newcar_path: Path):
    html = read_text_safe(newcar_path)
    soup = BeautifulSoup(html, "lxml")
    text = normalize_text(soup.get_text(" ", strip=True))

    note = find_first([r"(신차가는 판매자가 입력한 등급\+선택옵션 기준으로 산출되었습니다\.)"], text)

    direct_price = find_first([
        r"([\d,]+)만원\s*\(신차가\)",
        r"신차가\s*[:：]?\s*([\d,]+)\s*만원",
        r"출고가\s*[:：]?\s*([\d,]+)\s*만원",
    ], text, cast=to_int)

    grade_base_price = find_first([
        r"등급 기준\s*([\d,]+)\s*만원",
        r"선택옵션 미포함 가격입니다\.\s*.*?([\d,]+)\s*만원"
    ], text, cast=to_int)

    option_price = find_first([
        r"선택옵션\s*\d+종\s*([\d,]+)\s*만원"
    ], text, cast=to_int)

    return {
        "newcar_note": note,
        "newcar_direct_price_만원": direct_price,
        "newcar_grade_base_price_만원": grade_base_price,
        "newcar_option_price_만원": option_price,
        "newcar_raw_preview": text[:700] if text else None,
    }


def parse_option_file(option_path: Path):
    html = read_text_safe(option_path)
    soup = BeautifulSoup(html, "lxml")
    text = normalize_text(soup.get_text(" ", strip=True))

    option_count = len(re.findall(r"(있음|없음)", text))

    return {
        "option_raw_preview": text[:500] if text else None,
        "option_visible_status_token_count": option_count,
    }


def main():
    rows = []

    for car_dir in sorted([p for p in BASE_DIR.iterdir() if p.is_dir()]):
        detail_path = car_dir / "detail.html"
        newcar_path = car_dir / "newcar.html"
        option_path = car_dir / "option.html"

        if not detail_path.exists():
            continue

        row = parse_detail_file(detail_path)

        if newcar_path.exists():
            row.update(parse_newcar_file(newcar_path))
        if option_path.exists():
            row.update(parse_option_file(option_path))

        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv("encar_parsed_from_saved_html_v4.csv", index=False, encoding="utf-8-sig")

    with open("encar_parsed_from_saved_html_v4.json", "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

    print("저장 완료: encar_parsed_from_saved_html_v4.csv")
    print("저장 완료: encar_parsed_from_saved_html_v4.json")
    print(df.head())
    print(df.columns.tolist())


if __name__ == "__main__":
    main()

저장 완료: encar_parsed_from_saved_html_v4.csv
저장 완료: encar_parsed_from_saved_html_v4.json
       매물ID 제조사      모델             등급명  세부트림                    차량명  현재가격_만원  \
0  39486507  현대  YF 쏘나타           LPI 탑  None           YF 쏘나타 LPI 탑      250   
1  40903390  현대  YF 쏘나타  LPI 프리미어(장애인용)  None  YF 쏘나타 LPI 프리미어(장애인용)      330   
2  40955073  현대  YF 쏘나타         Y20 프라임   고급형     YF 쏘나타 Y20 프라임 고급형      430   
3  41054615  현대  YF 쏘나타        CVVL 럭셔리  None        YF 쏘나타 CVVL 럭셔리      430   
4  41509017  현대  YF 쏘나타         Y20 프라임    블랙      YF 쏘나타 Y20 프라임 블랙      399   

   신차기준가_만원   조회수  찜수  ... 주요옵션_열선시트 주요옵션_통풍시트 주요옵션_가죽시트  \
0       NaN  2450  58  ...         1         0         1   
1       NaN   129   1  ...         1         0         1   
2    2345.0  1458  20  ...         1         0         1   
3    2450.0  1057  26  ...         1         0         1   
4    2410.0   910  16  ...         1         0         1   

                           newcar_note newcar_direct_price_만원  \


In [ ]:
import re
import time
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


SEARCH_URL = (
    "https://car.encar.com/list/car?page=1"
    "&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._."
    "%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._."
    "%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C"
    "%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D"
)


def setup_driver(headless=False):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--lang=ko-KR")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=options)
    driver.implicitly_wait(3)
    return driver


def wait_body(driver, sec=10):
    WebDriverWait(driver, sec).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )


def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip() if text else ""


def extract_car_id(url):
    m = re.search(r"/cars/detail/(\d+)", url)
    return m.group(1) if m else None


def close_popups(driver):
    popup_xpaths = [
        "//button[contains(., '닫기')]",
        "//button[contains(., '나중에')]",
        "//button[contains(., '오늘 보지 않기')]",
        "//button[contains(., '취소')]",
        "//button[contains(., '확인')]",
        "//button[contains(., '다음에')]",
        "//a[contains(., '닫기')]",
    ]

    for xp in popup_xpaths:
        try:
            buttons = driver.find_elements(By.XPATH, xp)
            for btn in buttons:
                try:
                    if btn.is_displayed():
                        driver.execute_script("arguments[0].click();", btn)
                        time.sleep(0.3)
                except Exception:
                    pass
        except Exception:
            pass


def scroll_until_stable(driver, pause=2.0, max_rounds=30):
    last_height = driver.execute_script("return document.body.scrollHeight")
    stable_count = 0

    for _ in range(max_rounds):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)

        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            stable_count += 1
        else:
            stable_count = 0

        if stable_count >= 2:
            break

        last_height = new_height


def collect_detail_links(driver):
    anchors = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/cars/detail/"]')
    urls = []

    for a in anchors:
        try:
            href = a.get_attribute("href")
            if href and "/cars/detail/" in href:
                href = href.split("&advClickPosition=")[0]
                urls.append(href)
        except Exception:
            pass

    return list(dict.fromkeys(urls))


def collect_yf_links(driver, top_n=20):
    driver.get(SEARCH_URL)
    wait_body(driver)
    time.sleep(2)
    close_popups(driver)
    scroll_until_stable(driver, pause=2.0, max_rounds=25)

    links = collect_detail_links(driver)
    return links[:top_n]


def save_page_source(driver, url, html_path, txt_path=None, sleep_sec=2.0):
    driver.get(url)
    wait_body(driver)
    time.sleep(sleep_sec)
    close_popups(driver)
    time.sleep(0.5)

    html = driver.page_source
    html_path.write_text(html, encoding="utf-8")

    if txt_path is not None:
        body_text = normalize_text(driver.find_element(By.TAG_NAME, "body").text)
        txt_path.write_text(body_text, encoding="utf-8")


def collect_all_html_for_car(driver, detail_url, base_dir):
    car_id = extract_car_id(detail_url)
    if not car_id:
        return

    car_dir = Path(base_dir) / car_id
    car_dir.mkdir(parents=True, exist_ok=True)

    page_map = {
        "detail": detail_url,
        "newcar": f"https://fem.encar.com/cars/newcar/{car_id}",
        "option": f"https://fem.encar.com/cars/option/{car_id}",
    }

    for page_name, url in page_map.items():
        try:
            save_page_source(
                driver,
                url,
                html_path=car_dir / f"{page_name}.html",
                txt_path=car_dir / f"{page_name}.txt",
                sleep_sec=2.0
            )
            print(f"[OK] {car_id} - {page_name}")
        except Exception as e:
            print(f"[ERROR] {car_id} - {page_name}: {e}")


def main():
    base_dir = "encar_html_20"
    Path(base_dir).mkdir(parents=True, exist_ok=True)

    driver = setup_driver(headless=False)

    try:
        links = collect_yf_links(driver, top_n=20)
        print(f"[INFO] 대상 링크 수: {len(links)}")

        for idx, detail_url in enumerate(links, start=1):
            car_id = extract_car_id(detail_url)
            print(f"[{idx}/{len(links)}] {car_id}")
            collect_all_html_for_car(driver, detail_url, base_dir)

        print(f"\n[완료] 저장 폴더: {Path(base_dir).resolve()}")

    finally:
        driver.quit()


if __name__ == "__main__":
    main()

[INFO] 대상 링크 수: 20
[1/20] 40477859
[OK] 40477859 - detail
[OK] 40477859 - newcar


In [1]:
import re
import json
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup


BASE_DIR = Path("encar_html_20")
OUT_CSV = "encar_yf_20_parsed_v5.csv"
OUT_JSON = "encar_yf_20_parsed_v5.json"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}


def read_text_safe(path: Path) -> str:
    if path.exists():
        return path.read_text(encoding="utf-8", errors="ignore")
    return ""


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip() if text else ""


def find_first(patterns, text, cast=None, flags=re.S):
    for pat in patterns:
        m = re.search(pat, text, flags)
        if m:
            val = m.group(1) if m.lastindex else m.group(0)
            if cast:
                try:
                    return cast(val)
                except Exception:
                    return val
            return val
    return None


def to_int(val):
    if val is None:
        return None
    return int(str(val).replace(",", "").strip())


def parse_int_from_text(text):
    if not text:
        return None
    m = re.search(r"([\d,]+)", text)
    return int(m.group(1).replace(",", "")) if m else None


def parse_json_array_str(text):
    if not text:
        return []
    try:
        return json.loads(text)
    except Exception:
        return []


def parse_meta_description(soup):
    result = {}
    meta = soup.find("meta", attrs={"name": "description"})
    if not meta:
        return result

    content = meta.get("content", "")
    result["연식_메타"] = find_first([r"연식:([^,]+)"], content)
    result["주행거리_메타"] = find_first([r"주행거리:([^,]+)"], content)
    result["연료_메타"] = find_first([r"연료:([^,]+)"], content)
    result["색상_메타"] = find_first([r"색상:([^,]+)"], content)
    result["지역_메타"] = find_first([r"지역:([^,]+?) 중고차"], content)
    return result


def parse_dom_basic(soup):
    result = {
        "차량명_dom": None,
        "모델_dom": None,
        "세부트림_dom": None,
        "연식_원문_dom": None,
        "주행거리_원문_dom": None,
        "연료_dom": None,
        "차량번호_dom": None,
        "등록번호_dom": None,
        "조회수_dom": None,
        "찜수_dom": None,
        "해시태그_dom": None,
        "실촬영문구_dom": None,
    }

    h3 = soup.find("h3")
    if h3:
        spans = [normalize_text(x.get_text(" ", strip=True)) for x in h3.find_all("span")]
        spans = [x for x in spans if x]
        if len(spans) >= 2:
            result["모델_dom"] = spans[0]
            result["세부트림_dom"] = spans[1]
            result["차량명_dom"] = " ".join(spans).strip()
        else:
            result["차량명_dom"] = normalize_text(h3.get_text(" ", strip=True))

    dl = soup.select_one("dl.ar1Ivd7EgX")
    if dl:
        dts = dl.find_all("dt")
        dds = dl.find_all("dd")
        for dt, dd in zip(dts, dds):
            k = normalize_text(dt.get_text(" ", strip=True))
            v = normalize_text(dd.get_text(" ", strip=True))
            if "연식" in k:
                result["연식_원문_dom"] = v
            elif "주행거리" in k:
                result["주행거리_원문_dom"] = v
            elif "연료" in k:
                result["연료_dom"] = v
            elif "차량번호" in k:
                result["차량번호_dom"] = v

    for li in soup.select("ul.rVigc5A_1H li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt.startswith("등록번호"):
            result["등록번호_dom"] = txt.replace("등록번호", "").strip()
        elif txt.startswith("조회수"):
            result["조회수_dom"] = parse_int_from_text(txt)
        elif txt.startswith("찜"):
            result["찜수_dom"] = parse_int_from_text(txt)

    tags = [normalize_text(li.get_text(" ", strip=True)) for li in soup.select("ul.kYC8KS_YZ1 li")]
    tags = [x for x in tags if x]
    result["해시태그_dom"] = ",".join(tags) if tags else None

    shot = soup.select_one("p.s8FzQTBVpE")
    if shot:
        result["실촬영문구_dom"] = normalize_text(shot.get_text(" ", strip=True))

    return result


def parse_detail_embedded(html):
    result = {}

    str_patterns = {
        "manufacturerName": [r'"manufacturerName":"([^"]+)"'],
        "modelName": [r'"modelName":"([^"]+)"'],
        "gradeName": [r'"gradeName":"([^"]+)"'],
        "gradeDetailName": [r'"gradeDetailName":"([^"]+)"'],
        "vehicleNo": [r'"vehicleNo":"([^"]+)"'],
        "vin": [r'"vin":"([^"]+)"'],
        "requestUrl": [r'"requestUrl":"([^"]+)"'],
        "dealerName": [r'"dealer":\{"userId":"[^"]+","name":"([^"]+)"'],
        "firmName": [r'"firm":\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterName": [r'"diagnosisCenters":\[\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterPhone": [r'"telephoneNumber":"([^"]+)"'],
        "diagnosisCenterAddress": [r'"address":"([^"]+)"'],
        "pageAccessToken": [r'"pageAccessToken":"([^"]+)"'],
        "yearMonth": [r'"yearMonth":"([^"]+)"'],
        "formYear": [r'"formYear":"([^"]+)"'],
        "fuelName": [r'"fuelName":"([^"]+)"'],
        "colorName": [r'"colorName":"([^"]+)"'],
        "transmissionName": [r'"transmissionName":"([^"]+)"'],
        "bodyName": [r'"bodyName":"([^"]+)"'],
        "oneLineText": [r'"oneLineText":"([^"]+)"'],
    }

    int_patterns = {
        "originPrice": [r'"originPrice":(\d+)'],
        "price": [r'"price":(\d+)'],
        "viewCount": [r'"viewCount":(\d+)'],
        "subscribeCount": [r'"subscribeCount":(\d+)'],
        "vehicleId": [r'"vehicleId":(\d+)'],
        "seizingCount": [r'"seizingCount":(\d+)'],
        "pledgeCount": [r'"pledgeCount":(\d+)'],
        "encarDiagnosis": [r'"encarDiagnosis":(-?\d+)'],
        "encarMeetGo": [r'"encarMeetGo":(-?\d+)'],
        "mileage": [r'"mileage":(\d+)'],
        "displacement": [r'"displacement":(\d+)'],
        "seatCount": [r'"seatCount":(\d+)'],
    }

    for key, pats in str_patterns.items():
        result[key] = find_first(pats, html)

    for key, pats in int_patterns.items():
        result[key] = find_first(pats, html, cast=to_int)

    standard_raw = find_first([r'"standard":(\[[^\]]*\])'], html)
    choice_raw = find_first([r'"choice":(\[[^\]]*\])'], html)
    tuning_raw = find_first([r'"tuning":(\[[^\]]*\])'], html)
    etc_raw = find_first([r'"etc":(\[[^\]]*\]|"[^"]*")'], html)

    result["option_standard_codes"] = parse_json_array_str(standard_raw)
    result["option_choice_codes"] = parse_json_array_str(choice_raw)
    result["option_tuning_codes"] = parse_json_array_str(tuning_raw)

    if etc_raw and etc_raw.startswith("["):
        result["option_etc_values"] = parse_json_array_str(etc_raw)
    elif etc_raw:
        result["option_etc_values"] = [etc_raw.strip('"')]
    else:
        result["option_etc_values"] = []

    return result


def parse_dom_major_options(soup):
    result = {}
    for li in soup.select("ul.dz3qFYruNO li"):
        text = normalize_text(li.get_text(" ", strip=True))
        blind = li.select_one("span.blind")
        status = normalize_text(blind.get_text(" ", strip=True)) if blind else None

        if status:
            name = normalize_text(text.replace(status, ""))
            result[f"주요옵션_{name}"] = 1 if status == "있음" else 0 if status == "없음" else None

    return result


def parse_dom_seller_info(soup):
    result = {
        "판매자상호_dom": None,
        "판매자명_dom": None,
        "판매자유형_dom": None,
        "판매중대수_dom": None,
        "판매완료대수_dom": None,
        "판매자지역_dom": None,
        "종사원증번호_dom": None,
    }

    btn = soup.select_one("button.uPEfVsKnZx")
    if btn:
        brand = btn.select_one("span.ESlvHTih77")
        name = btn.select_one("strong.k3tS4rXdrQ")
        seller_type = btn.select_one("span.xf9ufEfgKR")

        if brand:
            result["판매자상호_dom"] = normalize_text(brand.get_text(" ", strip=True))
        if name:
            result["판매자명_dom"] = normalize_text(name.get_text(" ", strip=True))
        if seller_type:
            result["판매자유형_dom"] = normalize_text(seller_type.get_text(" ", strip=True))

    lis = soup.select("ul.VtNR8dNHOS li")
    for li in lis:
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt.startswith("판매중"):
            result["판매중대수_dom"] = parse_int_from_text(txt)
        elif txt.startswith("판매완료"):
            result["판매완료대수_dom"] = parse_int_from_text(txt)
        elif "종사원증번호" in txt:
            m = re.search(r"종사원증번호\s*([A-Z0-9\-]+)", txt)
            if m:
                result["종사원증번호_dom"] = m.group(1)
        elif any(region in txt for region in ["서울", "경기", "인천", "부산", "대구", "대전", "광주", "울산", "세종", "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"]):
            result["판매자지역_dom"] = txt

    return result


def parse_dom_extra_flags(soup):
    result = {
        "보험이력공개여부_dom": None,
        "성능점검유형_dom": None,
        "신차가격표링크_dom": None,
        "성능점검설명_dom": None,
    }

    for btn in soup.select("button.fSZoGuAX4M"):
        txt = normalize_text(btn.get_text(" ", strip=True))
        if "보험이력" in txt:
            result["보험이력공개여부_dom"] = "비공개" if "비공개" in txt else txt
        elif "성능점검내역" in txt:
            result["성능점검유형_dom"] = txt.replace("성능점검내역", "").strip()

    newprice_link = soup.select_one('div[data-impression="신차비교"] a[href*="newprice"]')
    if newprice_link:
        result["신차가격표링크_dom"] = newprice_link.get("href")

    perf_desc = soup.select_one("div.Hs3feBPsTj p.lGhsYmQaGE")
    if perf_desc:
        result["성능점검설명_dom"] = normalize_text(perf_desc.get_text(" ", strip=True))

    return result


def build_full_trim(model, grade, detail):
    parts = []
    for p in [model, grade, detail]:
        p = normalize_text(p)
        if p and p not in parts:
            parts.append(p)
    return " ".join(parts) if parts else None


def convert_year_from_embedded(year_month, form_year, fallback_text):
    if form_year:
        try:
            return int(form_year)
        except Exception:
            pass
    if year_month and len(year_month) >= 4:
        try:
            return int(year_month[:4])
        except Exception:
            pass
    if fallback_text:
        m = re.search(r"(\d{2})/", fallback_text)
        if m:
            return 2000 + int(m.group(1))
    return None


def parse_detail_file(detail_path: Path):
    html = read_text_safe(detail_path)
    soup = BeautifulSoup(html, "lxml")

    meta = parse_meta_description(soup)
    dom = parse_dom_basic(soup)
    emb = parse_detail_embedded(html)
    major_options = parse_dom_major_options(soup)
    seller_dom = parse_dom_seller_info(soup)
    flags_dom = parse_dom_extra_flags(soup)

    model = emb.get("modelName") or dom.get("모델_dom")
    grade = emb.get("gradeName")
    detail = emb.get("gradeDetailName") or dom.get("세부트림_dom")

    if normalize_text(grade) == normalize_text(detail):
        detail = None

    row = {
        "매물ID": emb.get("vehicleId") or detail_path.parent.name,
        "제조사": emb.get("manufacturerName"),
        "모델": model,
        "등급명": grade,
        "세부트림": detail,
        "차량명": build_full_trim(model, grade, detail),
        "현재가격_만원": emb.get("price"),
        "신차기준가_만원": emb.get("originPrice"),
        "조회수": emb.get("viewCount") or dom.get("조회수_dom"),
        "찜수": emb.get("subscribeCount") or dom.get("찜수_dom"),
        "차량번호": emb.get("vehicleNo") or dom.get("차량번호_dom"),
        "VIN": emb.get("vin"),
        "연식_원문": dom.get("연식_원문_dom") or meta.get("연식_메타"),
        "연식": convert_year_from_embedded(emb.get("yearMonth"), emb.get("formYear"), dom.get("연식_원문_dom") or meta.get("연식_메타")),
        "주행거리_원문": dom.get("주행거리_원문_dom") or meta.get("주행거리_메타"),
        "주행거리_km": emb.get("mileage") or parse_int_from_text(dom.get("주행거리_원문_dom") or meta.get("주행거리_메타")),
        "연료": emb.get("fuelName") or dom.get("연료_dom") or meta.get("연료_메타"),
        "색상": emb.get("colorName") or meta.get("색상_메타"),
        "지역": meta.get("지역_메타"),
        "변속기": emb.get("transmissionName"),
        "배기량_cc": emb.get("displacement"),
        "차급": emb.get("bodyName"),
        "좌석수": emb.get("seatCount"),
        "등록번호": dom.get("등록번호_dom"),
        "해시태그": dom.get("해시태그_dom"),
        "실촬영문구": dom.get("실촬영문구_dom"),
        "딜러명": emb.get("dealerName") or seller_dom.get("판매자명_dom"),
        "상사명": emb.get("firmName") or seller_dom.get("판매자상호_dom"),
        "판매자유형": seller_dom.get("판매자유형_dom"),
        "판매중대수": seller_dom.get("판매중대수_dom"),
        "판매완료대수": seller_dom.get("판매완료대수_dom"),
        "판매자지역": seller_dom.get("판매자지역_dom"),
        "종사원증번호": seller_dom.get("종사원증번호_dom"),
        "진단센터명": emb.get("diagnosisCenterName"),
        "진단센터전화": emb.get("diagnosisCenterPhone"),
        "진단센터주소": emb.get("diagnosisCenterAddress"),
        "압류건수": emb.get("seizingCount"),
        "저당건수": emb.get("pledgeCount"),
        "엔카진단여부값": emb.get("encarDiagnosis"),
        "엔카믿고값": emb.get("encarMeetGo"),
        "광고한줄문구": emb.get("oneLineText"),
        "보험이력공개여부": flags_dom.get("보험이력공개여부_dom"),
        "성능점검유형": flags_dom.get("성능점검유형_dom"),
        "성능점검설명": flags_dom.get("성능점검설명_dom"),
        "신차가격표링크": flags_dom.get("신차가격표링크_dom"),
        "requestUrl": emb.get("requestUrl"),
        "pageAccessToken": emb.get("pageAccessToken"),
        "옵션_기본코드개수": len(emb.get("option_standard_codes", [])),
        "옵션_선택코드개수": len(emb.get("option_choice_codes", [])),
        "옵션_튜닝코드개수": len(emb.get("option_tuning_codes", [])),
        "옵션_기타개수": len(emb.get("option_etc_values", [])),
        "옵션_기본코드": ",".join(emb.get("option_standard_codes", [])) if emb.get("option_standard_codes") else None,
        "옵션_선택코드": ",".join(emb.get("option_choice_codes", [])) if emb.get("option_choice_codes") else None,
        "옵션_튜닝코드": ",".join(emb.get("option_tuning_codes", [])) if emb.get("option_tuning_codes") else None,
        "옵션_기타값": " | ".join(emb.get("option_etc_values", [])) if emb.get("option_etc_values") else None,
    }

    row.update(major_options)
    return row


def parse_newcar_file(newcar_path: Path):
    html = read_text_safe(newcar_path)
    soup = BeautifulSoup(html, "lxml")
    text = normalize_text(soup.get_text(" ", strip=True))

    note = find_first([r"(신차가는 판매자가 입력한 등급\+선택옵션 기준으로 산출되었습니다\.)"], text)

    direct_price = find_first([
        r"([\d,]+)만원\s*\(신차가\)",
        r"신차가\s*[:：]?\s*([\d,]+)\s*만원",
        r"출고가\s*[:：]?\s*([\d,]+)\s*만원",
    ], text, cast=to_int)

    grade_base_price = find_first([
        r"등급 기준\s*([\d,]+)\s*만원",
        r"선택옵션 미포함 가격입니다\.\s*.*?([\d,]+)\s*만원"
    ], text, cast=to_int)

    option_price = find_first([
        r"선택옵션\s*\d+종\s*([\d,]+)\s*만원"
    ], text, cast=to_int)

    return {
        "newcar_note": note,
        "newcar_direct_price_만원": direct_price,
        "newcar_grade_base_price_만원": grade_base_price,
        "newcar_option_price_만원": option_price,
        "newcar_raw_preview": text[:700] if text else None,
    }


def parse_option_file(option_path: Path):
    html = read_text_safe(option_path)
    soup = BeautifulSoup(html, "lxml")
    text = normalize_text(soup.get_text(" ", strip=True))

    option_count = len(re.findall(r"(있음|없음)", text))

    return {
        "option_raw_preview": text[:500] if text else None,
        "option_visible_status_token_count": option_count,
    }


def parse_newprice_page(price_url):
    try:
        r = requests.get(price_url, headers=HEADERS, timeout=15)
        r.raise_for_status()

        soup = BeautifulSoup(r.text, "lxml")
        text = " ".join(soup.stripped_strings)

        patterns = [
            r"([\d,]+)\s*만원",
            r"판매가격\s*([\d,]+)",
            r"기본가격\s*([\d,]+)",
        ]

        prices = []
        for pat in patterns:
            for m in re.finditer(pat, text):
                try:
                    prices.append(int(m.group(1).replace(",", "")))
                except Exception:
                    pass

        prices = [p for p in prices if 500 <= p <= 10000]

        if prices:
            return min(prices), text[:2000]

        return None, text[:2000]

    except Exception as e:
        return None, f"ERROR: {e}"


def parse_newprice_link_fallback(row):
    link = row.get("신차가격표링크")
    if not link:
        row["newprice_link_price_만원"] = None
        row["newprice_link_raw_preview"] = None
        return row

    price, preview = parse_newprice_page(link)
    row["newprice_link_price_만원"] = price
    row["newprice_link_raw_preview"] = preview[:500] if preview else None
    return row


def finalize_newcar_price(row):
    for key in ["newcar_direct_price_만원", "신차기준가_만원", "newprice_link_price_만원"]:
        val = row.get(key)
        if val is not None:
            return val
    return None


def main():
    rows = []

    for car_dir in sorted([p for p in BASE_DIR.iterdir() if p.is_dir()]):
        detail_path = car_dir / "detail.html"
        newcar_path = car_dir / "newcar.html"
        option_path = car_dir / "option.html"

        if not detail_path.exists():
            continue

        row = parse_detail_file(detail_path)

        if newcar_path.exists():
            row.update(parse_newcar_file(newcar_path))
        if option_path.exists():
            row.update(parse_option_file(option_path))

        row = parse_newprice_link_fallback(row)
        row["최종신차가_만원"] = finalize_newcar_price(row)

        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

    with open(OUT_JSON, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

    print(f"저장 완료: {OUT_CSV}")
    print(f"저장 완료: {OUT_JSON}")
    print(df[[
        "매물ID", "차량명", "현재가격_만원",
        "신차기준가_만원", "newcar_direct_price_만원",
        "newprice_link_price_만원", "최종신차가_만원"
    ]].head())


if __name__ == "__main__":
    main()

저장 완료: encar_yf_20_parsed_v5.csv
저장 완료: encar_yf_20_parsed_v5.json
       매물ID                    차량명  현재가격_만원 신차기준가_만원 newcar_direct_price_만원  \
0  40477859  YF 쏘나타 LPI 프리미어(장애인용)      429     None                   None   

  newprice_link_price_만원 최종신차가_만원  
0                   None     None  


In [3]:
import re
import time
import json
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


TOP_N = 20
BASE_SAVE_DIR = Path("encar_yf_detail_20")
OUT_CSV = "encar_yf_detail_20.csv"
OUT_JSON = "encar_yf_detail_20.json"

SEARCH_URL = (
    "https://car.encar.com/list/car?page=1"
    "&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._."
    "%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._."
    "%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C"
    "%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D"
)


# -------------------------
# 공통 유틸
# -------------------------
def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip() if text else ""


def find_first(patterns, text, cast=None, flags=re.S):
    for pat in patterns:
        m = re.search(pat, text, flags)
        if m:
            val = m.group(1) if m.lastindex else m.group(0)
            if cast:
                try:
                    return cast(val)
                except Exception:
                    return val
            return val
    return None


def to_int(val):
    if val is None:
        return None
    return int(str(val).replace(",", "").strip())


def parse_int_from_text(text):
    if not text:
        return None
    m = re.search(r"([\d,]+)", text)
    return int(m.group(1).replace(",", "")) if m else None


def parse_json_array_str(text):
    if not text:
        return []
    try:
        return json.loads(text)
    except Exception:
        return []


def extract_car_id(url):
    m = re.search(r"/cars/detail/(\d+)", url)
    return m.group(1) if m else None


def build_full_trim(model, grade, detail):
    parts = []
    for p in [model, grade, detail]:
        p = normalize_text(p)
        if p and p not in parts:
            parts.append(p)
    return " ".join(parts) if parts else None


def convert_year_from_embedded(year_month, form_year, fallback_text):
    if form_year:
        try:
            return int(form_year)
        except Exception:
            pass
    if year_month and len(year_month) >= 4:
        try:
            return int(year_month[:4])
        except Exception:
            pass
    if fallback_text:
        m = re.search(r"(\d{2})/", fallback_text)
        if m:
            return 2000 + int(m.group(1))
    return None


# -------------------------
# Selenium
# -------------------------
def setup_driver(headless=False):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--lang=ko-KR")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=options)
    driver.implicitly_wait(3)
    return driver


def wait_body(driver, sec=10):
    WebDriverWait(driver, sec).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )


def close_popups(driver):
    popup_xpaths = [
        "//button[contains(., '닫기')]",
        "//button[contains(., '나중에')]",
        "//button[contains(., '오늘 보지 않기')]",
        "//button[contains(., '취소')]",
        "//button[contains(., '확인')]",
        "//button[contains(., '다음에')]",
        "//a[contains(., '닫기')]",
    ]

    for xp in popup_xpaths:
        try:
            buttons = driver.find_elements(By.XPATH, xp)
            for btn in buttons:
                try:
                    if btn.is_displayed():
                        driver.execute_script("arguments[0].click();", btn)
                        time.sleep(0.3)
                except Exception:
                    pass
        except Exception:
            pass


def scroll_until_stable(driver, pause=2.0, max_rounds=25):
    last_height = driver.execute_script("return document.body.scrollHeight")
    stable_count = 0

    for _ in range(max_rounds):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)

        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            stable_count += 1
        else:
            stable_count = 0

        if stable_count >= 2:
            break

        last_height = new_height


def collect_detail_links(driver):
    anchors = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/cars/detail/"]')
    urls = []

    for a in anchors:
        try:
            href = a.get_attribute("href")
            if href and "/cars/detail/" in href:
                href = href.split("&advClickPosition=")[0]
                urls.append(href)
        except Exception:
            pass

    return list(dict.fromkeys(urls))


def collect_yf_links(driver, top_n=20):
    driver.get(SEARCH_URL)
    wait_body(driver)
    time.sleep(2)
    close_popups(driver)
    scroll_until_stable(driver, pause=2.0, max_rounds=25)

    links = collect_detail_links(driver)
    return links[:top_n]


# -------------------------
# detail 파싱
# -------------------------
def parse_meta_description(soup):
    result = {}
    meta = soup.find("meta", attrs={"name": "description"})
    if not meta:
        return result

    content = meta.get("content", "")
    result["연식_메타"] = find_first([r"연식:([^,]+)"], content)
    result["주행거리_메타"] = find_first([r"주행거리:([^,]+)"], content)
    result["연료_메타"] = find_first([r"연료:([^,]+)"], content)
    result["색상_메타"] = find_first([r"색상:([^,]+)"], content)
    result["지역_메타"] = find_first([r"지역:([^,]+?) 중고차"], content)
    return result


def parse_dom_basic(soup):
    result = {
        "차량명_dom": None,
        "모델_dom": None,
        "세부트림_dom": None,
        "연식_원문_dom": None,
        "주행거리_원문_dom": None,
        "연료_dom": None,
        "차량번호_dom": None,
        "등록번호_dom": None,
        "조회수_dom": None,
        "찜수_dom": None,
        "해시태그_dom": None,
        "실촬영문구_dom": None,
    }

    h3 = soup.find("h3")
    if h3:
        spans = [normalize_text(x.get_text(" ", strip=True)) for x in h3.find_all("span")]
        spans = [x for x in spans if x]
        if len(spans) >= 2:
            result["모델_dom"] = spans[0]
            result["세부트림_dom"] = spans[1]
            result["차량명_dom"] = " ".join(spans).strip()
        else:
            result["차량명_dom"] = normalize_text(h3.get_text(" ", strip=True))

    dl = soup.select_one("dl.ar1Ivd7EgX")
    if dl:
        dts = dl.find_all("dt")
        dds = dl.find_all("dd")
        for dt, dd in zip(dts, dds):
            k = normalize_text(dt.get_text(" ", strip=True))
            v = normalize_text(dd.get_text(" ", strip=True))
            if "연식" in k:
                result["연식_원문_dom"] = v
            elif "주행거리" in k:
                result["주행거리_원문_dom"] = v
            elif "연료" in k:
                result["연료_dom"] = v
            elif "차량번호" in k:
                result["차량번호_dom"] = v

    for li in soup.select("ul.rVigc5A_1H li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt.startswith("등록번호"):
            result["등록번호_dom"] = txt.replace("등록번호", "").strip()
        elif txt.startswith("조회수"):
            result["조회수_dom"] = parse_int_from_text(txt)
        elif txt.startswith("찜"):
            result["찜수_dom"] = parse_int_from_text(txt)

    tags = [normalize_text(li.get_text(" ", strip=True)) for li in soup.select("ul.kYC8KS_YZ1 li")]
    tags = [x for x in tags if x]
    result["해시태그_dom"] = ",".join(tags) if tags else None

    shot = soup.select_one("p.s8FzQTBVpE")
    if shot:
        result["실촬영문구_dom"] = normalize_text(shot.get_text(" ", strip=True))

    return result


def parse_detail_embedded(html):
    result = {}

    str_patterns = {
        "manufacturerName": [r'"manufacturerName":"([^"]+)"'],
        "modelName": [r'"modelName":"([^"]+)"'],
        "gradeName": [r'"gradeName":"([^"]+)"'],
        "gradeDetailName": [r'"gradeDetailName":"([^"]+)"'],
        "vehicleNo": [r'"vehicleNo":"([^"]+)"'],
        "vin": [r'"vin":"([^"]+)"'],
        "requestUrl": [r'"requestUrl":"([^"]+)"'],
        "dealerName": [r'"dealer":\{"userId":"[^"]+","name":"([^"]+)"'],
        "firmName": [r'"firm":\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterName": [r'"diagnosisCenters":\[\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterPhone": [r'"telephoneNumber":"([^"]+)"'],
        "diagnosisCenterAddress": [r'"address":"([^"]+)"'],
        "pageAccessToken": [r'"pageAccessToken":"([^"]+)"'],
        "yearMonth": [r'"yearMonth":"([^"]+)"'],
        "formYear": [r'"formYear":"([^"]+)"'],
        "fuelName": [r'"fuelName":"([^"]+)"'],
        "colorName": [r'"colorName":"([^"]+)"'],
        "transmissionName": [r'"transmissionName":"([^"]+)"'],
        "bodyName": [r'"bodyName":"([^"]+)"'],
        "oneLineText": [r'"oneLineText":"([^"]+)"'],
    }

    int_patterns = {
        "originPrice": [r'"originPrice":(\d+)'],
        "price": [r'"price":(\d+)'],
        "viewCount": [r'"viewCount":(\d+)'],
        "subscribeCount": [r'"subscribeCount":(\d+)'],
        "vehicleId": [r'"vehicleId":(\d+)'],
        "seizingCount": [r'"seizingCount":(\d+)'],
        "pledgeCount": [r'"pledgeCount":(\d+)'],
        "encarDiagnosis": [r'"encarDiagnosis":(-?\d+)'],
        "encarMeetGo": [r'"encarMeetGo":(-?\d+)'],
        "mileage": [r'"mileage":(\d+)'],
        "displacement": [r'"displacement":(\d+)'],
        "seatCount": [r'"seatCount":(\d+)'],
    }

    for key, pats in str_patterns.items():
        result[key] = find_first(pats, html)

    for key, pats in int_patterns.items():
        result[key] = find_first(pats, html, cast=to_int)

    standard_raw = find_first([r'"standard":(\[[^\]]*\])'], html)
    choice_raw = find_first([r'"choice":(\[[^\]]*\])'], html)
    tuning_raw = find_first([r'"tuning":(\[[^\]]*\])'], html)
    etc_raw = find_first([r'"etc":(\[[^\]]*\]|"[^"]*")'], html)

    result["option_standard_codes"] = parse_json_array_str(standard_raw)
    result["option_choice_codes"] = parse_json_array_str(choice_raw)
    result["option_tuning_codes"] = parse_json_array_str(tuning_raw)

    if etc_raw and etc_raw.startswith("["):
        result["option_etc_values"] = parse_json_array_str(etc_raw)
    elif etc_raw:
        result["option_etc_values"] = [etc_raw.strip('"')]
    else:
        result["option_etc_values"] = []

    return result


def parse_dom_major_options(soup):
    result = {}
    for li in soup.select("ul.dz3qFYruNO li"):
        text = normalize_text(li.get_text(" ", strip=True))
        blind = li.select_one("span.blind")
        status = normalize_text(blind.get_text(" ", strip=True)) if blind else None

        if status:
            name = normalize_text(text.replace(status, ""))
            result[f"주요옵션_{name}"] = 1 if status == "있음" else 0 if status == "없음" else None

    return result


def parse_dom_seller_info(soup):
    result = {
        "판매자상호_dom": None,
        "판매자명_dom": None,
        "판매자유형_dom": None,
        "판매중대수_dom": None,
        "판매완료대수_dom": None,
        "판매자지역_dom": None,
        "종사원증번호_dom": None,
    }

    btn = soup.select_one("button.uPEfVsKnZx")
    if btn:
        brand = btn.select_one("span.ESlvHTih77")
        name = btn.select_one("strong.k3tS4rXdrQ")
        seller_type = btn.select_one("span.xf9ufEfgKR")

        if brand:
            result["판매자상호_dom"] = normalize_text(brand.get_text(" ", strip=True))
        if name:
            result["판매자명_dom"] = normalize_text(name.get_text(" ", strip=True))
        if seller_type:
            result["판매자유형_dom"] = normalize_text(seller_type.get_text(" ", strip=True))

    lis = soup.select("ul.VtNR8dNHOS li")
    for li in lis:
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt.startswith("판매중"):
            result["판매중대수_dom"] = parse_int_from_text(txt)
        elif txt.startswith("판매완료"):
            result["판매완료대수_dom"] = parse_int_from_text(txt)
        elif "종사원증번호" in txt:
            m = re.search(r"종사원증번호\s*([A-Z0-9\-]+)", txt)
            if m:
                result["종사원증번호_dom"] = m.group(1)
        elif any(region in txt for region in ["서울", "경기", "인천", "부산", "대구", "대전", "광주", "울산", "세종", "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"]):
            result["판매자지역_dom"] = txt

    return result


def parse_dom_extra_flags(soup):
    result = {
        "보험이력공개여부_dom": None,
        "보험이력건수_dom": None,
        "성능점검유형_dom": None,
        "신차가격표링크_dom": None,
        "성능점검설명_dom": None,
    }

    for btn in soup.select("button.fSZoGuAX4M"):
        txt = normalize_text(btn.get_text(" ", strip=True))
        if "보험이력" in txt:
            result["보험이력공개여부_dom"] = txt
            m = re.search(r"보험이력\s*(\d+)건", txt)
            if m:
                result["보험이력건수_dom"] = int(m.group(1))
        elif "성능점검내역" in txt:
            result["성능점검유형_dom"] = txt.replace("성능점검내역", "").strip()

    newprice_link = soup.select_one('div[data-impression="신차비교"] a[href*="newprice"]')
    if newprice_link:
        result["신차가격표링크_dom"] = newprice_link.get("href")

    perf_desc = soup.select_one("div.Hs3feBPsTj p.lGhsYmQaGE")
    if perf_desc:
        result["성능점검설명_dom"] = normalize_text(perf_desc.get_text(" ", strip=True))

    return result


def parse_damage_detail(soup):
    result = {
        "교환_개수": None,
        "판금_개수": None,
        "부식_여부": None,
        "성능기록부_raw": None,
    }

    items = []
    for li in soup.select("ul.wB0X7nC0cq li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt:
            items.append(txt)

        if "교환" in txt:
            if "없음" in txt:
                result["교환_개수"] = 0
            else:
                m = re.search(r"교환\s*(\d+)", txt)
                if m:
                    result["교환_개수"] = int(m.group(1))

        elif "판금" in txt:
            if "없음" in txt:
                result["판금_개수"] = 0
            else:
                m = re.search(r"판금\s*(\d+)", txt)
                if m:
                    result["판금_개수"] = int(m.group(1))

        elif "부식" in txt:
            result["부식_여부"] = 0 if "없음" in txt else 1

    result["성능기록부_raw"] = " || ".join(items) if items else None
    return result


def derive_accident_features(row):
    exchange_cnt = row.get("교환_개수")
    panel_cnt = row.get("판금_개수")
    corrosion = row.get("부식_여부")
    insurance_cnt = row.get("보험이력건수")

    row["교환_여부"] = None if exchange_cnt is None else int(exchange_cnt > 0)
    row["판금_여부"] = None if panel_cnt is None else int(panel_cnt > 0)
    row["외판수리_여부"] = None if (exchange_cnt is None and panel_cnt is None) else int((exchange_cnt or 0) + (panel_cnt or 0) > 0)

    if exchange_cnt is None and panel_cnt is None:
        row["사고강도점수"] = None
    else:
        row["사고강도점수"] = (exchange_cnt or 0) * 2 + (panel_cnt or 0)

    row["보험이력_여부"] = None if insurance_cnt is None else int(insurance_cnt > 0)

    signals = []
    for v in [row["교환_여부"], row["판금_여부"], row["보험이력_여부"]]:
        if v is not None:
            signals.append(v)

    row["사고종합_여부"] = None if not signals else int(any(signals))

    if row["사고강도점수"] is None:
        row["중대사고_추정"] = None
    else:
        row["중대사고_추정"] = int((exchange_cnt or 0) >= 2 or row["사고강도점수"] >= 4)

    row["부식_위험"] = corrosion
    return row


def parse_detail_file(detail_html: str, car_id_hint=None):
    soup = BeautifulSoup(detail_html, "lxml")

    meta = parse_meta_description(soup)
    dom = parse_dom_basic(soup)
    emb = parse_detail_embedded(detail_html)
    major_options = parse_dom_major_options(soup)
    seller_dom = parse_dom_seller_info(soup)
    flags_dom = parse_dom_extra_flags(soup)
    damage_dom = parse_damage_detail(soup)

    model = emb.get("modelName") or dom.get("모델_dom")
    grade = emb.get("gradeName")
    detail = emb.get("gradeDetailName") or dom.get("세부트림_dom")

    if normalize_text(grade) == normalize_text(detail):
        detail = None

    row = {
        "매물ID": emb.get("vehicleId") or car_id_hint,
        "제조사": emb.get("manufacturerName"),
        "모델": model,
        "등급명": grade,
        "세부트림": detail,
        "차량명": build_full_trim(model, grade, detail),
        "현재가격_만원": emb.get("price"),
        "신차기준가_만원": emb.get("originPrice"),
        "조회수": emb.get("viewCount") or dom.get("조회수_dom"),
        "찜수": emb.get("subscribeCount") or dom.get("찜수_dom"),
        "차량번호": emb.get("vehicleNo") or dom.get("차량번호_dom"),
        "VIN": emb.get("vin"),
        "연식_원문": dom.get("연식_원문_dom") or meta.get("연식_메타"),
        "연식": convert_year_from_embedded(emb.get("yearMonth"), emb.get("formYear"), dom.get("연식_원문_dom") or meta.get("연식_메타")),
        "주행거리_원문": dom.get("주행거리_원문_dom") or meta.get("주행거리_메타"),
        "주행거리_km": emb.get("mileage") or parse_int_from_text(dom.get("주행거리_원문_dom") or meta.get("주행거리_메타")),
        "연료": emb.get("fuelName") or dom.get("연료_dom") or meta.get("연료_메타"),
        "색상": emb.get("colorName") or meta.get("색상_메타"),
        "지역": meta.get("지역_메타"),
        "변속기": emb.get("transmissionName"),
        "배기량_cc": emb.get("displacement"),
        "차급": emb.get("bodyName"),
        "좌석수": emb.get("seatCount"),
        "등록번호": dom.get("등록번호_dom"),
        "해시태그": dom.get("해시태그_dom"),
        "실촬영문구": dom.get("실촬영문구_dom"),
        "딜러명": emb.get("dealerName") or seller_dom.get("판매자명_dom"),
        "상사명": emb.get("firmName") or seller_dom.get("판매자상호_dom"),
        "판매자유형": seller_dom.get("판매자유형_dom"),
        "판매중대수": seller_dom.get("판매중대수_dom"),
        "판매완료대수": seller_dom.get("판매완료대수_dom"),
        "판매자지역": seller_dom.get("판매자지역_dom"),
        "종사원증번호": seller_dom.get("종사원증번호_dom"),
        "진단센터명": emb.get("diagnosisCenterName"),
        "진단센터전화": emb.get("diagnosisCenterPhone"),
        "진단센터주소": emb.get("diagnosisCenterAddress"),
        "압류건수": emb.get("seizingCount"),
        "저당건수": emb.get("pledgeCount"),
        "엔카진단여부값": emb.get("encarDiagnosis"),
        "엔카믿고값": emb.get("encarMeetGo"),
        "광고한줄문구": emb.get("oneLineText"),
        "보험이력공개여부": flags_dom.get("보험이력공개여부_dom"),
        "보험이력건수": flags_dom.get("보험이력건수_dom"),
        "성능점검유형": flags_dom.get("성능점검유형_dom"),
        "성능점검설명": flags_dom.get("성능점검설명_dom"),
        "신차가격표링크": flags_dom.get("신차가격표링크_dom"),
        "requestUrl": emb.get("requestUrl"),
        "pageAccessToken": emb.get("pageAccessToken"),
        "옵션_기본코드개수": len(emb.get("option_standard_codes", [])),
        "옵션_선택코드개수": len(emb.get("option_choice_codes", [])),
        "옵션_튜닝코드개수": len(emb.get("option_tuning_codes", [])),
        "옵션_기타개수": len(emb.get("option_etc_values", [])),
        "옵션_기본코드": ",".join(emb.get("option_standard_codes", [])) if emb.get("option_standard_codes") else None,
        "옵션_선택코드": ",".join(emb.get("option_choice_codes", [])) if emb.get("option_choice_codes") else None,
        "옵션_튜닝코드": ",".join(emb.get("option_tuning_codes", [])) if emb.get("option_tuning_codes") else None,
        "옵션_기타값": " | ".join(emb.get("option_etc_values", [])) if emb.get("option_etc_values") else None,
    }

    row.update(major_options)
    row.update(damage_dom)
    row = derive_accident_features(row)

    return row


# -------------------------
# newcar / option / fallback
# -------------------------
def parse_newcar_file(newcar_html: str):
    soup = BeautifulSoup(newcar_html, "lxml")
    text = normalize_text(soup.get_text(" ", strip=True))

    note = find_first([r"(신차가는 판매자가 입력한 등급\+선택옵션 기준으로 산출되었습니다\.)"], text)

    direct_price = find_first([
        r"([\d,]+)만원\s*\(신차가\)",
        r"신차가\s*[:：]?\s*([\d,]+)\s*만원",
        r"출고가\s*[:：]?\s*([\d,]+)\s*만원",
    ], text, cast=to_int)

    grade_base_price = find_first([
        r"등급 기준\s*([\d,]+)\s*만원",
        r"선택옵션 미포함 가격입니다\.\s*.*?([\d,]+)\s*만원"
    ], text, cast=to_int)

    option_price = find_first([
        r"선택옵션\s*\d+종\s*([\d,]+)\s*만원"
    ], text, cast=to_int)

    return {
        "newcar_note": note,
        "newcar_direct_price_만원": direct_price,
        "newcar_grade_base_price_만원": grade_base_price,
        "newcar_option_price_만원": option_price,
        "newcar_raw_preview": text[:700] if text else None,
    }


def parse_option_file(option_html: str):
    soup = BeautifulSoup(option_html, "lxml")
    text = normalize_text(soup.get_text(" ", strip=True))

    option_count = len(re.findall(r"(있음|없음)", text))

    return {
        "option_raw_preview": text[:500] if text else None,
        "option_visible_status_token_count": option_count,
    }


def parse_newprice_page(price_url):
    if not price_url:
        return None, None

    try:
        r = requests.get(price_url, headers=HEADERS, timeout=15)
        r.raise_for_status()

        soup = BeautifulSoup(r.text, "lxml")
        text = " ".join(soup.stripped_strings)

        patterns = [
            r"([\d,]+)\s*만원",
            r"판매가격\s*([\d,]+)",
            r"기본가격\s*([\d,]+)",
        ]

        prices = []
        for pat in patterns:
            for m in re.finditer(pat, text):
                try:
                    prices.append(int(m.group(1).replace(",", "")))
                except Exception:
                    pass

        prices = [p for p in prices if 500 <= p <= 10000]

        if prices:
            return min(prices), text[:2000]

        return None, text[:2000]

    except Exception as e:
        return None, f"ERROR: {e}"


def finalize_newcar_price(row):
    for key in ["newcar_direct_price_만원", "신차기준가_만원", "newprice_link_price_만원"]:
        val = row.get(key)
        if val is not None:
            return val
    return None


# -------------------------
# 메인
# -------------------------
def main():
    BASE_SAVE_DIR.mkdir(parents=True, exist_ok=True)

    driver = setup_driver(headless=False)
    rows = []

    try:
        detail_links = collect_yf_links(driver, top_n=TOP_N)
        print(f"[INFO] 수집 대상 상세링크 수: {len(detail_links)}")

        for idx, detail_url in enumerate(detail_links, start=1):
            car_id = extract_car_id(detail_url)
            print(f"\n[{idx}/{len(detail_links)}] 매물ID={car_id}")

            car_dir = BASE_SAVE_DIR / str(car_id)
            car_dir.mkdir(parents=True, exist_ok=True)

            # detail 수집 + 저장 + 파싱
            detail_html = fetch_and_save_page(
                driver,
                detail_url,
                car_dir / "detail.html",
                car_dir / "detail.txt"
            )
            row = parse_detail_file(detail_html, car_id_hint=car_id)

            # newcar 수집 + 저장 + 파싱
            try:
                newcar_html = fetch_and_save_page(
                    driver,
                    f"https://fem.encar.com/cars/newcar/{car_id}",
                    car_dir / "newcar.html",
                    car_dir / "newcar.txt"
                )
                row.update(parse_newcar_file(newcar_html))
            except Exception as e:
                row["newcar_note"] = None
                row["newcar_direct_price_만원"] = None
                row["newcar_grade_base_price_만원"] = None
                row["newcar_option_price_만원"] = None
                row["newcar_raw_preview"] = f"ERROR: {e}"

            # option 수집 + 저장 + 파싱
            try:
                option_html = fetch_and_save_page(
                    driver,
                    f"https://fem.encar.com/cars/option/{car_id}",
                    car_dir / "option.html",
                    car_dir / "option.txt"
                )
                row.update(parse_option_file(option_html))
            except Exception as e:
                row["option_raw_preview"] = f"ERROR: {e}"
                row["option_visible_status_token_count"] = None

            # 신차 가격표 fallback
            price, preview = parse_newprice_page(row.get("신차가격표링크"))
            row["newprice_link_price_만원"] = price
            row["newprice_link_raw_preview"] = preview[:500] if preview else None

            # 최종 신차가
            row["최종신차가_만원"] = finalize_newcar_price(row)

            rows.append(row)
            print(f"[OK] {car_id} 완료")

    finally:
        driver.quit()

    df = pd.DataFrame(rows)
    df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

    with open(OUT_JSON, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

    print(f"\n저장 완료: {OUT_CSV}")
    print(f"저장 완료: {OUT_JSON}")
    print(f"HTML 저장 폴더: {BASE_SAVE_DIR.resolve()}")

    show_cols = [
        "매물ID", "차량명", "현재가격_만원",
        "교환_개수", "판금_개수", "부식_여부",
        "보험이력건수", "사고강도점수", "사고종합_여부",
        "최종신차가_만원"
    ]
    show_cols = [c for c in show_cols if c in df.columns]

    print("\n[HEAD]")
    print(df[show_cols].head())


if __name__ == "__main__":
    main()

[INFO] 수집 대상 상세링크 수: 20

[1/20] 매물ID=41509017
[OK] 41509017 완료

[2/20] 매물ID=41054615
[OK] 41054615 완료

[3/20] 매물ID=40903390
[OK] 40903390 완료

[4/20] 매물ID=40477859
[OK] 40477859 완료

[5/20] 매물ID=40955073
[OK] 40955073 완료

[6/20] 매물ID=39486507
[OK] 39486507 완료

[7/20] 매물ID=41225357
[OK] 41225357 완료

[8/20] 매물ID=40955073


KeyboardInterrupt: 

In [ ]:
import re
import time
import json
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# =========================
# 설정
# =========================
TOP_N = 
BASE_SAVE_DIR = Path("encar_yf_detail_5")
OUT_CSV = "encar_yf_detail_5.csv"
OUT_JSON = "encar_yf_detail_5.json"

SEARCH_URL = (
    "https://car.encar.com/list/car?page=1"
    "&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22%28And.Hidden.N._."
    "%28C.CarType.Y._.%28C.Manufacturer.%ED%98%84%EB%8C%80._."
    "%28C.ModelGroup.%EC%8F%98%EB%82%98%ED%83%80._.Model.YF+%EC%8F%98%EB%82%98%ED%83%80.%29%29%29%29%22%2C"
    "%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D"
)


# =========================
# 공통 유틸
# =========================
def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip() if text else ""


def find_first(patterns, text, cast=None, flags=re.S):
    for pat in patterns:
        m = re.search(pat, text, flags)
        if m:
            val = m.group(1) if m.lastindex else m.group(0)
            if cast:
                try:
                    return cast(val)
                except Exception:
                    return val
            return val
    return None


def to_int(val):
    if val is None:
        return None
    return int(str(val).replace(",", "").strip())


def parse_int_from_text(text):
    if not text:
        return None
    m = re.search(r"([\d,]+)", text)
    return int(m.group(1).replace(",", "")) if m else None


def parse_json_array_str(text):
    if not text:
        return []
    try:
        return json.loads(text)
    except Exception:
        return []


def extract_car_id(url):
    m = re.search(r"/cars/detail/(\d+)", url)
    return m.group(1) if m else None


def build_full_trim(model, grade, detail):
    parts = []
    for p in [model, grade, detail]:
        p = normalize_text(p)
        if p and p not in parts:
            parts.append(p)
    return " ".join(parts) if parts else None


def convert_year_from_embedded(year_month, form_year, fallback_text):
    if form_year:
        try:
            return int(form_year)
        except Exception:
            pass
    if year_month and len(year_month) >= 4:
        try:
            return int(year_month[:4])
        except Exception:
            pass
    if fallback_text:
        m = re.search(r"(\d{2})/", fallback_text)
        if m:
            return 2000 + int(m.group(1))
    return None


# =========================
# Selenium
# =========================
def setup_driver(headless=False):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--lang=ko-KR")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=options)
    driver.implicitly_wait(3)
    return driver


def wait_body(driver, sec=10):
    WebDriverWait(driver, sec).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )


def close_popups(driver):
    popup_xpaths = [
        "//button[contains(., '닫기')]",
        "//button[contains(., '나중에')]",
        "//button[contains(., '오늘 보지 않기')]",
        "//button[contains(., '취소')]",
        "//button[contains(., '확인')]",
        "//button[contains(., '다음에')]",
        "//a[contains(., '닫기')]",
    ]

    for xp in popup_xpaths:
        try:
            buttons = driver.find_elements(By.XPATH, xp)
            for btn in buttons:
                try:
                    if btn.is_displayed():
                        driver.execute_script("arguments[0].click();", btn)
                        time.sleep(0.3)
                except Exception:
                    pass
        except Exception:
            pass


def scroll_until_stable(driver, pause=2.0, max_rounds=25):
    last_height = driver.execute_script("return document.body.scrollHeight")
    stable_count = 0

    for _ in range(max_rounds):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)

        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            stable_count += 1
        else:
            stable_count = 0

        if stable_count >= 2:
            break

        last_height = new_height


def collect_detail_links(driver):
    anchors = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/cars/detail/"]')
    urls = []

    for a in anchors:
        try:
            href = a.get_attribute("href")
            if href and "/cars/detail/" in href:
                href = href.split("&advClickPosition=")[0]
                urls.append(href)
        except Exception:
            pass

    return list(dict.fromkeys(urls))


def collect_yf_links(driver, top_n=5000):
    driver.get(SEARCH_URL)
    wait_body(driver)
    time.sleep(2)
    close_popups(driver)
    scroll_until_stable(driver, pause=2.0, max_rounds=25)

    links = collect_detail_links(driver)
    return links[:top_n]


def fetch_and_save_detail_page(driver, url, html_path, txt_path=None, sleep_sec=2.0):
    driver.get(url)
    wait_body(driver)
    time.sleep(sleep_sec)
    close_popups(driver)
    time.sleep(0.5)

    html = driver.page_source
    html_path.write_text(html, encoding="utf-8")

    if txt_path is not None:
        body_text = normalize_text(driver.find_element(By.TAG_NAME, "body").text)
        txt_path.write_text(body_text, encoding="utf-8")

    return html


# =========================
# detail 파싱
# =========================
def parse_meta_description(soup):
    result = {}
    meta = soup.find("meta", attrs={"name": "description"})
    if not meta:
        return result

    content = meta.get("content", "")
    result["연식_메타"] = find_first([r"연식:([^,]+)"], content)
    result["주행거리_메타"] = find_first([r"주행거리:([^,]+)"], content)
    result["연료_메타"] = find_first([r"연료:([^,]+)"], content)
    result["색상_메타"] = find_first([r"색상:([^,]+)"], content)
    result["지역_메타"] = find_first([r"지역:([^,]+?) 중고차"], content)
    return result


def parse_dom_basic(soup):
    result = {
        "차량명_dom": None,
        "모델_dom": None,
        "세부트림_dom": None,
        "연식_원문_dom": None,
        "주행거리_원문_dom": None,
        "연료_dom": None,
        "차량번호_dom": None,
        "등록번호_dom": None,
        "조회수_dom": None,
        "찜수_dom": None,
        "해시태그_dom": None,
        "실촬영문구_dom": None,
    }

    h3 = soup.find("h3")
    if h3:
        spans = [normalize_text(x.get_text(" ", strip=True)) for x in h3.find_all("span")]
        spans = [x for x in spans if x]
        if len(spans) >= 2:
            result["모델_dom"] = spans[0]
            result["세부트림_dom"] = spans[1]
            result["차량명_dom"] = " ".join(spans).strip()
        else:
            result["차량명_dom"] = normalize_text(h3.get_text(" ", strip=True))

    dl = soup.select_one("dl.ar1Ivd7EgX")
    if dl:
        dts = dl.find_all("dt")
        dds = dl.find_all("dd")
        for dt, dd in zip(dts, dds):
            k = normalize_text(dt.get_text(" ", strip=True))
            v = normalize_text(dd.get_text(" ", strip=True))
            if "연식" in k:
                result["연식_원문_dom"] = v
            elif "주행거리" in k:
                result["주행거리_원문_dom"] = v
            elif "연료" in k:
                result["연료_dom"] = v
            elif "차량번호" in k:
                result["차량번호_dom"] = v

    for li in soup.select("ul.rVigc5A_1H li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt.startswith("등록번호"):
            result["등록번호_dom"] = txt.replace("등록번호", "").strip()
        elif txt.startswith("조회수"):
            result["조회수_dom"] = parse_int_from_text(txt)
        elif txt.startswith("찜"):
            result["찜수_dom"] = parse_int_from_text(txt)

    tags = [normalize_text(li.get_text(" ", strip=True)) for li in soup.select("ul.kYC8KS_YZ1 li")]
    tags = [x for x in tags if x]
    result["해시태그_dom"] = ",".join(tags) if tags else None

    shot = soup.select_one("p.s8FzQTBVpE")
    if shot:
        result["실촬영문구_dom"] = normalize_text(shot.get_text(" ", strip=True))

    return result


def parse_detail_embedded(html):
    result = {}

    str_patterns = {
        "manufacturerName": [r'"manufacturerName":"([^"]+)"'],
        "modelName": [r'"modelName":"([^"]+)"'],
        "gradeName": [r'"gradeName":"([^"]+)"'],
        "gradeDetailName": [r'"gradeDetailName":"([^"]+)"'],
        "vehicleNo": [r'"vehicleNo":"([^"]+)"'],
        "vin": [r'"vin":"([^"]+)"'],
        "requestUrl": [r'"requestUrl":"([^"]+)"'],
        "dealerName": [r'"dealer":\{"userId":"[^"]+","name":"([^"]+)"'],
        "firmName": [r'"firm":\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterName": [r'"diagnosisCenters":\[\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterPhone": [r'"telephoneNumber":"([^"]+)"'],
        "diagnosisCenterAddress": [r'"address":"([^"]+)"'],
        "pageAccessToken": [r'"pageAccessToken":"([^"]+)"'],
        "yearMonth": [r'"yearMonth":"([^"]+)"'],
        "formYear": [r'"formYear":"([^"]+)"'],
        "fuelName": [r'"fuelName":"([^"]+)"'],
        "colorName": [r'"colorName":"([^"]+)"'],
        "transmissionName": [r'"transmissionName":"([^"]+)"'],
        "bodyName": [r'"bodyName":"([^"]+)"'],
        "oneLineText": [r'"oneLineText":"([^"]+)"'],
    }

    int_patterns = {
        "originPrice": [r'"originPrice":(\d+)'],
        "price": [r'"price":(\d+)'],
        "viewCount": [r'"viewCount":(\d+)'],
        "subscribeCount": [r'"subscribeCount":(\d+)'],
        "vehicleId": [r'"vehicleId":(\d+)'],
        "seizingCount": [r'"seizingCount":(\d+)'],
        "pledgeCount": [r'"pledgeCount":(\d+)'],
        "encarDiagnosis": [r'"encarDiagnosis":(-?\d+)'],
        "encarMeetGo": [r'"encarMeetGo":(-?\d+)'],
        "mileage": [r'"mileage":(\d+)'],
        "displacement": [r'"displacement":(\d+)'],
        "seatCount": [r'"seatCount":(\d+)'],
    }

    for key, pats in str_patterns.items():
        result[key] = find_first(pats, html)

    for key, pats in int_patterns.items():
        result[key] = find_first(pats, html, cast=to_int)

    standard_raw = find_first([r'"standard":(\[[^\]]*\])'], html)
    choice_raw = find_first([r'"choice":(\[[^\]]*\])'], html)
    tuning_raw = find_first([r'"tuning":(\[[^\]]*\])'], html)
    etc_raw = find_first([r'"etc":(\[[^\]]*\]|"[^"]*")'], html)

    result["option_standard_codes"] = parse_json_array_str(standard_raw)
    result["option_choice_codes"] = parse_json_array_str(choice_raw)
    result["option_tuning_codes"] = parse_json_array_str(tuning_raw)

    if etc_raw and etc_raw.startswith("["):
        result["option_etc_values"] = parse_json_array_str(etc_raw)
    elif etc_raw:
        result["option_etc_values"] = [etc_raw.strip('"')]
    else:
        result["option_etc_values"] = []

    return result


def parse_dom_major_options(soup):
    result = {}
    for li in soup.select("ul.dz3qFYruNO li"):
        text = normalize_text(li.get_text(" ", strip=True))
        blind = li.select_one("span.blind")
        status = normalize_text(blind.get_text(" ", strip=True)) if blind else None

        if status:
            name = normalize_text(text.replace(status, ""))
            result[f"주요옵션_{name}"] = 1 if status == "있음" else 0 if status == "없음" else None

    return result


def parse_dom_seller_info(soup):
    result = {
        "판매자상호_dom": None,
        "판매자명_dom": None,
        "판매자유형_dom": None,
        "판매중대수_dom": None,
        "판매완료대수_dom": None,
        "판매자지역_dom": None,
        "종사원증번호_dom": None,
    }

    btn = soup.select_one("button.uPEfVsKnZx")
    if btn:
        brand = btn.select_one("span.ESlvHTih77")
        name = btn.select_one("strong.k3tS4rXdrQ")
        seller_type = btn.select_one("span.xf9ufEfgKR")

        if brand:
            result["판매자상호_dom"] = normalize_text(brand.get_text(" ", strip=True))
        if name:
            result["판매자명_dom"] = normalize_text(name.get_text(" ", strip=True))
        if seller_type:
            result["판매자유형_dom"] = normalize_text(seller_type.get_text(" ", strip=True))

    lis = soup.select("ul.VtNR8dNHOS li")
    for li in lis:
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt.startswith("판매중"):
            result["판매중대수_dom"] = parse_int_from_text(txt)
        elif txt.startswith("판매완료"):
            result["판매완료대수_dom"] = parse_int_from_text(txt)
        elif "종사원증번호" in txt:
            m = re.search(r"종사원증번호\s*([A-Z0-9\-]+)", txt)
            if m:
                result["종사원증번호_dom"] = m.group(1)
        elif any(region in txt for region in ["서울", "경기", "인천", "부산", "대구", "대전", "광주", "울산", "세종", "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"]):
            result["판매자지역_dom"] = txt

    return result


def parse_dom_extra_flags(soup):
    result = {
        "보험이력공개여부_dom": None,
        "보험이력건수_dom": None,
        "성능점검유형_dom": None,
        "성능점검설명_dom": None,
    }

    for btn in soup.select("button.fSZoGuAX4M"):
        txt = normalize_text(btn.get_text(" ", strip=True))
        if "보험이력" in txt:
            result["보험이력공개여부_dom"] = txt
            m = re.search(r"보험이력\s*(\d+)건", txt)
            if m:
                result["보험이력건수_dom"] = int(m.group(1))
        elif "성능점검내역" in txt:
            result["성능점검유형_dom"] = txt.replace("성능점검내역", "").strip()

    perf_desc = soup.select_one("div.Hs3feBPsTj p.lGhsYmQaGE")
    if perf_desc:
        result["성능점검설명_dom"] = normalize_text(perf_desc.get_text(" ", strip=True))

    return result


def parse_damage_detail(soup):
    result = {
        "교환_개수": None,
        "판금_개수": None,
        "부식_여부": None,
        "성능기록부_raw": None,
    }

    items = []
    for li in soup.select("ul.wB0X7nC0cq li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt:
            items.append(txt)

        if "교환" in txt:
            if "없음" in txt:
                result["교환_개수"] = 0
            else:
                m = re.search(r"교환\s*(\d+)", txt)
                if m:
                    result["교환_개수"] = int(m.group(1))

        elif "판금" in txt:
            if "없음" in txt:
                result["판금_개수"] = 0
            else:
                m = re.search(r"판금\s*(\d+)", txt)
                if m:
                    result["판금_개수"] = int(m.group(1))

        elif "부식" in txt:
            result["부식_여부"] = 0 if "없음" in txt else 1

    result["성능기록부_raw"] = " || ".join(items) if items else None
    return result


def derive_accident_features(row):
    exchange_cnt = row.get("교환_개수")
    panel_cnt = row.get("판금_개수")
    corrosion = row.get("부식_여부")
    insurance_cnt = row.get("보험이력건수")

    row["교환_여부"] = None if exchange_cnt is None else int(exchange_cnt > 0)
    row["판금_여부"] = None if panel_cnt is None else int(panel_cnt > 0)
    row["외판수리_여부"] = None if (exchange_cnt is None and panel_cnt is None) else int((exchange_cnt or 0) + (panel_cnt or 0) > 0)

    if exchange_cnt is None and panel_cnt is None:
        row["사고강도점수"] = None
    else:
        row["사고강도점수"] = (exchange_cnt or 0) * 2 + (panel_cnt or 0)

    row["보험이력_여부"] = None if insurance_cnt is None else int(insurance_cnt > 0)

    signals = []
    for v in [row["교환_여부"], row["판금_여부"], row["보험이력_여부"]]:
        if v is not None:
            signals.append(v)

    row["사고종합_여부"] = None if not signals else int(any(signals))

    if row["사고강도점수"] is None:
        row["중대사고_추정"] = None
    else:
        row["중대사고_추정"] = int((exchange_cnt or 0) >= 2 or row["사고강도점수"] >= 4)

    row["부식_위험"] = corrosion
    return row


def parse_detail_file(detail_html: str, car_id_hint=None):
    soup = BeautifulSoup(detail_html, "lxml")

    meta = parse_meta_description(soup)
    dom = parse_dom_basic(soup)
    emb = parse_detail_embedded(detail_html)
    major_options = parse_dom_major_options(soup)
    seller_dom = parse_dom_seller_info(soup)
    flags_dom = parse_dom_extra_flags(soup)
    damage_dom = parse_damage_detail(soup)

    model = emb.get("modelName") or dom.get("모델_dom")
    grade = emb.get("gradeName")
    detail = emb.get("gradeDetailName") or dom.get("세부트림_dom")

    if normalize_text(grade) == normalize_text(detail):
        detail = None

    row = {
        "매물ID": emb.get("vehicleId") or car_id_hint,
        "제조사": emb.get("manufacturerName"),
        "모델": model,
        "등급명": grade,
        "세부트림": detail,
        "차량명": build_full_trim(model, grade, detail),
        "현재가격_만원": emb.get("price"),
        "신차기준가_만원": emb.get("originPrice"),
        "조회수": emb.get("viewCount") or dom.get("조회수_dom"),
        "찜수": emb.get("subscribeCount") or dom.get("찜수_dom"),
        "차량번호": emb.get("vehicleNo") or dom.get("차량번호_dom"),
        "VIN": emb.get("vin"),
        "연식_원문": dom.get("연식_원문_dom") or meta.get("연식_메타"),
        "연식": convert_year_from_embedded(emb.get("yearMonth"), emb.get("formYear"), dom.get("연식_원문_dom") or meta.get("연식_메타")),
        "주행거리_원문": dom.get("주행거리_원문_dom") or meta.get("주행거리_메타"),
        "주행거리_km": emb.get("mileage") or parse_int_from_text(dom.get("주행거리_원문_dom") or meta.get("주행거리_메타")),
        "연료": emb.get("fuelName") or dom.get("연료_dom") or meta.get("연료_메타"),
        "색상": emb.get("colorName") or meta.get("색상_메타"),
        "지역": meta.get("지역_메타"),
        "변속기": emb.get("transmissionName"),
        "배기량_cc": emb.get("displacement"),
        "차급": emb.get("bodyName"),
        "좌석수": emb.get("seatCount"),
        "등록번호": dom.get("등록번호_dom"),
        "해시태그": dom.get("해시태그_dom"),
        "실촬영문구": dom.get("실촬영문구_dom"),
        "딜러명": emb.get("dealerName") or seller_dom.get("판매자명_dom"),
        "상사명": emb.get("firmName") or seller_dom.get("판매자상호_dom"),
        "판매자유형": seller_dom.get("판매자유형_dom"),
        "판매중대수": seller_dom.get("판매중대수_dom"),
        "판매완료대수": seller_dom.get("판매완료대수_dom"),
        "판매자지역": seller_dom.get("판매자지역_dom"),
        "종사원증번호": seller_dom.get("종사원증번호_dom"),
        "진단센터명": emb.get("diagnosisCenterName"),
        "진단센터전화": emb.get("diagnosisCenterPhone"),
        "진단센터주소": emb.get("diagnosisCenterAddress"),
        "압류건수": emb.get("seizingCount"),
        "저당건수": emb.get("pledgeCount"),
        "엔카진단여부값": emb.get("encarDiagnosis"),
        "엔카믿고값": emb.get("encarMeetGo"),
        "광고한줄문구": emb.get("oneLineText"),
        "보험이력공개여부": flags_dom.get("보험이력공개여부_dom"),
        "보험이력건수": flags_dom.get("보험이력건수_dom"),
        "성능점검유형": flags_dom.get("성능점검유형_dom"),
        "성능점검설명": flags_dom.get("성능점검설명_dom"),
        "requestUrl": emb.get("requestUrl"),
        "pageAccessToken": emb.get("pageAccessToken"),
        "옵션_기본코드개수": len(emb.get("option_standard_codes", [])),
        "옵션_선택코드개수": len(emb.get("option_choice_codes", [])),
        "옵션_튜닝코드개수": len(emb.get("option_tuning_codes", [])),
        "옵션_기타개수": len(emb.get("option_etc_values", [])),
        "옵션_기본코드": ",".join(emb.get("option_standard_codes", [])) if emb.get("option_standard_codes") else None,
        "옵션_선택코드": ",".join(emb.get("option_choice_codes", [])) if emb.get("option_choice_codes") else None,
        "옵션_튜닝코드": ",".join(emb.get("option_tuning_codes", [])) if emb.get("option_tuning_codes") else None,
        "옵션_기타값": " | ".join(emb.get("option_etc_values", [])) if emb.get("option_etc_values") else None,
    }

    row.update(major_options)
    row.update(damage_dom)
    row = derive_accident_features(row)

    return row


# -------------------------
# 메인
# -------------------------
def main():
    BASE_SAVE_DIR.mkdir(parents=True, exist_ok=True)

    driver = setup_driver(headless=False)
    rows = []

    try:
        detail_links = collect_yf_links(driver, top_n=TOP_N)
        print(f"[INFO] 수집 대상 상세링크 수: {len(detail_links)}")

        for idx, detail_url in enumerate(detail_links, start=1):
            car_id = extract_car_id(detail_url)
            print(f"\n[{idx}/{len(detail_links)}] 매물ID={car_id}")

            car_dir = BASE_SAVE_DIR / str(car_id)
            car_dir.mkdir(parents=True, exist_ok=True)

            detail_html = fetch_and_save_detail_page(
                driver,
                detail_url,
                car_dir / "detail.html",
                car_dir / "detail.txt"
            )

            row = parse_detail_file(detail_html, car_id_hint=car_id)
            rows.append(row)
            print(f"[OK] {car_id} 완료")

    finally:
        driver.quit()

    df = pd.DataFrame(rows)
    df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

    with open(OUT_JSON, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

    print(f"\n저장 완료: {OUT_CSV}")
    print(f"저장 완료: {OUT_JSON}")
    print(f"HTML 저장 폴더: {BASE_SAVE_DIR.resolve()}")

    show_cols = [
        "매물ID", "차량명", "현재가격_만원",
        "교환_개수", "판금_개수", "부식_여부",
        "보험이력건수", "사고강도점수", "사고종합_여부"
    ]
    show_cols = [c for c in show_cols if c in df.columns]

    print("\n[HEAD]")
    print(df[show_cols].head())


if __name__ == "__main__":
    main()

[INFO] 수집 대상 상세링크 수: 5

[1/5] 매물ID=40903390
[OK] 40903390 완료

[2/5] 매물ID=41509017
[OK] 41509017 완료

[3/5] 매물ID=41225357
[OK] 41225357 완료

[4/5] 매물ID=40955073
[OK] 40955073 완료

[5/5] 매물ID=41054615
[OK] 41054615 완료

저장 완료: encar_yf_detail_5.csv
저장 완료: encar_yf_detail_5.json
HTML 저장 폴더: C:\Users\Admin\hipython\버뮤다_프로젝트_2\encar_yf_detail_5

[HEAD]
       매물ID                    차량명  현재가격_만원  교환_개수  판금_개수  부식_여부  보험이력건수  \
0  40903390  YF 쏘나타 LPI 프리미어(장애인용)      330      0      0      0       5   
1  41509017      YF 쏘나타 Y20 프라임 블랙      399      2      1      0       3   
2  41225357         YF 쏘나타 Y20 프라임      330      4      1      0       4   
3  40955073     YF 쏘나타 Y20 프라임 고급형      430      1      2      0       5   
4  41054615        YF 쏘나타 CVVL 럭셔리      430      0      0      0       4   

   사고강도점수  사고종합_여부  
0       0        1  
1       5        1  
2       9        1  
3       4        1  
4       0        1  


In [ ]:
import re
import time
import json
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# =========================
# 설정
# =========================
TARGET_N = 5000
SAVE_EVERY = 50

BASE_SAVE_DIR = Path("encar_hyundai_all")
HTML_DIR = BASE_SAVE_DIR / "html"
OUTPUT_DIR = BASE_SAVE_DIR / "output"

BATCH_CSV = OUTPUT_DIR / "hyundai_detail_batch.csv"
ERROR_CSV = OUTPUT_DIR / "hyundai_detail_errors.csv"
FINAL_JSON = OUTPUT_DIR / "hyundai_detail_final.json"

# "현대 전체" 검색 URL로 바꿔서 사용
SEARCH_URL = (
    "https://car.encar.com/list/car?page=1&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22(And.Hidden.N._.(C.CarType.Y._.Manufacturer.%ED%98%84%EB%8C%80.))%22%2C%22title%22%3A%22%ED%98%84%EB%8C%80%22%2C%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D"
)


# =========================
# 공통 유틸
# =========================
def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip() if text else ""


def find_first(patterns, text, cast=None, flags=re.S):
    for pat in patterns:
        m = re.search(pat, text, flags)
        if m:
            val = m.group(1) if m.lastindex else m.group(0)
            if cast:
                try:
                    return cast(val)
                except Exception:
                    return val
            return val
    return None


def to_int(val):
    if val is None:
        return None
    return int(str(val).replace(",", "").strip())


def parse_int_from_text(text):
    if not text:
        return None
    m = re.search(r"([\d,]+)", text)
    return int(m.group(1).replace(",", "")) if m else None


def parse_json_array_str(text):
    if not text:
        return []
    try:
        return json.loads(text)
    except Exception:
        return []


def extract_car_id(url):
    m = re.search(r"/cars/detail/(\d+)", url)
    return m.group(1) if m else None


def build_full_trim(model, grade, detail):
    parts = []
    for p in [model, grade, detail]:
        p = normalize_text(p)
        if p and p not in parts:
            parts.append(p)
    return " ".join(parts) if parts else None


def convert_year_from_embedded(year_month, form_year, fallback_text):
    if form_year:
        try:
            return int(form_year)
        except Exception:
            pass
    if year_month and len(year_month) >= 4:
        try:
            return int(year_month[:4])
        except Exception:
            pass
    if fallback_text:
        m = re.search(r"(\d{2})/", fallback_text)
        if m:
            return 2000 + int(m.group(1))
    return None


# =========================
# Selenium
# =========================
def setup_driver(headless=False):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--lang=ko-KR")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=options)
    driver.implicitly_wait(3)
    return driver


def wait_body(driver, sec=10):
    WebDriverWait(driver, sec).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )


def close_popups(driver):
    popup_xpaths = [
        "//button[contains(., '닫기')]",
        "//button[contains(., '나중에')]",
        "//button[contains(., '오늘 보지 않기')]",
        "//button[contains(., '취소')]",
        "//button[contains(., '확인')]",
        "//button[contains(., '다음에')]",
        "//a[contains(., '닫기')]",
    ]
    for xp in popup_xpaths:
        try:
            buttons = driver.find_elements(By.XPATH, xp)
            for btn in buttons:
                try:
                    if btn.is_displayed():
                        driver.execute_script("arguments[0].click();", btn)
                        time.sleep(0.2)
                except Exception:
                    pass
        except Exception:
            pass


def collect_detail_links(driver):
    anchors = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/cars/detail/"]')
    urls = []
    for a in anchors:
        try:
            href = a.get_attribute("href")
            if href and "/cars/detail/" in href:
                href = href.split("&advClickPosition=")[0]
                urls.append(href)
        except Exception:
            pass
    return list(dict.fromkeys(urls))


def collect_sonata_links(driver, target_n=4000, max_scrolls=300):
    driver.get(SEARCH_URL)
    wait_body(driver)
    time.sleep(2)
    close_popups(driver)

    links = []
    last_count = 0
    stable_round = 0

    for i in range(max_scrolls):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

        links = collect_detail_links(driver)
        current_count = len(links)
        print(f"[SCROLL {i+1}] 링크 수: {current_count}")

        if current_count >= target_n:
            return links[:target_n]

        if current_count == last_count:
            stable_round += 1
        else:
            stable_round = 0

        if stable_round >= 5:
            print("[INFO] 더 이상 링크가 늘지 않아 중단")
            break

        last_count = current_count

    return links[:target_n]


# =========================
# detail 파싱
# =========================
def parse_meta_description(soup):
    result = {}
    meta = soup.find("meta", attrs={"name": "description"})
    if not meta:
        return result

    content = meta.get("content", "")
    result["연식_메타"] = find_first([r"연식:([^,]+)"], content)
    result["주행거리_메타"] = find_first([r"주행거리:([^,]+)"], content)
    result["연료_메타"] = find_first([r"연료:([^,]+)"], content)
    result["색상_메타"] = find_first([r"색상:([^,]+)"], content)
    result["지역_메타"] = find_first([r"지역:([^,]+?) 중고차"], content)
    return result


def parse_dom_basic(soup):
    result = {
        "차량명_dom": None,
        "모델_dom": None,
        "세부트림_dom": None,
        "연식_원문_dom": None,
        "주행거리_원문_dom": None,
        "연료_dom": None,
        "차량번호_dom": None,
        "등록번호_dom": None,
        "조회수_dom": None,
        "찜수_dom": None,
        "해시태그_dom": None,
        "실촬영문구_dom": None,
    }

    h3 = soup.find("h3")
    if h3:
        spans = [normalize_text(x.get_text(" ", strip=True)) for x in h3.find_all("span")]
        spans = [x for x in spans if x]
        if len(spans) >= 2:
            result["모델_dom"] = spans[0]
            result["세부트림_dom"] = spans[1]
            result["차량명_dom"] = " ".join(spans).strip()
        else:
            result["차량명_dom"] = normalize_text(h3.get_text(" ", strip=True))

    dl = soup.select_one("dl.ar1Ivd7EgX")
    if dl:
        dts = dl.find_all("dt")
        dds = dl.find_all("dd")
        for dt, dd in zip(dts, dds):
            k = normalize_text(dt.get_text(" ", strip=True))
            v = normalize_text(dd.get_text(" ", strip=True))
            if "연식" in k:
                result["연식_원문_dom"] = v
            elif "주행거리" in k:
                result["주행거리_원문_dom"] = v
            elif "연료" in k:
                result["연료_dom"] = v
            elif "차량번호" in k:
                result["차량번호_dom"] = v

    for li in soup.select("ul.rVigc5A_1H li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt.startswith("등록번호"):
            result["등록번호_dom"] = txt.replace("등록번호", "").strip()
        elif txt.startswith("조회수"):
            result["조회수_dom"] = parse_int_from_text(txt)
        elif txt.startswith("찜"):
            result["찜수_dom"] = parse_int_from_text(txt)

    tags = [normalize_text(li.get_text(" ", strip=True)) for li in soup.select("ul.kYC8KS_YZ1 li")]
    tags = [x for x in tags if x]
    result["해시태그_dom"] = ",".join(tags) if tags else None

    shot = soup.select_one("p.s8FzQTBVpE")
    if shot:
        result["실촬영문구_dom"] = normalize_text(shot.get_text(" ", strip=True))

    return result


def parse_detail_embedded(html):
    result = {}

    str_patterns = {
        "manufacturerName": [r'"manufacturerName":"([^"]+)"'],
        "modelName": [r'"modelName":"([^"]+)"'],
        "gradeName": [r'"gradeName":"([^"]+)"'],
        "gradeDetailName": [r'"gradeDetailName":"([^"]+)"'],
        "vehicleNo": [r'"vehicleNo":"([^"]+)"'],
        "vin": [r'"vin":"([^"]+)"'],
        "requestUrl": [r'"requestUrl":"([^"]+)"'],
        "dealerName": [r'"dealer":\{"userId":"[^"]+","name":"([^"]+)"'],
        "firmName": [r'"firm":\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterName": [r'"diagnosisCenters":\[\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterPhone": [r'"telephoneNumber":"([^"]+)"'],
        "diagnosisCenterAddress": [r'"address":"([^"]+)"'],
        "pageAccessToken": [r'"pageAccessToken":"([^"]+)"'],
        "yearMonth": [r'"yearMonth":"([^"]+)"'],
        "formYear": [r'"formYear":"([^"]+)"'],
        "fuelName": [r'"fuelName":"([^"]+)"'],
        "colorName": [r'"colorName":"([^"]+)"'],
        "transmissionName": [r'"transmissionName":"([^"]+)"'],
        "bodyName": [r'"bodyName":"([^"]+)"'],
        "oneLineText": [r'"oneLineText":"([^"]+)"'],
    }

    int_patterns = {
        "price": [r'"price":(\d+)'],
        "viewCount": [r'"viewCount":(\d+)'],
        "subscribeCount": [r'"subscribeCount":(\d+)'],
        "vehicleId": [r'"vehicleId":(\d+)'],
        "seizingCount": [r'"seizingCount":(\d+)'],
        "pledgeCount": [r'"pledgeCount":(\d+)'],
        "encarDiagnosis": [r'"encarDiagnosis":(-?\d+)'],
        "encarMeetGo": [r'"encarMeetGo":(-?\d+)'],
        "mileage": [r'"mileage":(\d+)'],
        "displacement": [r'"displacement":(\d+)'],
        "seatCount": [r'"seatCount":(\d+)'],
    }

    for key, pats in str_patterns.items():
        result[key] = find_first(pats, html)

    for key, pats in int_patterns.items():
        result[key] = find_first(pats, html, cast=to_int)

    standard_raw = find_first([r'"standard":(\[[^\]]*\])'], html)
    choice_raw = find_first([r'"choice":(\[[^\]]*\])'], html)
    tuning_raw = find_first([r'"tuning":(\[[^\]]*\])'], html)
    etc_raw = find_first([r'"etc":(\[[^\]]*\]|"[^"]*")'], html)

    result["option_standard_codes"] = parse_json_array_str(standard_raw)
    result["option_choice_codes"] = parse_json_array_str(choice_raw)
    result["option_tuning_codes"] = parse_json_array_str(tuning_raw)

    if etc_raw and etc_raw.startswith("["):
        result["option_etc_values"] = parse_json_array_str(etc_raw)
    elif etc_raw:
        result["option_etc_values"] = [etc_raw.strip('"')]
    else:
        result["option_etc_values"] = []

    return result


def parse_dom_major_options(soup):
    result = {}
    for li in soup.select("ul.dz3qFYruNO li"):
        text = normalize_text(li.get_text(" ", strip=True))
        blind = li.select_one("span.blind")
        status = normalize_text(blind.get_text(" ", strip=True)) if blind else None

        if status:
            name = normalize_text(text.replace(status, ""))
            result[f"주요옵션_{name}"] = 1 if status == "있음" else 0 if status == "없음" else None

    return result


def parse_dom_seller_info(soup):
    result = {
        "판매자상호_dom": None,
        "판매자명_dom": None,
        "판매자유형_dom": None,
        "판매중대수_dom": None,
        "판매완료대수_dom": None,
        "판매자지역_dom": None,
        "종사원증번호_dom": None,
    }

    btn = soup.select_one("button.uPEfVsKnZx")
    if btn:
        brand = btn.select_one("span.ESlvHTih77")
        name = btn.select_one("strong.k3tS4rXdrQ")
        seller_type = btn.select_one("span.xf9ufEfgKR")

        if brand:
            result["판매자상호_dom"] = normalize_text(brand.get_text(" ", strip=True))
        if name:
            result["판매자명_dom"] = normalize_text(name.get_text(" ", strip=True))
        if seller_type:
            result["판매자유형_dom"] = normalize_text(seller_type.get_text(" ", strip=True))

    lis = soup.select("ul.VtNR8dNHOS li")
    for li in lis:
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt.startswith("판매중"):
            result["판매중대수_dom"] = parse_int_from_text(txt)
        elif txt.startswith("판매완료"):
            result["판매완료대수_dom"] = parse_int_from_text(txt)
        elif "종사원증번호" in txt:
            m = re.search(r"종사원증번호\s*([A-Z0-9\-]+)", txt)
            if m:
                result["종사원증번호_dom"] = m.group(1)
        elif any(region in txt for region in ["서울", "경기", "인천", "부산", "대구", "대전", "광주", "울산", "세종", "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"]):
            result["판매자지역_dom"] = txt

    return result


def parse_dom_extra_flags(soup):
    result = {
        "보험이력공개여부_dom": None,
        "보험이력건수_dom": None,
        "성능점검유형_dom": None,
        "성능점검설명_dom": None,
    }

    for btn in soup.select("button.fSZoGuAX4M"):
        txt = normalize_text(btn.get_text(" ", strip=True))
        if "보험이력" in txt:
            result["보험이력공개여부_dom"] = txt
            m = re.search(r"보험이력\s*(\d+)건", txt)
            if m:
                result["보험이력건수_dom"] = int(m.group(1))
        elif "성능점검내역" in txt:
            result["성능점검유형_dom"] = txt.replace("성능점검내역", "").strip()

    perf_desc = soup.select_one("div.Hs3feBPsTj p.lGhsYmQaGE")
    if perf_desc:
        result["성능점검설명_dom"] = normalize_text(perf_desc.get_text(" ", strip=True))

    return result


def parse_damage_detail(soup):
    result = {
        "교환_개수": None,
        "판금_개수": None,
        "부식_여부": None,
        "성능기록부_raw": None,
    }

    items = []
    for li in soup.select("ul.wB0X7nC0cq li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt:
            items.append(txt)

        if "교환" in txt:
            if "없음" in txt:
                result["교환_개수"] = 0
            else:
                m = re.search(r"교환\s*(\d+)", txt)
                if m:
                    result["교환_개수"] = int(m.group(1))

        elif "판금" in txt:
            if "없음" in txt:
                result["판금_개수"] = 0
            else:
                m = re.search(r"판금\s*(\d+)", txt)
                if m:
                    result["판금_개수"] = int(m.group(1))

        elif "부식" in txt:
            result["부식_여부"] = 0 if "없음" in txt else 1

    result["성능기록부_raw"] = " || ".join(items) if items else None
    return result


def derive_accident_features(row):
    exchange_cnt = row.get("교환_개수")
    panel_cnt = row.get("판금_개수")
    corrosion = row.get("부식_여부")
    insurance_cnt = row.get("보험이력건수")

    row["교환_여부"] = None if exchange_cnt is None else int(exchange_cnt > 0)
    row["판금_여부"] = None if panel_cnt is None else int(panel_cnt > 0)
    row["외판수리_여부"] = None if (exchange_cnt is None and panel_cnt is None) else int((exchange_cnt or 0) + (panel_cnt or 0) > 0)

    if exchange_cnt is None and panel_cnt is None:
        row["사고강도점수"] = None
    else:
        row["사고강도점수"] = (exchange_cnt or 0) * 2 + (panel_cnt or 0)

    row["보험이력_여부"] = None if insurance_cnt is None else int(insurance_cnt > 0)

    signals = []
    for v in [row["교환_여부"], row["판금_여부"], row["보험이력_여부"]]:
        if v is not None:
            signals.append(v)

    row["사고종합_여부"] = None if not signals else int(any(signals))

    if row["사고강도점수"] is None:
        row["중대사고_추정"] = None
    else:
        row["중대사고_추정"] = int((exchange_cnt or 0) >= 2 or row["사고강도점수"] >= 4)

    row["부식_위험"] = corrosion
    return row


def parse_detail_file(detail_html: str, car_id_hint=None):
    soup = BeautifulSoup(detail_html, "lxml")

    meta = parse_meta_description(soup)
    dom = parse_dom_basic(soup)
    emb = parse_detail_embedded(detail_html)
    major_options = parse_dom_major_options(soup)
    seller_dom = parse_dom_seller_info(soup)
    flags_dom = parse_dom_extra_flags(soup)
    damage_dom = parse_damage_detail(soup)

    model = emb.get("modelName") or dom.get("모델_dom")
    grade = emb.get("gradeName")
    detail = emb.get("gradeDetailName") or dom.get("세부트림_dom")

    if normalize_text(grade) == normalize_text(detail):
        detail = None

    row = {
        "매물ID": emb.get("vehicleId") or car_id_hint,
        "제조사": emb.get("manufacturerName"),
        "모델": model,
        "등급명": grade,
        "세부트림": detail,
        "차량명": build_full_trim(model, grade, detail),
        "현재가격_만원": emb.get("price"),
        "조회수": emb.get("viewCount") or dom.get("조회수_dom"),
        "찜수": emb.get("subscribeCount") or dom.get("찜수_dom"),
        "차량번호": emb.get("vehicleNo") or dom.get("차량번호_dom"),
        "VIN": emb.get("vin"),
        "연식_원문": dom.get("연식_원문_dom") or meta.get("연식_메타"),
        "연식": convert_year_from_embedded(emb.get("yearMonth"), emb.get("formYear"), dom.get("연식_원문_dom") or meta.get("연식_메타")),
        "주행거리_원문": dom.get("주행거리_원문_dom") or meta.get("주행거리_메타"),
        "주행거리_km": emb.get("mileage") or parse_int_from_text(dom.get("주행거리_원문_dom") or meta.get("주행거리_메타")),
        "연료": emb.get("fuelName") or dom.get("연료_dom") or meta.get("연료_메타"),
        "색상": emb.get("colorName") or meta.get("색상_메타"),
        "지역": meta.get("지역_메타"),
        "변속기": emb.get("transmissionName"),
        "배기량_cc": emb.get("displacement"),
        "차급": emb.get("bodyName"),
        "좌석수": emb.get("seatCount"),
        "등록번호": dom.get("등록번호_dom"),
        "해시태그": dom.get("해시태그_dom"),
        "실촬영문구": dom.get("실촬영문구_dom"),
        "딜러명": emb.get("dealerName") or seller_dom.get("판매자명_dom"),
        "상사명": emb.get("firmName") or seller_dom.get("판매자상호_dom"),
        "판매자유형": seller_dom.get("판매자유형_dom"),
        "판매중대수": seller_dom.get("판매중대수_dom"),
        "판매완료대수": seller_dom.get("판매완료대수_dom"),
        "판매자지역": seller_dom.get("판매자지역_dom"),
        "종사원증번호": seller_dom.get("종사원증번호_dom"),
        "진단센터명": emb.get("diagnosisCenterName"),
        "진단센터전화": emb.get("diagnosisCenterPhone"),
        "진단센터주소": emb.get("diagnosisCenterAddress"),
        "압류건수": emb.get("seizingCount"),
        "저당건수": emb.get("pledgeCount"),
        "엔카진단여부값": emb.get("encarDiagnosis"),
        "엔카믿고값": emb.get("encarMeetGo"),
        "광고한줄문구": emb.get("oneLineText"),
        "보험이력공개여부": flags_dom.get("보험이력공개여부_dom"),
        "보험이력건수": flags_dom.get("보험이력건수_dom"),
        "성능점검유형": flags_dom.get("성능점검유형_dom"),
        "성능점검설명": flags_dom.get("성능점검설명_dom"),
        "requestUrl": emb.get("requestUrl"),
        "pageAccessToken": emb.get("pageAccessToken"),
        "옵션_기본코드개수": len(emb.get("option_standard_codes", [])),
        "옵션_선택코드개수": len(emb.get("option_choice_codes", [])),
        "옵션_튜닝코드개수": len(emb.get("option_tuning_codes", [])),
        "옵션_기타개수": len(emb.get("option_etc_values", [])),
        "옵션_기본코드": ",".join(emb.get("option_standard_codes", [])) if emb.get("option_standard_codes") else None,
        "옵션_선택코드": ",".join(emb.get("option_choice_codes", [])) if emb.get("option_choice_codes") else None,
        "옵션_튜닝코드": ",".join(emb.get("option_tuning_codes", [])) if emb.get("option_tuning_codes") else None,
        "옵션_기타값": " | ".join(emb.get("option_etc_values", [])) if emb.get("option_etc_values") else None,
    }

    row.update(major_options)
    row.update(damage_dom)
    row = derive_accident_features(row)

    return row


# -------------------------
# 메인
# -------------------------
def main():
    BASE_SAVE_DIR.mkdir(parents=True, exist_ok=True)

    driver = setup_driver(headless=False)
    rows = []

    try:
        detail_links = collect_yf_links(driver, top_n=TOP_N)
        print(f"[INFO] 수집 대상 상세링크 수: {len(detail_links)}")

        for idx, detail_url in enumerate(detail_links, start=1):
            car_id = extract_car_id(detail_url)
            print(f"\n[{idx}/{len(detail_links)}] 매물ID={car_id}")

            car_dir = BASE_SAVE_DIR / str(car_id)
            car_dir.mkdir(parents=True, exist_ok=True)

            detail_html = fetch_and_save_detail_page(
                driver,
                detail_url,
                car_dir / "detail.html",
                car_dir / "detail.txt"
            )

            row = parse_detail_file(detail_html, car_id_hint=car_id)
            rows.append(row)
            print(f"[OK] {car_id} 완료")

    finally:
        driver.quit()

    df = pd.DataFrame(rows)
    df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

    with open(OUT_JSON, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

    print(f"\n저장 완료: {OUT_CSV}")
    print(f"저장 완료: {OUT_JSON}")
    print(f"HTML 저장 폴더: {BASE_SAVE_DIR.resolve()}")

    show_cols = [
        "매물ID", "차량명", "현재가격_만원",
        "교환_개수", "판금_개수", "부식_여부",
        "보험이력건수", "사고강도점수", "사고종합_여부"
    ]
    show_cols = [c for c in show_cols if c in df.columns]

    print("\n[HEAD]")
    print(df[show_cols].head())


if __name__ == "__main__":
    main()

[INFO] 수집 대상 상세링크 수: 5

[1/5] 매물ID=41152405
[OK] 41152405 완료

[2/5] 매물ID=41303793
[OK] 41303793 완료

[3/5] 매물ID=41489486
[OK] 41489486 완료

[4/5] 매물ID=41451115
[OK] 41451115 완료

[5/5] 매물ID=40878239
[OK] 40878239 완료

저장 완료: encar_yf_detail_5.csv
저장 완료: encar_yf_detail_5.json
HTML 저장 폴더: C:\Users\Admin\hipython\버뮤다_프로젝트_2\encar_sonata_all

[HEAD]
       매물ID                          차량명  현재가격_만원  교환_개수  판금_개수  부식_여부  \
0  41152405         아반떼 (CN7) 1.6 인스퍼레이션     1950      0      0      0   
1  41303793  쏘나타 디 엣지(DN8) 1.6 터보 인스퍼레이션     2900      0      0      0   
2  41489486      팰리세이드 가솔린 3.8 2WD 프레스티지     2670      0      0      0   
3  41451115   더 뉴 팰리세이드 디젤 2.2 2WD 캘리그래피     4220      0      0      0   
4  40878239  싼타페 (MX5) HEV 1.6 4WD 캘리그래피     4380      0      0      0   

   보험이력건수  사고강도점수  사고종합_여부  
0       0       0        0  
1       1       0        1  
2       0       0        0  
3       0       0        0  
4       1       0        1  
